# CRRT / TMP — 공개용 재조립 노트북

*Cumulative operating time and early transmembrane pressure surge in relation to mortality during continuous renal replacement therapy* (Scientific Reports, under revision)

**이 노트북은 원본을 재조립한 것이다.** 계산 로직은 원본과 1글자도 다르지 않다 (유일한 예외: 사내 서버 경로 2줄 → 환경변수, 아래 첫 코드 셀 · `../재조립-설계.md` §3-A).

- 원본: `projects/CRRT/source/[최종] 코드 통합_260624.ipynb` (동결, 수정 금지)
- 원본 셀 ↔ 이 노트북 위치 대응: 각 코드 셀 바로 위 마크다운의 `원본 셀 N` 표기 참조, 표·그림 산출 근거는 `../20260804-셀산출물-대조표.md`
- 설계 근거: `../재조립-설계.md` · 수치 대조기: `../재현확인.py`
- 🔴 실행 전 `MIMIC4_PATH`·`EICU_PATH` 환경변수를 설정한다 (`README.md` 참조). 원자료는 PhysioNet 자격인증 접근 대상이며 이 저장소에 포함되지 않는다.


## 0. 공용 정의 로드

원본 노트북 **셀 0**(import·상수·함수, 부작용 없음)을 `src/common.py` 로 옮긴 것을 그대로 불러온다. 이후 모든 셀은 원본과 동일하게 이 모듈의 이름만 참조한다(재정의하지 않는다 — 원본 셀 0 원칙 그대로).


In [ ]:
import sys
from pathlib import Path

# 이 노트북(notebooks/analysis.ipynb)의 부모.부모 == public/, 그 아래 src/ 를 경로에 추가한다.
# Jupyter 기본 동작(cwd = 노트북이 있는 폴더)을 전제로 한다. README.md "실행 순서" 참조.
sys.path.insert(0, str(Path.cwd().parent / "src"))
from common import *  # noqa: F401,F403 — 원본 셀 0 과 동일하게 * import (원본도 이후 셀에서
                       # 모든 이름을 접두사 없이 직접 참조하므로 동작을 그대로 재현하려면 필요하다)


## 1. 데이터 적재 (Data loading)


*원본 셀 1*


In [ ]:
# =====================================================================
# [셀 1] MIMIC 코호트 빌더 → cohort_final_v2.parquet  (REBUILD 가드)
#   원본: 확인 노트북 셀 26. 상수·SOFA·헬퍼는 셀 0 참조.
#   빌드 전용 추출 함수(baseline_vals/peak_and_validate/closest/get_*)는
#     t0·시간창에 결합돼 다른 셀에서 재사용 안 되므로 이 셀에 둠.
#   산출: subject당 1행. t0=가장 이른 CRRT start, 그 stay만.
#     측정 보유(n_meas>0)=2층위 후보 / 측정 없음=1층위만.
#     baseline/peak(measured TMP) + 공변량(t0-24h~+3h) + 비신장 SOFA.
#   REBUILD=False면 기존 parquet 로드만. True면 raw에서 재빌드.
#   전제: 셀 0 (상수·chunked_load·cached·sofa_*)
# =====================================================================
if not REBUILD and Path(COHORT_PARQUET).exists():
    cohort = pd.read_parquet(COHORT_PARQUET)
    print(f"[셀 1] REBUILD=False → {COHORT_PARQUET} 로드: N={len(cohort)}, "
          f"cols={len(cohort.columns)}")
    print(f"  measured TMP 보유(2층위 후보): {(cohort['n_meas']>0).sum()}, "
          f"measured 없음(1층위만): {(cohort['n_meas']==0).sum()}")
else:
    print(f"[셀 1] REBUILD={REBUILD} → raw에서 코호트 빌드 시작")

    # ---- 빌드 전용 추출 함수 (이 셀에서만 사용) ----
    def baseline_vals(g, t0):
        """measured TMP baseline 3종 (first/median/min), 창=t0-24h~t0+3h."""
        if g is None or len(g) == 0: return (np.nan, np.nan, np.nan)
        sub = g[(g["charttime"] >= t0 - pd.Timedelta(hours=COV_PRE_H)) &
                (g["charttime"] <= t0 + pd.Timedelta(hours=BASE_WIN_H))]
        if len(sub) == 0: return (np.nan, np.nan, np.nan)
        sub = sub.sort_values("charttime")
        return (sub.iloc[0]["tmp"], sub["tmp"].median(), sub["tmp"].min())

    def peak_and_validate(g, t0, win_h):
        """peak(t0+3h~win_h) + 이후 DROP_FOLLOW_H 내 하락 검증."""
        nan6 = (np.nan,) * 6
        if g is None or len(g) == 0: return nan6
        seg = g[(g["charttime"] >= t0 + pd.Timedelta(hours=PEAK_START_H)) &
                (g["charttime"] <= t0 + pd.Timedelta(hours=win_h))]
        if len(seg) == 0: return nan6
        pidx = seg["tmp"].idxmax(); peak_val = seg.loc[pidx, "tmp"]
        peak_t = seg.loc[pidx, "charttime"]
        peak_h = (peak_t - t0).total_seconds() / 3600
        after = g[(g["charttime"] > peak_t) &
                  (g["charttime"] <= peak_t + pd.Timedelta(hours=DROP_FOLLOW_H))]
        no_follow = 1 if len(after) == 0 else 0
        drop50_h = drop100_h = np.nan
        if len(after) > 0:
            d50 = after[peak_val - after["tmp"] >= 50]
            d100 = after[peak_val - after["tmp"] >= 100]
            if len(d50):  drop50_h = (d50.iloc[0]["charttime"] - t0).total_seconds()/3600
            if len(d100): drop100_h = (d100.iloc[0]["charttime"] - t0).total_seconds()/3600
        return (peak_val, peak_h, no_follow, drop50_h, drop100_h, len(after))

    def closest(by, s, t0, ids, col="valuenum"):
        """t0에 가장 가까운 값 (창=t0-24h~t0+3h). by=subject별 그룹 dict."""
        if s not in by: return np.nan
        g = by[s]; ws, we = t0 - pd.Timedelta(hours=COV_PRE_H), t0 + pd.Timedelta(hours=COV_POST_H)
        sub = g[g["itemid"].isin(ids) & (g["charttime"] >= ws) & (g["charttime"] <= we)]
        if len(sub) == 0: return np.nan
        sub = sub.copy(); sub["_td"] = (sub["charttime"] - t0).abs()
        return sub.sort_values("_td").iloc[0][col]

    def rfilter(df, ids, lo, hi):
        """itemid별 생리적 범위 밖 제거."""
        m = df["itemid"].isin(ids)
        return df[~(m & ((df["valuenum"] < lo) | (df["valuenum"] > hi)))].copy()

    # ===== [1] CRRT procedure 로드 =====
    # t0 = subject별 가장 이른 CRRT start. 추적 = 그 t0가 속한 첫 CRRT stay.
    print("[1] CRRT procedure load")
    proc = pd.read_csv(MIMIC_ICU / "procedureevents.csv.gz", compression="gzip",
                       low_memory=False,
                       usecols=["subject_id", "hadm_id", "stay_id",
                                "starttime", "endtime", "itemid"])
    crrt = proc[proc["itemid"].isin(CRRT_PROC)].copy()
    crrt["starttime"] = pd.to_datetime(crrt["starttime"], errors="coerce")
    crrt["endtime"]   = pd.to_datetime(crrt["endtime"],   errors="coerce")
    crrt = crrt.dropna(subset=["starttime"])
    crrt_stays = set(crrt["stay_id"].dropna().astype("int64"))

    first_start = crrt.groupby("subject_id")["starttime"].min().rename("subj_first_start")
    crrt = crrt.merge(first_start, on="subject_id", how="left")
    first_stay = (crrt[crrt["starttime"] == crrt["subj_first_start"]]
                  .sort_values(["subject_id", "stay_id"])
                  .drop_duplicates("subject_id")[["subject_id", "stay_id"]]
                  .rename(columns={"stay_id": "first_crrt_stay"}))
    proc_start = crrt.groupby("stay_id")["starttime"].min().rename("crrt_start")
    proc_end   = crrt.groupby("stay_id")["endtime"].max().rename("crrt_end")
    print(f"  CRRT procedure stays: {len(crrt_stays):,}, "
          f"subjects: {crrt['subject_id'].nunique():,}")

    # ===== [2] measured TMP(값) + 가동 시각축(measured TMP ∪ filter_p 시각) =====
    print("[2] measured TMP (값) + 가동 시각축 (measured TMP ∪ FP charttime)")
    ce = cached("ce_tmp_press.parquet", lambda: chunked_load(
        MIMIC_ICU / "chartevents.csv.gz",
        ["stay_id", "charttime", "itemid", "valuenum"],
        [TMP_ID] + list(PRESS), crrt_stays, "stay_id"))
    ce["charttime"] = pd.to_datetime(ce["charttime"], errors="coerce")

    meas = ce[(ce["itemid"] == TMP_ID) & ce["valuenum"].notna() &
              (ce["valuenum"] >= 0) & (ce["valuenum"] <= TMP_CLIP)] \
           [["stay_id", "charttime", "valuenum"]].rename(columns={"valuenum": "tmp"})
    meas["tmp_source"] = "measured"
    print(f"  measured TMP rows {len(meas):,}, stay {meas['stay_id'].nunique():,}")

    # filter_p(224150) sanity 범위 내 기록의 시각만(값 미사용, '가동 중' 증거)
    fp = ce[(ce["itemid"] == 224150) & ce["valuenum"].notna()].copy()
    lo, hi = PRESS_BOUNDS["filter_p"]; fp = fp[fp["valuenum"].between(lo, hi)]
    fp_time = fp[["stay_id", "charttime"]].copy()
    print(f"  filter_p 시각 rows {len(fp_time):,}, stay {fp_time['stay_id'].nunique():,}")

    clock = pd.concat([meas[["stay_id", "charttime"]], fp_time], ignore_index=True) \
              .drop_duplicates(["stay_id", "charttime"]) \
              .sort_values(["stay_id", "charttime"]).reset_index(drop=True)
    print(f"  시각축(measured∪FP) rows {len(clock):,}, stay {clock['stay_id'].nunique():,}")

    # ===== [3] stay 단위 집계 =====
    print("[3] stay 단위 집계")
    ep = clock.groupby("stay_id").agg(
        ep_first_clock=("charttime", "min"),
        last_clock=("charttime", "max"),
        n_clock=("charttime", "size")).reset_index()
    n_meas = meas.groupby("stay_id").size().rename("n_meas")
    ep = ep.merge(n_meas, on="stay_id", how="left")
    ep["n_meas"] = ep["n_meas"].fillna(0).astype(int)
    FIRST_STAYS = set(first_stay["first_crrt_stay"].astype("int64"))

    # ===== [4] demographics + outcome time =====
    print("[4] merge demographics + outcome times")
    ep = ep.merge(proc_start, on="stay_id", how="left").merge(proc_end, on="stay_id", how="left")
    icu = pd.read_csv(MIMIC_ICU / "icustays.csv.gz", compression="gzip",
                      usecols=["subject_id", "hadm_id", "stay_id"])
    ep = ep.merge(icu, on="stay_id", how="left")
    pat = pd.read_csv(MIMIC_HOSP / "patients.csv.gz", compression="gzip",
                      usecols=["subject_id", "anchor_age", "gender", "dod"])
    pat["dod"] = pd.to_datetime(pat["dod"], errors="coerce")
    ep = ep.merge(pat, on="subject_id", how="left")
    ep["male"] = (ep["gender"] == "M").astype(int)
    adm = pd.read_csv(MIMIC_HOSP / "admissions.csv.gz", compression="gzip",
                      usecols=["subject_id", "hadm_id", "deathtime", "dischtime"])
    adm["deathtime"] = pd.to_datetime(adm["deathtime"], errors="coerce")
    adm["dischtime"] = pd.to_datetime(adm["dischtime"], errors="coerce")
    ep = ep.merge(adm[["hadm_id", "deathtime"]].drop_duplicates("hadm_id"),
                  on="hadm_id", how="left")
    ep["death_time_final"] = ep["deathtime"].fillna(ep["dod"])
    ep = ep.merge(adm.groupby("subject_id")["dischtime"].max().rename("last_dischtime"),
                  on="subject_id", how="left")

    ep = ep[ep["stay_id"].isin(FIRST_STAYS)].copy()
    ep["t0"] = ep["crrt_start"]
    ep["t0_to_firstclock_h"] = (ep["ep_first_clock"] - ep["t0"]).dt.total_seconds() / 3600
    ep["t0_clock_mismatch"] = ep["t0_to_firstclock_h"].abs() > 1
    ep["crrt_range_h"] = (ep["crrt_end"] - ep["t0"]).dt.total_seconds() / 3600

    # ===== [5] inclusion =====
    print("[5] inclusion")
    first_ep = ep.copy()
    n_all = len(first_ep); print(f"  subjects(첫 CRRT stay): {n_all}")
    c_age   = (first_ep["anchor_age"] >= 18)
    c_range = (first_ep["crrt_range_h"] >= MIN_CRRT_RANGE_H)
    c_align = (~first_ep["t0_clock_mismatch"])
    _dh = (first_ep["death_time_final"] - first_ep["t0"]).dt.total_seconds() / 3600
    _ih = (first_ep["last_dischtime"] - first_ep["t0"]).dt.total_seconds() / 3600
    c_time = ~((_dh <= 0) | (_ih <= 0))
    first_ep["eligible"] = c_age & c_range & c_align & c_time
    print(f"  age<18 제외                          : {(~c_age).sum()}")
    print(f"  CRRT 범위<3h 제외(age OK)             : {(c_age & ~c_range).sum()}")
    print(f"  t0-시각축 불일치(>1h) 제외(age&range OK): {(c_age & c_range & ~c_align).sum()}")
    print(f"    └ 시각축이 t0보다 6h+ 앞              : {(c_age & c_range & (first_ep['t0_to_firstclock_h']<-6)).sum()}")
    print(f"    └ 시각 기록 없어 검증 면제            : {(first_ep['t0_to_firstclock_h'].isna()).sum()}")
    print(f"  시각역전(death/disch<t0) 제외(나머지OK) : {(c_age & c_range & c_align & ~c_time).sum()}")
    cohort = first_ep[first_ep["eligible"]].copy().reset_index(drop=True)
    cohort["subject_id"] = cohort["subject_id"].astype("int64")
    cohort["stay_id"] = cohort["stay_id"].astype("int64")
    print(f"  최종 cohort: {len(cohort)}, measured 보유: {(cohort['n_meas']>0).sum()}, "
          f"measured 없음: {(cohort['n_meas']==0).sum()}")

    # ===== [6] baseline / peak (measured TMP, stay_id 기준) =====
    print("[6] baseline / peak (measured TMP)")
    cohort_stays = set(cohort["stay_id"])
    meas_by = {s: g.sort_values("charttime").reset_index(drop=True)
               for s, g in meas[meas["stay_id"].isin(cohort_stays)].groupby("stay_id")}
    src = "meas"
    bv = cohort.apply(lambda r: baseline_vals(meas_by.get(r["stay_id"]), r["t0"]), axis=1)
    cohort[f"base_first_{src}"]  = [x[0] for x in bv]
    cohort[f"base_median_{src}"] = [x[1] for x in bv]
    cohort[f"base_min_{src}"]    = [x[2] for x in bv]
    for w in PEAK_WINDOWS:
        pv = cohort.apply(lambda r: peak_and_validate(meas_by.get(r["stay_id"]), r["t0"], w), axis=1)
        cohort[f"peak_{w}h_{src}"]          = [x[0] for x in pv]
        cohort[f"peak_{w}h_time_{src}"]     = [x[1] for x in pv]
        cohort[f"peak_{w}h_nofollow_{src}"] = [x[2] for x in pv]
        cohort[f"peak_{w}h_drop50h_{src}"]  = [x[3] for x in pv]
        cohort[f"peak_{w}h_drop100h_{src}"] = [x[4] for x in pv]
        cohort[f"peak_{w}h_nafter_{src}"]   = [x[5] for x in pv]

    # ===== [7] covariates (t0-24h ~ t0+3h) =====
    print("[7] covariates")
    subs = set(cohort["subject_id"])
    chart_cov = cached("chart_cov.parquet", lambda: chunked_load(
        MIMIC_ICU / "chartevents.csv.gz",
        ["subject_id", "stay_id", "charttime", "itemid", "valuenum"],
        ALL_CHART_COV, subs, "subject_id"))
    chart_cov["charttime"] = pd.to_datetime(chart_cov["charttime"], errors="coerce")
    lab_cov = cached("lab_cov.parquet", lambda: chunked_load(
        MIMIC_HOSP / "labevents.csv.gz",
        ["subject_id", "charttime", "itemid", "valuenum"],
        ALL_LAB_IDS, subs, "subject_id"))
    lab_cov["charttime"] = pd.to_datetime(lab_cov["charttime"], errors="coerce")
    input_all = cached("input.parquet", lambda: chunked_load(
        MIMIC_ICU / "inputevents.csv.gz",
        ["subject_id", "stay_id", "starttime", "endtime", "itemid", "rate", "rateuom"],
        sorted(set(VASO_IDS + ALL_INPUT_ANTICOAG)), subs, "subject_id"))
    for c in ["starttime", "endtime"]:
        input_all[c] = pd.to_datetime(input_all[c], errors="coerce")
    chart_ac = cached("chart_ac.parquet", lambda: chunked_load(
        MIMIC_ICU / "chartevents.csv.gz",
        ["subject_id", "stay_id", "charttime", "itemid", "valuenum"],
        ALL_CHART_ANTICOAG, subs, "subject_id"))
    chart_ac["charttime"] = pd.to_datetime(chart_ac["charttime"], errors="coerce")

    def load_mv():
        p = pd.read_csv(MIMIC_ICU / "procedureevents.csv.gz", compression="gzip",
                        low_memory=False,
                        usecols=["subject_id", "stay_id", "starttime", "endtime", "itemid"])
        pm = p[p["itemid"].isin(MV_ITEM) & p["subject_id"].astype("Int64").isin(subs)].copy()
        for c in ["starttime", "endtime"]:
            pm[c] = pd.to_datetime(pm[c], errors="coerce")
        return pm
    proc_mv = cached("proc_mv.parquet", load_mv)

    # 생리적 범위 필터
    lab_f = lab_cov[lab_cov["valuenum"].notna()].copy()
    chart_f = chart_cov[chart_cov["valuenum"].notna()].copy()
    for n, s in LAB_SPEC.items():
        if s["lab"]:   lab_f   = rfilter(lab_f,   s["lab"],   s["lo"], s["hi"])
        if s["chart"]: chart_f = rfilter(chart_f, s["chart"], s["lo"], s["hi"])
    chart_f = rfilter(chart_f, MAP_ITEMS, 20, 200)
    for gn, (gi, glo, ghi) in GCS_SPEC.items(): chart_f = rfilter(chart_f, [gi], glo, ghi)
    chart_f = rfilter(chart_f, BF_ITEM, 50, 500)
    lab_by = {s: g for s, g in lab_f.groupby("subject_id")}
    chart_by = {s: g for s, g in chart_f.groupby("subject_id")}

    # weight (kg/lb 통합, 우선순위)
    wt = chart_f[chart_f["itemid"].isin(WEIGHT_KG + WEIGHT_LB)].copy()
    wt["weight_kg"] = np.where(wt["itemid"].isin(WEIGHT_LB),
                               wt["valuenum"] * 0.453592, wt["valuenum"])
    wt = wt[(wt["weight_kg"] >= 30) & (wt["weight_kg"] <= 300)].copy()
    wt["priority"] = wt["itemid"].map(WEIGHT_PRIORITY)
    wt_by = {s: g for s, g in wt.groupby("subject_id")}
    def get_wt(r):
        s, t0 = r["subject_id"], r["t0"]
        if s not in wt_by: return np.nan
        g = wt_by[s]
        for lo, cols in [(3, ["priority", "_td"]), (168, ["_td", "priority"])]:
            sub = g[(g["charttime"] >= t0 - pd.Timedelta(hours=lo)) &
                    (g["charttime"] <= t0 + pd.Timedelta(hours=lo))].copy()
            if len(sub):
                sub["_td"] = (sub["charttime"] - t0).abs()
                return sub.sort_values(cols).iloc[0]["weight_kg"]
        return np.nan
    cohort["weight_kg"] = cohort.apply(get_wt, axis=1)

    for ln, spec in LAB_SPEC.items():
        def _ex(r, _s=spec):
            v = closest(lab_by, r["subject_id"], r["t0"], _s["lab"]) if _s["lab"] else np.nan
            if pd.isna(v) and _s["chart"]:
                v = closest(chart_by, r["subject_id"], r["t0"], _s["chart"])
            return v
        cohort[ln] = cohort.apply(_ex, axis=1)
    cohort["map_value"] = cohort.apply(
        lambda r: closest(chart_by, r["subject_id"], r["t0"], MAP_ITEMS), axis=1)
    for gn, (gi, _, _) in GCS_SPEC.items():
        cohort[gn] = cohort.apply(
            lambda r, _i=gi: closest(chart_by, r["subject_id"], r["t0"], [_i]), axis=1)
    cohort["gcs_total"] = cohort["gcs_eye"] + cohort["gcs_verbal"] + cohort["gcs_motor"]

    # FiO2 (비율/퍼센트 통합) → PF ratio
    fio2 = chart_f[chart_f["itemid"].isin(FIO2_ITEM)].copy()
    fio2["v"] = np.where(fio2["valuenum"] <= 1.5, fio2["valuenum"] * 100, fio2["valuenum"])
    fio2 = fio2[(fio2["v"] >= 21) & (fio2["v"] <= 100)].copy()
    fio2_by = {s: g for s, g in fio2.groupby("subject_id")}
    def get_fio2(r):
        s, t0 = r["subject_id"], r["t0"]
        if s not in fio2_by: return np.nan
        g = fio2_by[s]; ws, we = t0 - pd.Timedelta(hours=COV_PRE_H), t0 + pd.Timedelta(hours=COV_POST_H)
        sub = g[(g["charttime"] >= ws) & (g["charttime"] <= we)]
        if len(sub) == 0: return np.nan
        sub = sub.copy(); sub["_td"] = (sub["charttime"] - t0).abs()
        return sub.sort_values("_td").iloc[0]["v"]
    cohort["fio2"] = cohort.apply(get_fio2, axis=1)
    cohort["blood_flow"] = cohort.apply(
        lambda r: closest(chart_by, r["subject_id"], r["t0"], BF_ITEM), axis=1)
    cohort["pf_ratio"] = np.where(
        cohort["pao2"].notna() & cohort["fio2"].notna() & (cohort["fio2"] > 0),
        cohort["pao2"] / (cohort["fio2"] / 100), np.nan)

    # mechanical ventilation (창 중첩)
    mv_by = {s: g for s, g in proc_mv.groupby("subject_id")}
    def get_mv(r):
        s, t0 = r["subject_id"], r["t0"]
        if s not in mv_by: return 0
        g = mv_by[s]; ws, we = t0 - pd.Timedelta(hours=COV_PRE_H), t0 + pd.Timedelta(hours=COV_POST_H)
        return int(((g["starttime"] <= we) & (g["endtime"] >= ws)).any())
    cohort["mech_vent"] = cohort.apply(get_mv, axis=1)

    # vasopressor (단위 보정 + 범위 필터 + 최대 rate)
    vi = input_all[input_all["itemid"].isin(VASO_IDS) & input_all["rate"].notna()].copy()
    vi.loc[(vi["itemid"] == 221906) & (vi["rateuom"] == "mg/kg/min"), "rate"] *= 1000
    vi.loc[(vi["itemid"] == 222315) & (vi["rateuom"] == "units/min"), "rate"] *= 60
    vi = vi[~((vi["itemid"] == 221749) & (vi["rateuom"] == "mcg/min"))].copy()
    for vn, spec in VASO_SPEC.items():
        m = vi["itemid"].isin(spec["ids"])
        vi = vi[~(m & ((vi["rate"] < 0) | (vi["rate"] > spec["max"])))]
    vaso_by = {s: g for s, g in vi.groupby("subject_id")}
    def get_vaso(r):
        s, t0 = r["subject_id"], r["t0"]
        out = {f"vaso_{v}_max": 0.0 for v in VASO_SPEC}; out["vaso_use"] = 0
        if s not in vaso_by: return pd.Series(out)
        g = vaso_by[s]; ws, we = t0 - pd.Timedelta(hours=COV_PRE_H), t0 + pd.Timedelta(hours=COV_POST_H)
        ov = g[(g["starttime"] <= we) & (g["endtime"] >= ws)]
        if len(ov) == 0: return pd.Series(out)
        out["vaso_use"] = 1
        for vn, spec in VASO_SPEC.items():
            vo = ov[ov["itemid"].isin(spec["ids"])]
            if len(vo): out[f"vaso_{vn}_max"] = vo["rate"].max()
        return pd.Series(out)
    vr = cohort.apply(get_vaso, axis=1)
    for c in vr.columns:
        if c in cohort.columns: cohort = cohort.drop(columns=[c])
    cohort = pd.concat([cohort, vr], axis=1)

    # anticoagulation 플래그 (stay 단위)
    def stays_for(df, ids):
        return set(df[df["itemid"].isin(ids)]["stay_id"].dropna().astype("int64"))
    input_all["stay_id"] = input_all["stay_id"].astype("int64")
    chart_ac["stay_id"] = chart_ac["stay_id"].astype("int64")
    for col, ss in {
        "sig_systemic":      stays_for(input_all, HEPARIN_SYSTEMIC),
        "sig_proph":         stays_for(input_all, HEPARIN_PROPH),
        "sig_crrt_hep":      stays_for(input_all, HEPARIN_CRRT),
        "sig_other_anticoag":stays_for(input_all, OTHER_ANTICOAG),
        "sig_citrate_input": stays_for(input_all, CITRATE_INPUT),
        "sig_citrate_chart": stays_for(chart_ac,  CITRATE_CHART),
        "sig_calcium_crrt":  stays_for(input_all, CALCIUM_CRRT)}.items():
        cohort[col] = cohort["stay_id"].isin(ss)
    ce224 = chart_ac[chart_ac["itemid"].isin(HEPARIN_CHART_224145) & chart_ac["valuenum"].notna()]
    pos = set(ce224.groupby("stay_id")["valuenum"].max().loc[lambda x: x > 0].index)
    cohort["sig_chart_224145_max_pos"] = cohort["stay_id"].isin(pos)
    cohort["v3_systemic_hep"] = (cohort["sig_systemic"] | cohort["sig_crrt_hep"] |
                                 cohort["sig_other_anticoag"] |
                                 cohort["sig_chart_224145_max_pos"]).astype(int)
    cohort["v3_prophylaxis"] = cohort["sig_proph"].astype(int)

    # ===== [8] 비신장 SOFA (셀 0의 sofa_* 사용) =====
    print("[8] non-renal SOFA")
    cohort["sofa_resp"]  = cohort.apply(lambda r: sofa_resp(r["pf_ratio"], r["mech_vent"]), axis=1)
    cohort["sofa_coag"]  = cohort["platelet"].apply(sofa_coag)
    cohort["sofa_liver"] = cohort["bilirubin"].apply(sofa_liver)
    cohort["sofa_cv"]    = cohort.apply(
        lambda r: sofa_cv(r["vaso_norepi_max"], r["vaso_epi_max"], r["vaso_dopa_max"],
                          r["vaso_dobu_max"], r["map_value"]), axis=1)
    cohort["sofa_cns"]   = cohort["gcs_total"].apply(sofa_cns)
    SC = ["sofa_resp", "sofa_coag", "sofa_liver", "sofa_cv", "sofa_cns"]
    cohort["sofa_total"] = cohort[SC].sum(axis=1, min_count=5)

    # ===== [QC] + 저장 =====
    print("\n" + "=" * 60); print("[셀 1] 코호트 빌드 QC"); print("=" * 60)
    chk = cohort["t0_to_firstclock_h"]
    print(f"  t0-시각축 |차이| 최대(기록有): {chk.abs().max():.3f}h (≤1 정상)")
    print(f"  ±1h 초과 잔존(0이어야): {(chk.abs() > 1).sum()}")
    print(f"  시각 기록 없어 검증면제: {chk.isna().sum()}")
    print(f"  crrt_range_h <3 (0이어야): {(cohort['crrt_range_h'] < MIN_CRRT_RANGE_H).sum()}")
    print(f"  crrt_end < t0 (0이어야): {(cohort['crrt_range_h'] < 0).sum()}")
    print(f"  base_median_meas 결측: {cohort['base_median_meas'].isna().sum()}/{len(cohort)}")
    print(f"  measured 보유(2층위): {(cohort['n_meas'] > 0).sum()}, "
          f"없음(1층위만): {(cohort['n_meas'] == 0).sum()}")
    _dh = (cohort["death_time_final"] - cohort["t0"]).dt.total_seconds() / 3600
    _ih = (cohort["last_dischtime"] - cohort["t0"]).dt.total_seconds() / 3600
    print(f"  death<t0 (시각역전): {(_dh <= 0).sum()}, disch<t0: {(_ih <= 0).sum()}")
    print("  [연속공변량 결측%]")
    for cc in ["weight_kg", "platelet", "hemoglobin", "lactate", "inr", "aptt",
               "bilirubin", "blood_flow", "map_value", "sofa_total", "base_median_meas"]:
        if cc in cohort.columns:
            n = cohort[cc].isna().sum()
            print(f"    {cc:<18}: {n}/{len(cohort)} ({n/len(cohort)*100:.1f}%)")
    print("=" * 60)

    cohort.to_parquet(COHORT_PARQUET, index=False)
    print(f"\nSaved {COHORT_PARQUET}: N={len(cohort)}, cols={len(cohort.columns)}")

*원본 셀 2*


In [ ]:
# =====================================================================
# [셀 2] measured TMP 타당성 검증 (supplement)  (REBUILD 가드)
#   원본: 확인 노트북 셀 1~5 통합.
#   목적: measured TMP(229247)가 회로 압력 표준식 재구성값과 일치하는지 →
#         surge 노출을 measured TMP에 둘 정당성. 본문 측정 정의의 근거.
#   표준식: tmp_calc = (filter_p + return_p)/2 - effluent_p  (Prismaflex)
#   산출: Pearson/Spearman r, OLS slope·r², Bland-Altman bias·LoA,
#         재구성 가능 시점/환자 수.
#   이 셀에서만 쓰는 검증 전용 itemid(access_p·pressure_drop)는 로컬 상수.
#   분석에 영향 없음(measured TMP는 빌더가 이미 추출). 무거우므로 REBUILD 가드.
#   전제: 셀 0 (TMP_ID, PRESS, PRESS_BOUNDS, chunked_load, linregress)
# =====================================================================
RUN_TMP_VALIDATION = REBUILD   # 기본은 빌드 시에만. 단독 실행하려면 True로.

if not RUN_TMP_VALIDATION:
    print("[셀 2] RUN_TMP_VALIDATION=False → TMP 타당성 검증 건너뜀 "
          "(measured TMP는 빌더가 추출 완료). 실행하려면 True로.")
else:
    print("[셀 2] measured TMP 타당성 검증 시작")

    # 검증 전용 itemid (셀 0 PRESS + access_p·pressure_drop·tmp_recorded)
    VAL_ITEMS = {224149: "access_p", 224150: "filter_p", 224151: "effluent_p",
                 224152: "return_p", TMP_ID: "tmp_recorded", 229248: "pressure_drop"}
    VAL_BOUNDS = {"access_p": (-300, 50), "filter_p": (-50, 500),
                  "effluent_p": (-300, 300), "return_p": (-50, 500),
                  "tmp_recorded": (-50, 500), "pressure_drop": (-50, 500)}

    # ---- 로드 (전체 stay, 검증 itemid) ----
    _val = cached("tmp_validation.parquet", lambda: chunked_load(
        MIMIC_ICU / "chartevents.csv.gz",
        ["stay_id", "charttime", "itemid", "valuenum"],
        list(VAL_ITEMS), subject_filter=None, id_col="stay_id"))
    _val["charttime"] = pd.to_datetime(_val["charttime"], errors="coerce")
    _val["item"] = _val["itemid"].map(VAL_ITEMS)
    print("  [항목별 값 분포]")
    print(_val.groupby("item")["valuenum"].describe().round(1).to_string())

    # ---- 이상치 제거 + wide pivot ----
    d = _val.copy()
    for item, (lo, hi) in VAL_BOUNDS.items():
        m = d["item"].eq(item)
        d.loc[m & (~d["valuenum"].between(lo, hi)), "valuenum"] = np.nan
    d = d.dropna(subset=["valuenum"])
    wide = (d.pivot_table(index=["stay_id", "charttime"], columns="item",
                          values="valuenum", aggfunc="mean").reset_index())

    # ---- 표준식 재구성 + 일치도 ----
    wide["tmp_calc"] = (wide["filter_p"] + wide["return_p"]) / 2 - wide["effluent_p"]
    pair = wide.dropna(subset=["tmp_calc", "tmp_recorded"]).copy()
    pair["diff"] = pair["tmp_calc"] - pair["tmp_recorded"]

    print("\n" + "=" * 60); print("  [A] measured vs 표준식 재구성 일치도"); print("=" * 60)
    print(f"  비교 가능 페어: {len(pair):,}")
    r_p = pair["tmp_calc"].corr(pair["tmp_recorded"])
    r_s = pair["tmp_calc"].corr(pair["tmp_recorded"], method="spearman")
    print(f"  Pearson r = {r_p:.4f}, Spearman = {r_s:.4f}")
    bias = pair["diff"].mean(); sd = pair["diff"].std()
    print(f"  Bland-Altman: bias={bias:.1f}, LoA [{bias-1.96*sd:.1f}, {bias+1.96*sd:.1f}]")

    # 극단 0.1% 제거 후 OLS
    q_lo, q_hi = pair["diff"].quantile([0.001, 0.999])
    p_ols = pair[pair["diff"].between(q_lo, q_hi)]
    ols = linregress(p_ols["tmp_calc"], p_ols["tmp_recorded"])
    print(f"  OLS: TMP_recorded = {ols.slope:.4f}·TMP_calc + {ols.intercept:.2f}")
    print(f"       r²={ols.rvalue**2:.4f}, slope 95%CI "
          f"[{ols.slope-1.96*ols.stderr:.4f}, {ols.slope+1.96*ols.stderr:.4f}]")

    # ---- 후보 B: pressure_drop 절반 보정 ----
    b = wide.dropna(subset=["filter_p", "return_p", "effluent_p",
                            "pressure_drop", "tmp_recorded"]).copy()
    if len(b):
        b["tmp_B"] = ((b["filter_p"] + b["return_p"]) / 2 - b["effluent_p"]
                      - b["pressure_drop"] / 2)
        b["diff_B"] = b["tmp_B"] - b["tmp_recorded"]
        print(f"\n  [후보 B: pressure_drop/2 보정] bias={b['diff_B'].mean():.1f}, "
              f"Pearson={b['tmp_B'].corr(b['tmp_recorded']):.4f} (n={len(b):,})")

    # ---- 재구성 가능성(measured 없이 압력만 있는 시점/환자) ----
    print("\n" + "=" * 60); print("  [B] 재구성 가능성 (measured 결측 보완 여지)"); print("=" * 60)
    need = ["filter_p", "return_p", "effluent_p"]
    has_inputs = wide[need].notna().all(axis=1)
    has_tmp = wide["tmp_recorded"].notna()
    print(pd.crosstab(has_inputs, has_tmp,
                      rownames=["3압력 완비"], colnames=["TMP 기록"]))
    recon = wide[has_inputs & ~has_tmp]
    print(f"  TMP 결측 & 3압력 완비 시점: {len(recon):,}, "
          f"해당 stay: {recon['stay_id'].nunique()} / {wide['stay_id'].nunique()}")
    tmp_stays = set(wide.loc[has_tmp, "stay_id"])
    input_stays = set(wide.loc[has_inputs, "stay_id"])
    print(f"  TMP 한 번도 없으나 압력은 있는 stay: {len(input_stays - tmp_stays)}")
    print("=" * 60)
    print("  → measured TMP는 표준식 재구성과 높은 일치(r²↑) → surge 노출 근거 타당")

## 2. 코호트 구성 (Cohort construction)


*원본 셀 3*


In [ ]:
# =====================================================================
# [셀 3] MIMIC 공용 데이터: 코호트 c · measured TMP · cmeas · MICE 세트
#   생성: c(1층위 전체), tmp(measured TMP h축), cmeas(2층위), XC_C, XC_MEAS
#   변경점:
#     - sofa_total은 빌더(셀1)가 저장한 값을 그대로 사용(재계산 제거).
#     - MICE는 셀0 mice_impute() = 결과변수(event)+Nelson-Aalen 누적위험 포함
#       (White & Royston 2009). 기존 공변량-only 대치 → 표준 대치로 교체.
#     - measured TMP 추출은 cached로 1회만(반복 실행 시 chartevents 재읽기 방지).
#   전제: 셀 0 (상수·mice_impute·chunked_load·cached), 셀 1 (cohort parquet)
# =====================================================================
# ---- 코호트 로드 + 종점 ----
c = pd.read_parquet(COHORT_PARQUET)
for col in ["t0", "last_dischtime", "death_time_final"]:
    if col in c.columns: c[col] = pd.to_datetime(c[col], errors="coerce")
c["stay_id"] = c["stay_id"].astype("int64")

# sofa_total: 빌더 저장값 사용. 결측 시에만 구성요소 합으로 보정(중복 재계산 아님).
SOFA_COMP = ["sofa_resp", "sofa_coag", "sofa_liver", "sofa_cv", "sofa_cns"]
assert "sofa_total" in c.columns, "sofa_total 없음(빌더 산출물 확인)"
assert all(x in c.columns for x in SOFA_COMP), "SOFA 구성요소 없음"
_na_sofa = c["sofa_total"].isna()
if _na_sofa.any():
    c.loc[_na_sofa, "sofa_total"] = (c.loc[_na_sofa, SOFA_COMP]
                                     .apply(pd.to_numeric, errors="coerce")
                                     .sum(axis=1, min_count=5))

c["death_h"] = (c["death_time_final"] - c["t0"]).dt.total_seconds() / 3600
c["disch_h"] = (c["last_dischtime"] - c["t0"]).dt.total_seconds() / 3600

# 28d 사망: 사망 우선, 28d내 퇴원 censoring, 아니면 672h
c["event28"] = ((c["death_h"].notna()) & (c["death_h"] <= FU_H)).astype(int)
end28 = np.minimum(c["death_h"].fillna(np.inf), FU_H)
cz = (c["event28"] == 0) & c["disch_h"].notna() & (c["disch_h"] < end28)
end28 = end28.copy(); end28[cz] = c["disch_h"][cz]; c["end28_h"] = end28

# 원내 사망: 퇴원 전(또는 동시) 사망만 event=1. 종료=min(사망,퇴원). 시점제한 없음.
death_in = (c["death_h"].notna() & c["disch_h"].notna() & (c["death_h"] <= c["disch_h"]))
death_in = death_in | (c["death_h"].notna() & c["disch_h"].isna())  # 퇴원정보 없는 사망=원내(보수적)
c["death_inhosp"] = death_in.astype(int)
c["end_inhosp_h"] = c[["death_h", "disch_h"]].min(axis=1, skipna=True)

c = c[c["end28_h"] > 0].reset_index(drop=True)

# ---- 공변량 가용성 점검 (CONT_M/BIN_M은 셀0 정의. 결측 전부인 변수만 제외) ----
CONT_M_use = [x for x in CONT_M if x in c.columns and not c[x].isna().all()]
BIN_M_use  = [x for x in BIN_M if x in c.columns]
for x in BIN_M_use:
    c[x] = pd.to_numeric(c[x], errors="coerce").fillna(0).astype(int)
assert CONT_M_use == CONT_M, f"CONT_M 일부 누락: {set(CONT_M)-set(CONT_M_use)}"
assert BIN_M_use == BIN_M,   f"BIN_M 일부 누락: {set(BIN_M)-set(BIN_M_use)}"

# ---- measured TMP 추출 (빌더 ce_tmp_press.parquet 재사용, 빌더와 동일 조건 >=0) ----
def _load_tmp():
    ce = pd.read_parquet(CACHE / "ce_tmp_press.parquet")
    ce["charttime"] = pd.to_datetime(ce["charttime"], errors="coerce")
    m = ce[(ce["itemid"] == TMP_ID) & ce["valuenum"].notna() &
           (ce["valuenum"] >= 0) & (ce["valuenum"] <= TMP_CLIP)]
    return m[["stay_id", "charttime", "valuenum"]].copy()
tmp_all = cached("tmp_haxis.parquet", _load_tmp)

# 코호트 stay 한정 + h축. dtype 안전하게: charttime·t0 모두 tz-naive datetime으로 통일.
tmp = tmp_all[tmp_all["stay_id"].isin(set(c["stay_id"]))].copy()
tmp["charttime"] = pd.to_datetime(tmp["charttime"], errors="coerce")
try:
    tmp["charttime"] = tmp["charttime"].dt.tz_localize(None)
except (TypeError, AttributeError):
    pass  # 이미 tz-naive
_t0 = pd.to_datetime(c.set_index("stay_id")["t0"]).dt.tz_localize(None) \
      if pd.to_datetime(c["t0"]).dt.tz is not None else c.set_index("stay_id")["t0"]
_t0_mapped = pd.to_datetime(tmp["stay_id"].map(_t0))
tmp["h"] = (tmp["charttime"].values - _t0_mapped.values) / np.timedelta64(1, "h")
tmp = tmp.dropna(subset=["h"]).sort_values(["stay_id", "h"]).reset_index(drop=True)

# ---- cmeas (정의 A): baseline(-24~+3h) ∩ 3h이후 ∩ base_median_meas 보유 ----
_fb = tmp.assign(_b=(tmp["h"] >= -24) & (tmp["h"] <= 3), _p=(tmp["h"] > 3))
_agg = _fb.groupby("stay_id").agg(has_base=("_b", "any"), has_post3=("_p", "any"))
elig_A = set(_agg.index[_agg["has_base"] & _agg["has_post3"]])
assert "base_median_meas" in c.columns, "base_median_meas 없음(빌더 산출물 확인)"
cmeas = c[c["stay_id"].isin(elig_A) & c["base_median_meas"].notna()].reset_index(drop=True)
cmeas["event28"] = ((cmeas["death_h"].notna()) & (cmeas["death_h"] <= FU_H)).astype(int)
_end = np.minimum(cmeas["death_h"].fillna(np.inf), FU_H)
_cz = (cmeas["event28"] == 0) & cmeas["disch_h"].notna() & (cmeas["disch_h"] < _end)
_end = _end.copy(); _end[_cz] = cmeas["disch_h"][_cz]; cmeas["end28_h"] = _end
cmeas = cmeas[cmeas["end28_h"] > 0].reset_index(drop=True)

# ---- MICE 대치 (셀0 mice_impute: event + Nelson-Aalen 누적위험 포함) ----
#   1층위(c)·2층위(cmeas) 각각 28d 종점 기준 1회 생성 → 이후 셀 재사용(재대치 금지).
XC_C    = mice_impute(c,     CONT_M, "event28", "end28_h")
XC_MEAS = mice_impute(cmeas, CONT_M, "event28", "end28_h")

# ---- QC ----
print("=" * 68); print("[셀 3] MIMIC 공용 데이터 QC"); print("=" * 68)
print(f"  코호트 N={len(c)}, 28d 사망={int(c['event28'].sum())} "
      f"({c['event28'].mean()*100:.1f}%), 원내 사망={int(c['death_inhosp'].sum())}")
print(f"  공변량: CONT_M {len(CONT_M)}개, BIN_M {len(BIN_M)}개")
print(f"  measured TMP: rows={len(tmp):,}, stay={tmp['stay_id'].nunique()}")
print(f"  base_median_meas 보유={int(c['base_median_meas'].notna().sum())}")
print(f"  cmeas(정의 A)={len(cmeas)}, 28d 사망={int(cmeas['event28'].sum())} "
      f"({cmeas['event28'].mean()*100:.1f}%)")
print(f"  MICE: XC_C {len(XC_C)}세트 {XC_C[0].shape}, "
      f"XC_MEAS {len(XC_MEAS)}세트 {XC_MEAS[0].shape} (event+NA 누적위험 포함)")
_miss = c[CONT_M].isna().mean().mul(100).round(1)
print("  [연속공변량 결측%] " + ", ".join(f"{k}={v}" for k, v in _miss.items()))

# ---- 시각화: measured TMP 시간분포 (t0 정합 QC) ----
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(tmp["h"], bins=80, color="#34495e")
ax[0].axvline(0, ls="--", color="#c0392b", lw=1)
ax[0].axvline(3, ls=":", color="#7f8c8d", lw=1); ax[0].set_xlim(-1, 72)
ax[0].set_xlabel("Hours from t0"); ax[0].set_ylabel("measured TMP count")
ax[0].set_title("measured TMP timing vs t0 (-1~72h)", fontsize=11, loc="left")
npp = tmp.groupby("stay_id").size()
ax[1].hist(npp.clip(upper=150), bins=30, color="#16a085")
ax[1].set_xlabel("measurements per patient (clip 150)"); ax[1].set_ylabel("patients")
ax[1].set_title(f"measurements/patient (median={int(npp.median())})", fontsize=11, loc="left")
plt.tight_layout(); plt.savefig(FIGDIR / "qc_tmp_timing.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"  저장: {FIGDIR/'qc_tmp_timing.png'}")

*원본 셀 5*


In [ ]:
assert isinstance(c, pd.DataFrame), f"c가 DataFrame이 아님: {type(c)}"
print(f"c 복구 확인: {type(c).__name__}, N={len(c)}, has_base 컬럼={'has_base' in c.columns}")

*원본 셀 7*


In [ ]:
# =====================================================================
# [셀 4] sepsis-associated 플래그 + 연도그룹 (c, cmeas 양쪽 부착)
#   sepsis3 = 감염의심(배양↔항생제 시간결합, Seymour 2016) & t0-SOFA≥2
#     - 항생제: emar ∪ prescriptions / 배양: microbiologyevents
#     - 항생제 먼저→+24h내 배양, 배양 먼저→+72h내 항생제
#     - SOFA는 감염의심 시점 재계산 아닌 t0-SOFA 사용(시점 근사). 한계 기재.
#   변경점: 항생제 키워드는 셀0 ABX_KW 참조(중복 제거).
#   ★ A5: sepsis3가 코호트의 대부분(>90%)이면 "하위군"이 아니라 사실상 전체.
#     본문은 "패혈증 하위군에서도 유지"를 과장 말고 "거의 전체가 sepsis-associated"
#     로 정직하게 기술 → 그 판단 근거를 출력에 명시.
#   재사용 가드: c에 sepsis3·year_grp 있으면 추출 건너뜀(무거운 emar 재읽기 방지).
#   전제: 셀 0 (ABX_KW), 셀 3 (c, cmeas, sofa_total)
# =====================================================================
_need_sepsis = "sepsis3" not in c.columns
_need_year   = "year_grp" not in c.columns

if _need_sepsis:
    _hadms = set(c["hadm_id"].dropna().astype(int))
    def _abx_hits(df, col):
        dn = df[col].astype(str).str.lower(); m = pd.Series(False, index=df.index)
        for a in ABX_KW: m = m | dn.str.contains(a, regex=False)
        return m
    _abx = []
    for ch in pd.read_csv(MIMIC_HOSP / "emar.csv.gz", chunksize=1_000_000,
            low_memory=False, compression="gzip",
            usecols=["hadm_id", "charttime", "medication"]):
        ch = ch[ch["hadm_id"].isin(_hadms) & ch["medication"].notna()]
        if len(ch) == 0: continue
        ch = ch[_abx_hits(ch, "medication")]
        if len(ch): _abx.append(ch[["hadm_id", "charttime"]].rename(columns={"charttime": "t"}))
    for ch in pd.read_csv(MIMIC_HOSP / "prescriptions.csv.gz", chunksize=1_000_000,
            low_memory=False, compression="gzip",
            usecols=["hadm_id", "starttime", "drug"]):
        ch = ch[ch["hadm_id"].isin(_hadms) & ch["drug"].notna()]
        if len(ch) == 0: continue
        ch = ch[_abx_hits(ch, "drug")]
        if len(ch): _abx.append(ch[["hadm_id", "starttime"]].rename(columns={"starttime": "t"}))
    _abxall = pd.concat(_abx, ignore_index=True)
    _abxall["t"] = pd.to_datetime(_abxall["t"], errors="coerce"); _abxall = _abxall.dropna()

    _mbl = []
    for ch in pd.read_csv(MIMIC_HOSP / "microbiologyevents.csv.gz", chunksize=1_000_000,
            low_memory=False, compression="gzip",
            usecols=["hadm_id", "charttime", "chartdate"]):
        ch = ch[ch["hadm_id"].isin(_hadms)]
        if len(ch): _mbl.append(ch)
    _mb = pd.concat(_mbl, ignore_index=True)
    _mb["cx_time"] = pd.to_datetime(_mb["charttime"], errors="coerce").fillna(
                     pd.to_datetime(_mb["chartdate"], errors="coerce"))
    _mb = _mb.dropna(subset=["cx_time"])[["hadm_id", "cx_time"]]

    _cand = set(_abxall["hadm_id"]) & set(_mb["hadm_id"])
    _ag = _abxall.groupby("hadm_id")["t"].apply(lambda s: s.values)
    _xg = _mb.groupby("hadm_id")["cx_time"].apply(lambda s: s.values)
    def _suspect(h):
        a = _ag.get(h); x = _xg.get(h)
        if a is None or x is None: return False
        for at in a:
            if ((x >= at) & (x <= at + np.timedelta64(24, "h"))).any(): return True
        for xt in x:
            if ((a >= xt) & (a <= xt + np.timedelta64(72, "h"))).any(): return True
        return False
    _susp = {h: _suspect(h) for h in _cand}
    c["infection_susp"] = c["hadm_id"].map(lambda h: 1 if _susp.get(h, False) else 0)
    c["sepsis3"] = ((c["infection_susp"] == 1) &
                    (pd.to_numeric(c["sofa_total"], errors="coerce") >= 2)).astype(int)
    print(f"[sepsis] 감염의심 {int(c['infection_susp'].sum())} "
          f"({c['infection_susp'].mean()*100:.0f}%), sepsis3 {int(c['sepsis3'].sum())} "
          f"({c['sepsis3'].mean()*100:.0f}%)")
else:
    print(f"[재사용] sepsis3 이미 존재 ({int(c['sepsis3'].sum())}명)")

if _need_year:
    _pt = pd.read_csv(MIMIC_HOSP / "patients.csv.gz", compression="gzip",
                      usecols=["subject_id", "anchor_year_group"])
    _ymap = _pt.set_index("subject_id")["anchor_year_group"].to_dict()
    c["year_grp"] = c["subject_id"].map(_ymap)
    print("[year_grp] join 완료")
else:
    print("[재사용] year_grp 이미 존재")

# ---- cmeas에 부착 (c의 부분집합 → map. cmeas N=862 유지) ----
_cmap = c.set_index("stay_id")
for col in ["sepsis3", "infection_susp", "year_grp"]:
    if col in c.columns:
        cmeas[col] = cmeas["stay_id"].map(_cmap[col])

# ---- QC + A5 판단 ----
print("\n" + "=" * 68); print("[셀 4] sepsis3 / year_grp QC"); print("=" * 68)
_sep_c = c["sepsis3"].mean() * 100
_sep_m = cmeas["sepsis3"].mean() * 100
print(f"  [c]     N={len(c)}  감염의심={int(c['infection_susp'].sum())} "
      f"({c['infection_susp'].mean()*100:.0f}%)  sepsis3={int(c['sepsis3'].sum())} ({_sep_c:.0f}%)")
print(f"  [cmeas] N={len(cmeas)}  sepsis3={int(cmeas['sepsis3'].sum())} ({_sep_m:.0f}%)")
print(f"  cmeas sepsis3 결측={int(cmeas['sepsis3'].isna().sum())}, "
      f"year_grp 결측={int(cmeas['year_grp'].isna().sum())}")
print(f"\n  [c year_grp 분포]     {c['year_grp'].value_counts().sort_index().to_dict()}")
print(f"  [cmeas year_grp 분포] {cmeas['year_grp'].value_counts().sort_index().to_dict()}")

# sepsis3 군 vs 비sepsis 28d 사망률 (subgroup이 분리력이 있는지)
_d1 = c[c["sepsis3"] == 1]["event28"].mean() * 100
_d0 = c[c["sepsis3"] == 0]["event28"].mean() * 100
print(f"\n  sepsis3 28d 사망률: {_d1:.1f}% vs 비sepsis {_d0:.1f}% (차이 {_d1-_d0:+.1f}%p)")

# ★ A5 명시: sepsis 비율이 높으면 '하위군'이 아니라 사실상 전체
print("\n" + "-" * 68)
print("  [A5 해석 가이드 — 본문 작성 시 반영]")
if _sep_m >= 90:
    print(f"  cmeas의 sepsis-associated 비율 {_sep_m:.0f}% (≥90%).")
    print(f"  → 이는 '패혈증 특이적 하위군'이 아니라 '코호트의 대부분이 sepsis-associated'.")
    print(f"  → 본문은 '하위군에서도 유지'로 강조하지 말고, '본 코호트의 {_sep_m:.0f}%가")
    print(f"     sepsis-associated에 해당하여 sepsis 분석은 사실상 전체 코호트를 반영한다'")
    print(f"     로 정직하게 기술. subgroup 분석이 아니라 코호트 특성으로 다룰 것.")
    print(f"  → 사망률 차이도 {_d1-_d0:+.1f}%p로 미미 → sepsis 여부가 예후를 가르지 않음.")
else:
    print(f"  cmeas sepsis-associated {_sep_m:.0f}% (<90%) → 하위군 분석으로 해석 가능.")
print("=" * 68)

print("cmeas 감염의심:", int(cmeas["infection_susp"].sum()))
print("cmeas sepsis3 :", int(cmeas["sepsis3"].sum()))

# ---- 시각화: year_grp 분포 (c vs cmeas), sepsis3 비율 ----
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
yc = c["year_grp"].value_counts().sort_index()
ym = cmeas["year_grp"].value_counts().reindex(yc.index, fill_value=0)
x = np.arange(len(yc)); w = 0.38
ax[0].bar(x - w/2, yc.values/yc.sum()*100, w, label=f"c (N={len(c)})", color="#34495e")
ax[0].bar(x + w/2, ym.values/ym.sum()*100, w, label=f"cmeas (N={len(cmeas)})", color="#16a085")
ax[0].set_xticks(x); ax[0].set_xticklabels(yc.index, rotation=30, ha="right", fontsize=8)
ax[0].set_ylabel("% of cohort"); ax[0].legend(fontsize=9)
ax[0].set_title("anchor_year_group: c vs cmeas", fontsize=11, loc="left")
grp = ["c", "cmeas"]; val = [_sep_c, _sep_m]
ax[1].bar(grp, val, color=["#34495e", "#16a085"], width=0.5)
for i, v in enumerate(val): ax[1].text(i, v + 1, f"{v:.0f}%", ha="center", fontsize=10)
ax[1].set_ylabel("sepsis-associated %"); ax[1].set_ylim(0, 100)
ax[1].set_title("sepsis-associated proportion", fontsize=11, loc="left")
plt.tight_layout(); plt.savefig(FIGDIR / "qc_sepsis_year.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"  저장: {FIGDIR/'qc_sepsis_year.png'}")

*원본 셀 8*


In [ ]:
# =====================================================================
# [셀 5] 2층위 모집단 진단 (cmeas=862 구성 확인, 새 객체 생성 없음)
#   정의 A = baseline(-24~+3h) ∩ 3h이후(무제한) ∩ base_median_meas 보유
#   분석 아님: 흐름도(Fig 1)·대상자 선정 근거. cmeas/tmp/c 재사용.
#   정합 검증: 여기 정의 A 집합 == 셀 3 cmeas 집합 (862 일치 확인).
#   전제: 셀 0, 셀 3 (c, tmp, cmeas)
# =====================================================================
_fb = tmp.assign(_base=(tmp["h"] >= -24) & (tmp["h"] <= 3),
                 _post3=(tmp["h"] > 3),
                 _post12=(tmp["h"] > 3) & (tmp["h"] <= 12))
chk2 = _fb.groupby("stay_id").agg(
    has_base=("_base", "any"), has_post3=("_post3", "any"),
    has_post12=("_post12", "any"), n_meas=("h", "size")).reset_index()
chk2 = chk2[chk2["stay_id"].isin(set(c["stay_id"]))]

m = c[["stay_id", "base_median_meas"]].merge(chk2, on="stay_id", how="left")
for col in ["has_base", "has_post3", "has_post12"]:
    m[col] = m[col].fillna(False)

print("=" * 64); print("[셀 5] 2층위 모집단 진단 (Fig 1 흐름)"); print("=" * 64)
print(f"  전체 코호트 N: {len(c)}")
print(f"  measured TMP 보유(tmp에 있음): {int(m['n_meas'].notna().sum())}")
print(f"  base_median_meas 보유(빌더): {int(m['base_median_meas'].notna().sum())}")
print(f"  has_base(-24~+3h 측정): {int(m['has_base'].sum())}")
print()
print(f"  baseline ∩ 3h이후(무제한): {int((m['has_base'] & m['has_post3']).sum())}")
print(f"  baseline ∩ 3~12h:         {int((m['has_base'] & m['has_post12']).sum())}")
print()
elig_A = m["has_base"] & m["has_post3"] & m["base_median_meas"].notna()
elig_B = m["has_base"] & m["has_post12"] & m["base_median_meas"].notna()
print(f"  [정의 A] baseline ∩ 3h이후 ∩ base_median 보유: {int(elig_A.sum())}명  ← cmeas")
print(f"  [정의 B] baseline ∩ 3~12h ∩ base_median 보유:  {int(elig_B.sum())}명")
print()
only_base = m["has_base"] & ~m["has_post3"]
print(f"  baseline 있으나 3h이후 측정 없음(제외): {int(only_base.sum())}명")
if only_base.sum() > 0:
    print(f"    이들 n_meas: median={m.loc[only_base,'n_meas'].median():.0f}, "
          f"max={int(m.loc[only_base,'n_meas'].max())}")

# 정합 확인: 셀 3 cmeas와 여기 정의 A가 같은 집합인지
_setA = set(m.loc[elig_A, "stay_id"]); _setC = set(cmeas["stay_id"])
print(f"\n  [정합] 정의 A 집합 == 셀3 cmeas 집합: {_setA == _setC} "
      f"(정의A {len(_setA)}, cmeas {len(_setC)}, 차집합 {len(_setA ^ _setC)})")
_mismatch = int((m["base_median_meas"].notna() != m["has_base"]).sum())
print(f"  [정합] base_median 보유 != has_base: {_mismatch}명 "
      f"(0이 이상적, 빌더 baseline 창과 진단 창 -24~+3h 일치 여부)")
assert _setA == _setC, f"정의 A != cmeas (차집합 {len(_setA ^ _setC)}) — 정의 불일치"
print("=" * 64)

# ---- 시각화: 측정 보유 단계별 환자 수 (선정 흐름) ----
steps = ["Cohort\n(Tier 1)", "measured\nTMP", "base_median\nmeas", "def A\n(cmeas)"]
vals = [len(c), int(m["n_meas"].notna().sum()),
        int(m["base_median_meas"].notna().sum()), int(elig_A.sum())]
fig, ax = plt.subplots(figsize=(7, 4.2))
bars = ax.bar(steps, vals, color=["#34495e", "#2980b9", "#16a085", "#c0392b"], width=0.6)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 15, str(v), ha="center", fontsize=11)
ax.set_ylabel("patients"); ax.set_title("Tier 2 (TMP surge) cohort selection", fontsize=11, loc="left")
plt.tight_layout(); plt.savefig(FIGDIR / "l2_selection.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"  저장: {FIGDIR/'l2_selection.png'}")

*원본 셀 11*


In [ ]:
print("cmeas N:", len(cmeas))
print("감염의심:", int(cmeas["infection_susp"].sum()))
print("sofa_total 결측:", int(cmeas["sofa_total"].isna().sum()))
print("sofa_total>=2 (결측제외):", int((pd.to_numeric(cmeas["sofa_total"],errors='coerce')>=2).sum()))
print("감염의심 & sofa>=2:", int(((cmeas["infection_susp"]==1)&(pd.to_numeric(cmeas["sofa_total"],errors='coerce')>=2)).sum()))
print("감염의심 & (sofa>=2 or sofa결측):", int(((cmeas["infection_susp"]==1)&((pd.to_numeric(cmeas["sofa_total"],errors='coerce')>=2)|(cmeas["sofa_total"].isna()))).sum()))

## 3. 주분석 — Tier 1: 누적 CRRT 가동시간 (Primary analysis, Tier 1)


*원본 셀 4*


In [ ]:
# =====================================================================
# [셀 3A] 1층위: 누적 CRRT 가동시간(per 24h) → 사망  (MIMIC 12칸 + eICU 4칸)
#   ★ 통합본 누락분 복원. 셀 3 다음, 셀 4(sepsis) 앞에 위치시킬 것.
#   노출: 누적 가동시간(TVC 연속). on/off·필터수명 제외(역인과·측정불가).
#   MIMIC: 소스(procedure/투입물union)×gap(3/6/12h)×종점(28d/원내)=12칸.
#   eICU : 기록수(≥2/≥3)×종료(마지막/+1gap)=4칸 (I/O proxy, 대리변수).
#   변경점: 로컬 build_grid/run_* 제거 → 셀0 fit_grid_cox·build_segments 사용.
#   1층위 null 강건성: BF 대신 정밀도 논거(연속 TVC라 BIC-BF 수치 불안정→부적합).
#     BF는 2층위(surge, 이분 노출)에만 적용(셀 6).
#   전역 산출: SEG_MAIN(=procedure/gap6 segs) → 셀15 B부가 재사용.
#   전제: 셀0(build_segments,fit_grid_cox,pool_hr,forest_plot,
#         CRRT_PROC,CRRT_IN,CONT_E,BIN_E), 셀3(c,XC_C,CONT_M,BIN_M)
# =====================================================================
import os

# 원내 종점 90일 cap
CAP_INHOSP = 90 * 24  # 2160h
c["end_inhosp_h_capped"] = c["end_inhosp_h"].clip(upper=CAP_INHOSP)
# 90일 초과 사망은 90일 시점에 censoring (그 시점까지 생존)
c["death_inhosp_capped"] = c["death_inhosp"].where(c["end_inhosp_h"] <= CAP_INHOSP, 0)
print(f"원내 사망 cap 전 {c.death_inhosp.sum()} → cap 후 {c.death_inhosp_capped.sum()}")
print(f"end_inhosp_h max: {c.end_inhosp_h.max()/24:.0f}일 → cap {c.end_inhosp_h_capped.max()/24:.0f}일")

# ---- 가동구간 원자료 (1층위 전용, 1회 로드) ----
_t0map = c.set_index("stay_id")["t0"]
pe = pd.read_csv(os.path.join(MIMIC_ICU, "procedureevents.csv.gz"), compression="gzip",
                 usecols=["stay_id", "starttime", "endtime", "itemid"])
pe = pe[pe["itemid"].isin(CRRT_PROC) & pe["stay_id"].isin(set(c["stay_id"]))].copy()
pe["starttime"] = pd.to_datetime(pe["starttime"]); pe["endtime"] = pd.to_datetime(pe["endtime"])
pe = pe[pe["endtime"] > pe["starttime"]]
iv = pd.read_csv(os.path.join(MIMIC_ICU, "inputevents.csv.gz"), compression="gzip", low_memory=False,
                 usecols=["stay_id", "starttime", "endtime", "itemid"])
iv = iv[iv["itemid"].isin(CRRT_IN) & iv["stay_id"].isin(set(c["stay_id"]))].copy()
iv["starttime"] = pd.to_datetime(iv["starttime"]); iv["endtime"] = pd.to_datetime(iv["endtime"])
iv = iv[iv["endtime"] > iv["starttime"]]
print(f"[셀 3A] 가동구간 원자료: procedure {len(pe)}건/{pe['stay_id'].nunique()}명, "
      f"투입물 {len(iv)}건/{iv['stay_id'].nunique()}명")

# ---- MIMIC 12칸 (셀0 fit_grid_cox, builder='mimic', XC_C 재사용) ----
print("\n" + "=" * 84)
print("  MIMIC 1층위 누적 가동시간(per 24h) → 사망 : 소스×gap×종점 12칸")
print("=" * 84)
print(f"  {'소스':<14}{'gap':>5}{'종점':>10}{'HR':>9}{'95%CI':>18}{'p':>9}{'N':>7}{'ev':>6}")
SOURCES = {"procedure": pe, "투입물union": iv}
ENDPOINTS = [("28d", "end28_h", "event28"), ("원내", "end_inhosp_h_capped", "death_inhosp_capped")]
rows = []; seg_cache = {}
for sname, raw in SOURCES.items():
    for gap in [3, 6, 12]:
        segs = build_segments(raw, _t0map, gap); seg_cache[(sname, gap)] = segs
        for ename, ecol, evcol in ENDPOINTS:
            _mask = c[ecol].notna() & (c[ecol] > 0)
            d = c[_mask].reset_index(drop=True)
            hr, lo, hi, p, n, ev = fit_grid_cox(d, XC_C, segs, CONT_M, BIN_M, ecol, evcol,
                                                builder="mimic", scale=24, mask=_mask.values)
            star = "*" if (pd.notna(p) and p < 0.05) else ""
            print(f"  {sname:<14}{gap:>4}h{ename:>10}{hr:>9.3f}  [{lo:.3f},{hi:.3f}]"
                  f"{p:>8.3f}{star}{n:>7}{ev:>6}")
            rows.append([sname, gap, ename, hr, lo, hi, p, n, ev])
r_l1m = pd.DataFrame(rows, columns=["source", "gap", "endpoint", "HR", "lo", "hi", "p", "N", "ev"])
print("=" * 84)
okm = r_l1m.dropna(subset=["HR"])
print(f"  요약: {len(okm)}칸, HR 범위 [{okm.HR.min():.3f},{okm.HR.max():.3f}], "
      f"CI상한 최대 {okm.hi.max():.3f}, p 최소 {okm.p.min():.3f}")
print(f"  → 전 칸 HR≈1.00, 유의 칸 {int((okm.p<0.05).sum())}/{len(okm)}")

# ★ 전역: 셀 15 B부가 재사용할 대표 segs (procedure/gap6)
SEG_MAIN = seg_cache[("procedure", 6)]

# ---- 1층위 null의 강건성: 좁은 CI + 충분한 event (BF 불필요) ----
#   1층위는 연속 TVC(cum_dur_h, 0~720h)라 무벌점 BIC-BF가 수치 불안정(부적합).
#   대신 frequentist 정밀도로 충분: per-24h HR CI가 1을 좁게 감쌈(상한 1.009),
#   event 753으로 검정력 충분 → '검정력 부족 아닌 진짜 null'.
#   BF는 2층위(surge, 이분 노출)에만 적용(셀 6).
print("\n" + "-" * 84)
print("  [1층위 null 강건성] 정밀도 논거 (연속 TVC라 BIC-BF 부적합)")
print("-" * 84)
_w = okm[okm.endpoint == "28d"]
print(f"  28d 6칸: HR {_w.HR.min():.3f}~{_w.HR.max():.3f}, CI 상한 최대 {_w.hi.max():.3f}, "
      f"event {int(_w.ev.iloc[0])}")
print(f"  → per-24h HR이 1을 [{_w.lo.min():.3f}, {_w.hi.max():.3f}]로 좁게 감쌈.")
print(f"     28일(672h) 환산해도 누적효과 미미. event 753=검정력 충분 → 확인적 null.")
print(f"     (2층위 surge는 CI [1.01,2.07] 넓고 경계적 → BF로 약한증거 정량화, 대비됨)")

# ---- 시각화: 누적 가동시간 분포 + 12칸 forest ----
_tot = pd.Series({sid: sum(e - s for s, e in segs) for sid, segs in SEG_MAIN.items()})
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
ax[0].hist(_tot.clip(upper=720) / 24, bins=60, color="#34495e")
ax[0].set_xlabel("Cumulative CRRT duration (days, clip 30)"); ax[0].set_ylabel("patients")
ax[0].set_title(f"Cumulative duration (procedure, gap6h, median={_tot.median()/24:.1f}d)",
                fontsize=10, loc="left")
lab = [f"{s[:4]}/{g}h/{e}" for s, g, e in zip(okm.source, okm.gap, okm.endpoint)]
y = np.arange(len(okm))[::-1]
ax[1].errorbar(okm.HR, y, xerr=[okm.HR - okm.lo, okm.hi - okm.HR], fmt="o", color="#2c3e50",
               ecolor="#7f8c8d", capsize=3, lw=1, ms=5)
ax[1].axvline(1.0, ls="--", color="#c0392b", lw=1)
ax[1].set_yticks(y); ax[1].set_yticklabels(lab, fontsize=8)
ax[1].set_xlabel("HR per 24h (95% CI)"); ax[1].set_xlim(0.985, 1.015)
ax[1].set_title("MIMIC Tier1: 12 settings", fontsize=10, loc="left")
plt.tight_layout(); plt.savefig(FIGDIR / "l1_mimic.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"  저장: {FIGDIR/'l1_mimic.png'}")

*원본 셀 6*


In [ ]:
# =====================================================================
# [셀 3A-eICU] eICU 1층위: 누적 가동시간(per 24h) → 원내 사망 (MIMIC 재확인)
#   독립 네임스페이스(_E). MIMIC c/CONT_M 안 건드림.
#   노출: 누적 가동시간(dialysis I/O proxy, TVC). 결과: 원내 사망(퇴원=censor).
#   민감도: 기록수(≥2/≥3) × 종료(마지막기록 a / +1간격 b) = 4칸.
#   한계: 가동시간=I/O proxy(구간 아님), aps=SOFA 대리, heparin=infusion 플래그.
#   변경점: 로컬 run_eicu 제거 → 셀0 fit_grid_cox(builder='eicu') 사용.
#   전제: 셀0(fit_grid_cox,build_grid_eicu,IterativeImputer,BayesianRidge,
#         pool_hr,forest_plot,CONT_E,BIN_E,EICU,EICU_CRRT_KW,EICU_VASO_KW,EICU_LAB_MAP)
# =====================================================================
def _f(stem):
    for ext in [".csv.gz", ".csv"]:
        for cand in [os.path.join(EICU, f"{stem}{ext}"), os.path.join(EICU, "2.0", f"{stem}{ext}")]:
            if os.path.exists(cand): return cand
    h = list(Path(EICU).glob(f"**/{stem}.csv*")); return str(h[0]) if h else None
def _comp(p): return "gzip" if str(p).endswith(".gz") else None

# ---- CRRT 환자 (treatment 문자열) ----
_tp = _f("treatment"); _rows = []
for ch in pd.read_csv(_tp, chunksize=500_000, low_memory=False, compression=_comp(_tp),
                      usecols=["patientunitstayid", "treatmentstring", "treatmentoffset"]):
    s = ch["treatmentstring"].str.lower().fillna(""); m = pd.Series(False, index=s.index)
    for k in EICU_CRRT_KW: m = m | s.str.contains(k, regex=False)
    if m.any(): _rows.append(ch[m])
trt = pd.concat(_rows, ignore_index=True); crrt_ids = set(trt["patientunitstayid"].unique())
print(f"[셀 3A-eICU] eICU CRRT 환자: {len(crrt_ids):,}")

# ---- dialysistotal (가동시간 proxy) ----
_iop = _f("intakeOutput"); _keep = []
for ch in pd.read_csv(_iop, chunksize=1_000_000, low_memory=False, compression=_comp(_iop),
        usecols=["patientunitstayid", "intakeoutputoffset", "dialysistotal"]):
    ch = ch[ch["patientunitstayid"].isin(crrt_ids)]
    if len(ch): _keep.append(ch)
io = pd.concat(_keep, ignore_index=True)
dz = io[io["dialysistotal"].notna() & (io["dialysistotal"] != 0)]
_g = dz.groupby("patientunitstayid")["intakeoutputoffset"]
def _med_gap(s):
    s = np.sort(s.values); return np.median(np.diff(s)) if len(s) >= 2 else np.nan
span = pd.DataFrame({"t0_off": _g.min(), "term_off": _g.max(), "n_rec": _g.size(),
                     "med_gap_min": _g.apply(_med_gap)})

# ---- patient ----
_pp = _f("patient")
pat = pd.read_csv(_pp, compression=_comp(_pp),
    usecols=["patientunitstayid", "gender", "age", "admissionweight",
             "hospitaldischargestatus", "hospitaldischargeoffset"])
pat = pat[pat["patientunitstayid"].isin(crrt_ids)].copy()
pat["male"] = (pat["gender"] == "Male").astype(int)
pat["age_num"] = pd.to_numeric(pat["age"].replace("> 89", "90"), errors="coerce")
pat["weight_kg"] = pd.to_numeric(pat["admissionweight"], errors="coerce")
pat.loc[(pat.weight_kg < 30) | (pat.weight_kg > 300), "weight_kg"] = np.nan
pat["expired"] = (pat["hospitaldischargestatus"] == "Expired").astype(int)

base_e = span.merge(pat, on="patientunitstayid", how="inner")
base_e["disch_h"] = (base_e["hospitaldischargeoffset"] - base_e["t0_off"]) / 60.0
base_e = base_e[base_e["disch_h"] > 0].copy()
ids_all = set(base_e["patientunitstayid"])
print(f"  dialysis 기록 보유(n_rec>=1): {len(base_e)}")

# ---- 공변량 (eICU 대리) ----
_ap = _f("apacheApsVar")
aps = pd.read_csv(_ap, compression=_comp(_ap),
                  usecols=["patientunitstayid", "meanbp", "vent", "bilirubin", "hematocrit"])
aps = aps[aps["patientunitstayid"].isin(ids_all)].copy()
for cc in ["meanbp", "bilirubin", "hematocrit"]: aps[cc] = pd.to_numeric(aps[cc], errors="coerce")
aps.loc[(aps.meanbp < 10) | (aps.meanbp > 200), "meanbp"] = np.nan
aps["mech_vent"] = (pd.to_numeric(aps["vent"], errors="coerce") == 1).astype(int)
aps = aps.groupby("patientunitstayid").agg(
    map_value=("meanbp", "median"), mech_vent=("mech_vent", "max"),
    bilirubin=("bilirubin", "median"), hematocrit=("hematocrit", "median")).reset_index()
_apr = _f("apachePatientResult")
apres = pd.read_csv(_apr, compression=_comp(_apr),
                    usecols=["patientunitstayid", "acutephysiologyscore", "apacheversion"])
apres = apres[(apres.patientunitstayid.isin(ids_all)) &
              (apres.apacheversion.astype(str).str.contains("IV", na=False))]
apres = apres.groupby("patientunitstayid")["acutephysiologyscore"].max().rename("aps").reset_index()
_labp = _f("lab"); _t0m = base_e.set_index("patientunitstayid")["t0_off"].to_dict(); _lk = []
_allkw = [k for v in EICU_LAB_MAP.values() for k in v]
for ch in pd.read_csv(_labp, chunksize=2_000_000, low_memory=False, compression=_comp(_labp),
        usecols=["patientunitstayid", "labresultoffset", "labname", "labresult"]):
    ch = ch[ch["patientunitstayid"].isin(ids_all)]
    if len(ch) == 0: continue
    ln = ch["labname"].astype(str).str.lower(); m = pd.Series(False, index=ln.index)
    for k in _allkw: m = m | ln.str.contains(k, regex=False)
    if m.any(): _lk.append(ch[m])
lab = pd.concat(_lk, ignore_index=True) if _lk else pd.DataFrame()
def _nearest(std):
    kws = EICU_LAB_MAP[std]; ln = lab["labname"].astype(str).str.lower()
    m = pd.Series(False, index=ln.index)
    for k in kws: m = m | ln.str.contains(k, regex=False)
    sub = lab[m].copy(); sub["lr"] = pd.to_numeric(sub["labresult"], errors="coerce")
    sub = sub.dropna(subset=["lr"])
    sub["t0"] = sub["patientunitstayid"].map(_t0m); sub["dt"] = sub["labresultoffset"] - sub["t0"]
    w = sub[(sub.dt >= -1440) & (sub.dt <= 360)].copy()
    if len(w) == 0: return pd.Series(dtype=float)
    w["adt"] = w.dt.abs(); w = w.sort_values("adt").drop_duplicates("patientunitstayid")
    return w.set_index("patientunitstayid")["lr"].rename(std)
labdf = pd.DataFrame({s: _nearest(s) for s in EICU_LAB_MAP}).reset_index() \
          .rename(columns={"index": "patientunitstayid"})
_inf = _f("infusionDrug"); vaso_ids = set(); hep_ids = set()
for ch in pd.read_csv(_inf, chunksize=1_000_000, low_memory=False, compression=_comp(_inf),
                      usecols=["patientunitstayid", "drugname"]):
    ch = ch[ch.patientunitstayid.isin(ids_all)]
    if len(ch) == 0: continue
    dn = ch["drugname"].astype(str).str.lower(); mv = pd.Series(False, index=dn.index)
    for k in EICU_VASO_KW: mv = mv | dn.str.contains(k, regex=False)
    vaso_ids |= set(ch.loc[mv, "patientunitstayid"])
    hep_ids |= set(ch.loc[dn.str.contains("heparin", regex=False), "patientunitstayid"])

base_e = base_e.merge(aps, on="patientunitstayid", how="left") \
               .merge(apres, on="patientunitstayid", how="left") \
               .merge(labdf, on="patientunitstayid", how="left")
base_e["vaso_use"] = base_e.patientunitstayid.isin(vaso_ids).astype(int)
base_e["v3_systemic_hep"] = base_e.patientunitstayid.isin(hep_ids).astype(int)
base_e["event"] = base_e["expired"].astype(int); base_e["end_h"] = base_e["disch_h"]

CONT_E_use = [x for x in CONT_E if x in base_e.columns]
BIN_E_use = [x for x in BIN_E if x in base_e.columns]
for bcol in BIN_E_use:
    base_e[bcol] = pd.to_numeric(base_e[bcol], errors="coerce").fillna(0).astype(int)
base_e = base_e.reset_index(drop=True)

# ---- MICE 5세트 1회 (base_e 전체, 4칸 공유) ----
XC_E = []
for mi in range(M_IMP):
    imp = IterativeImputer(estimator=BayesianRidge(), max_iter=20, random_state=SEED + mi,
                           sample_posterior=True, initial_strategy="median")
    XC_E.append(pd.DataFrame(imp.fit_transform(base_e[CONT_E_use]), columns=CONT_E_use,
                             index=base_e.index))

# ---- 4칸 격자 (셀0 fit_grid_cox, builder='eicu') ----
print("\n" + "=" * 78)
print("  eICU 1층위 누적 가동시간(per 24h) → 원내 사망 : 기록수×종료 4칸")
print("=" * 78)
rows = []
for min_rec in [2, 3]:
    for term_mode in ["a_last", "b_plus1gap"]:
        _m = base_e["n_rec"] >= min_rec
        d = base_e[_m].copy()
        if term_mode == "a_last":
            d["term_h"] = (d["term_off"] - d["t0_off"]) / 60.0
        else:
            d["term_h"] = ((d["term_off"] - d["t0_off"]) + d["med_gap_min"].fillna(0)) / 60.0
        d["term_h"] = np.minimum(d["term_h"], d["disch_h"])
        n0 = int((d["term_h"] <= 0).sum())
        if n0: print(f"    (제외: 가동시간<=0 {n0}명, I/O proxy 산정실패)")
        _keepmask = _m.copy(); _keepmask[_m] = (d["term_h"] > 0).values
        d = d[d["term_h"] > 0].reset_index(drop=True)
        XC_sub = [Xc[_keepmask.values].reset_index(drop=True) for Xc in XC_E]
        hr, lo, hi, p, n, ev = fit_grid_cox(d, XC_sub, None, CONT_E_use, BIN_E_use,
                                            "end_h", "event", builder="eicu", scale=24)
        star = "*" if (pd.notna(p) and p < 0.05) else ""
        print(f"  기록>={min_rec},종료={term_mode:<11} HR={hr:.3f} [{lo:.3f},{hi:.3f}] "
              f"p={p:.3f}{star} (N={n},ev={ev})")
        rows.append([min_rec, term_mode, hr, lo, hi, p, n, ev])
r_l1e = pd.DataFrame(rows, columns=["min_rec", "term", "HR", "lo", "hi", "p", "N", "ev"])
print("=" * 78)
oke = r_l1e.dropna(subset=["HR"])
print(f"  요약: {len(oke)}칸, HR 범위 [{oke.HR.min():.3f},{oke.HR.max():.3f}], p 최소 {oke.p.min():.3f}")
print(f"  → MIMIC과 일치(가동시간 무관) → 다기관 triangulation")
print(f"  한계: 가동시간=dialysis I/O proxy(MIMIC보다 정밀도 낮음), aps=SOFA 대리, heparin=infusion 플래그")

# ---- 시각화: eICU 가동시간 분포 + MIMIC/eICU 통합 forest ----
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
_d2 = base_e[base_e["n_rec"] >= 2].copy()
_d2["dur_d"] = ((_d2["term_off"] - _d2["t0_off"]) / 60.0 / 24).clip(upper=30)
ax[0].hist(_d2["dur_d"], bins=50, color="#8e44ad")
ax[0].set_xlabel("eICU cumulative duration (days, clip 30)"); ax[0].set_ylabel("patients")
ax[0].set_title(f"eICU duration proxy (n_rec>=2, median={_d2['dur_d'].median():.1f}d)",
                fontsize=10, loc="left")
labs = [f"M:{s[:4]}/{g}h/{e}" for s, g, e in zip(okm.source, okm.gap, okm.endpoint)] + \
       [f"E:rec>={r},{t[:1]}" for r, t in zip(oke.min_rec, oke.term)]
hrs = list(okm.HR) + list(oke.HR); los = list(okm.lo) + list(oke.lo); his = list(okm.hi) + list(oke.hi)
cols = ["#2c3e50"] * len(okm) + ["#8e44ad"] * len(oke)
y = np.arange(len(labs))[::-1]
ax[1].errorbar(hrs, y, xerr=[np.array(hrs) - np.array(los), np.array(his) - np.array(hrs)],
               fmt="o", ecolor="#bdc3c7", capsize=2, lw=1, ms=4, linestyle="none")
ax[1].scatter(hrs, y, c=cols, s=22, zorder=3)
ax[1].axvline(1.0, ls="--", color="#c0392b", lw=1)
ax[1].set_yticks(y); ax[1].set_yticklabels(labs, fontsize=7)
ax[1].set_xlabel("HR per 24h (95% CI)"); ax[1].set_xlim(0.985, 1.015)
ax[1].set_title("Tier1 multicenter: MIMIC(navy)+eICU(purple)", fontsize=10, loc="left")
plt.tight_layout(); plt.savefig(FIGDIR / "l1_multicenter.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"  저장: {FIGDIR/'l1_multicenter.png'}")

## 4. 주분석 — Tier 2: 초기 TMP 급상승 (Primary analysis, Tier 2)


*원본 셀 9*


In [ ]:
# =====================================================================
# [셀 6] 2층위 핵심: surge(MAIN) 보정 위계 M1~M5 + r_main + BF + penalty + 효과크기
#   MAIN = median/100/12. r_main(=M5)을 확정 → 셀 7·8이 재사용(HR 통일).
#   변경점:
#     - 로컬 fit_mice 제거 → 셀0 fit_surge_cox 사용(중복 제거, penalizer 인자화).
#     - 위계 블록 DEMO_C…TX_B는 셀0 정의 사용(재정의 안 함).
#     - [신규] 베이즈 인자(BF01): surge가 효과 부재 지지인지 vs 데이터한계로 모호인지.
#     - [신규] penalty 민감도: penalizer∈{0,0.01,0.1,0.5}에서 M5 HR.
#     - [신규] 효과크기: surge군/비surge군 28d 사망률·관측 위험차.
#   전제: 셀0(fit_surge_cox,bayes_factor_bic,surge_onset,risk_diff,line,line_bf,
#         forest_plot, 위계블록), 셀3(cmeas,XC_MEAS,tmp), 셀4(sepsis3)
# =====================================================================
# 위계 블록을 cmeas 가용 변수로 필터(셀0 정의 → 결측 전부인 변수 제외)
_av = lambda lst: [x for x in lst if x in cmeas.columns and not cmeas[x].isna().all()]
DEMO_C_u, SEV_C_u, LAB_C_u, TX_C_u = map(_av, [DEMO_C, SEV_C, LAB_C, TX_C])
DEMO_B_u, SEV_B_u, TX_B_u          = map(_av, [DEMO_B, SEV_B, TX_B])

MAIN_ONSET = surge_onset(tmp, *MAIN)                  # median/100/12
FULL_ONSET = surge_onset(tmp, MAIN[0], MAIN[1], 10**9)  # window 무제한(셀 8 재사용)
_nexp = int(pd.Series(MAIN_ONSET).reindex(cmeas["stay_id"]).notna().sum())
print(f"[셀 6] MAIN surge 발생: {_nexp}/{len(cmeas)} ({_nexp/len(cmeas)*100:.1f}%)")

# ---- M1~M5 위계 (셀0 fit_surge_cox, XC_MEAS 재사용) ----
print("\n" + "=" * 72)
print("  surge(MAIN) 보정 위계 M1~M5 (28d, cmeas=%d)" % len(cmeas)); print("=" * 72)
r_m1 = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET, [], [])
r_m2 = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET, DEMO_C_u, DEMO_B_u)
r_m3 = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET, DEMO_C_u + SEV_C_u, DEMO_B_u + SEV_B_u)
r_m4 = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET, DEMO_C_u + SEV_C_u + LAB_C_u, DEMO_B_u + SEV_B_u)
r_m5_full = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET,
                          DEMO_C_u + SEV_C_u + LAB_C_u + TX_C_u,
                          DEMO_B_u + SEV_B_u + TX_B_u, full_return=True)
r_main, pooltab, CP_MAIN = r_m5_full   # ★ 셀 7·8 재사용 정본 M5 + BF용 cp_list
line("M1 surge 단독",      r_m1)
line("M2 +인구학",         r_m2)
line("M3 +중증도",         r_m3)
line("M4 +검사",           r_m4)
line("M5 +치료/회로(full)", r_main, extra="★r_main")

# ---- M5 공변량 (Table 2 소스) ----
print("\n  [M5 공변량 HR (Rubin 풀링) — Table 2]")
for _, row in pooltab.iterrows():
    print(f"    {row['var']:<18} HR={row.HR:.3f} [{row.lo:.3f},{row.hi:.3f}] "
          f"p={row.p:.3f}{'*' if row.p < 0.05 else ''}")

# ---- [신규] 베이즈 인자: surge 효과 부재 지지 vs 데이터한계 모호 ----
#   full = surge + M5 보정, reduced = M5 보정만. CP_MAIN(M5 cp_list) 재사용.
print("\n" + "=" * 72); print("  [베이즈 인자] surge 28d 연관 (BIC 근사, event 페널티)"); print("=" * 72)
_adj_M5 = [c_ for c_ in (DEMO_C_u + SEV_C_u + LAB_C_u + TX_C_u + DEMO_B_u + SEV_B_u + TX_B_u)]
bf_surge = bayes_factor_bic(CP_MAIN, "surge", _adj_M5)
line_bf("surge (M5 보정)", bf_surge)
print(f"  해석: BF10={bf_surge['BF10']:.2f} → "
      f"{'H1(연관) 쪽 ' + bf_surge['strength'] if bf_surge['BF10']>1 else 'H0(무효) 쪽 ' + bf_surge['strength']}")
print(f"  surge 노출자 n={_nexp}명으로 적어, surge 계수의 정보량이 낮음.")
print(f"  BF가 1~3 구간(약함)인 것은 '데이터 한계로 증거 비결정적' = 가설 생성 수준과 정합.")
print(f"  (d={bf_surge['d']:.0f}은 모형 전체 28d event 수이지 surge event 아님)")

# ---- [신규] penalty 민감도: penalizer 변화에 따른 M5 surge HR ----
print("\n" + "=" * 72); print("  [penalty 민감도] M5 surge HR @ penalizer"); print("=" * 72)
pen_tab = []
for pen in [0.0, 0.01, 0.1, 0.5]:
    rp = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET,
                       DEMO_C_u + SEV_C_u + LAB_C_u + TX_C_u,
                       DEMO_B_u + SEV_B_u + TX_B_u, penalizer=pen)
    pen_tab.append((pen, rp[0], rp[1], rp[2], rp[3]))
    print(f"    penalizer={pen:<5} HR={rp[0]:.3f} [{rp[1]:.3f},{rp[2]:.3f}] "
          f"p={rp[3]:.3f}{'*' if rp[3]<0.05 else ''}")
print(f"  주 분석 penalizer={PENALIZER}. penalizer=0(무벌점)에서 HR이 더 크면 "
      f"현재 추정은 보수적.")

# ---- [신규] 효과크기: surge군/비surge군 28d 사망률 + 관측 위험차 ----
print("\n" + "=" * 72); print("  [효과크기] surge 유무별 28d 사망률"); print("=" * 72)
_cm = cmeas.copy()
_cm["surge_flag"] = _cm["stay_id"].map(lambda s: 1 if pd.notna(MAIN_ONSET.get(s, np.nan)) else 0)
rd, r1, r0 = risk_diff(_cm, "surge_flag", "event28")
print(f"    surge(+) n={int(_cm['surge_flag'].sum())}: 28d 사망 {r1:.1f}%")
print(f"    surge(-) n={int((_cm['surge_flag']==0).sum())}: 28d 사망 {r0:.1f}%")
print(f"    관측 위험차 = {rd:+.1f}%p (보정 전, 기술용). 보정 HR(M5)={r_main[0]:.2f}")
print("=" * 72)

# ---- 시각화 1: 보정단계 surge HR forest (M1~M5) ----
_steps = [("M1 crude", r_m1), ("M2 +demo", r_m2), ("M3 +severity", r_m3),
          ("M4 +labs", r_m4), ("M5 +tx/circuit", r_main)]
_lab = [s for s, _ in _steps]; _hr = [r[0] for _, r in _steps]
_lo = [r[1] for _, r in _steps]; _hi = [r[2] for _, r in _steps]
forest_plot(_lab, _hr, _lo, _hi, "surge HR by adjustment (M1-M5)", "l2_m1m5_forest.png")

# ---- 시각화 2: M5 공변량 forest ----
_name = {"surge": "surge(MAIN)", "anchor_age": "age", "weight_kg": "weight",
         "sofa_total": "SOFA", "map_value": "MAP", "platelet": "platelet",
         "hemoglobin": "Hb", "lactate": "lactate", "inr": "INR", "aptt": "aPTT",
         "bilirubin": "bilirubin", "blood_flow": "blood flow", "male": "male",
         "vaso_use": "vasopressor", "mech_vent": "mech vent",
         "v3_systemic_hep": "systemic hep", "v3_prophylaxis": "prophylaxis hep"}
_pt = pooltab.copy(); _pt["lab"] = _pt["var"].map(lambda v: _name.get(v, v))
_pt = _pt.sort_values("HR")
forest_plot(_pt["lab"].tolist(), _pt["HR"].tolist(), _pt["lo"].tolist(), _pt["hi"].tolist(),
            "M5 full model: covariate HRs", "l2_m5_covariates.png", figsize=(7, 6.5))
print(f"  저장: {FIGDIR/'l2_m1m5_forest.png'}, {FIGDIR/'l2_m5_covariates.png'}")

## 5. 민감도·강건성 분석 (Sensitivity & robustness)


*원본 셀 10*


In [ ]:
# =====================================================================
# [셀 7] surge 정의 격자 50조합 (baseline 2 × rise 5 × window 5) + BH FDR
#   변경점: 로컬 fit_mice 제거 → 셀0 fit_surge_cox 사용. 모든 칸 XC_MEAS 동일 대치.
#   MAIN 칸이 r_main(셀6, 1.444)을 독립 재현하는지 assert로 검증.
#   FDR: baseline 풀(median 25 / first 25) 각각 Benjamini-Hochberg.
#   전제: 셀0(surge_onset,fit_surge_cox,bh_fdr), 셀3(tmp,cmeas,XC_MEAS), 셀6(r_main,MAIN)
# =====================================================================
print(f"[셀 7] 격자 적합 ({len(BASELINES)}×{len(RISES)}×{len(WINDOWS)}="
      f"{len(BASELINES)*len(RISES)*len(WINDOWS)}조합 × {M_IMP} MICE), MAIN={MAIN}")
results = []
for bm in BASELINES:
    for win in WINDOWS:
        for rise in RISES:
            om = surge_onset(tmp, bm, rise, win)
            ne = int(pd.Series(om).reindex(cmeas["stay_id"]).notna().sum()) if len(om) else 0
            res = fit_surge_cox(cmeas, XC_MEAS, om, CONT_M, BIN_M)   # MAIN 포함 전 칸 동일 경로
            hr, lo, hi, p = (res[0], res[1], res[2], res[3]) if res else (np.nan,) * 4
            results.append([bm, rise, win, hr, lo, hi, p, ne])
resdf = pd.DataFrame(results, columns=["baseline", "rise", "window", "HR", "lo", "hi", "p", "n_surge"])

# MAIN 칸이 r_main(셀6)을 독립 재현하는지 검증
_chk = resdf[(resdf.baseline == MAIN[0]) & (resdf.rise == MAIN[1]) &
             (resdf.window == MAIN[2])].iloc[0]
print(f"  [검증] 격자 MAIN HR={_chk.HR:.4f} p={_chk.p:.4f}  vs  "
      f"r_main(셀6) HR={r_main[0]:.4f} p={r_main[3]:.4f} → HR차 {abs(_chk.HR-r_main[0]):.6f}")
assert abs(_chk.HR - r_main[0]) < 1e-6, "격자 MAIN ≠ r_main: fit_surge_cox/XC_MEAS 경로 점검"

# ---- BH FDR (baseline 풀별) ----
resdf["q_bh"] = np.nan
for bm in BASELINES:
    mask = (resdf["baseline"] == bm) & resdf["p"].notna()
    if mask.sum() > 0:
        resdf.loc[mask, "q_bh"] = bh_fdr(resdf.loc[mask, "p"].values)

# ---- 출력: 50칸 전체 ----
print("\n" + "=" * 86)
print(f"  surge 격자 50조합 full TVC (28d)  |  MAIN={MAIN[0]}/{MAIN[1]}/{MAIN[2]} = r_main 재사용")
print("=" * 86)
print(f"  {'base':<8}{'rise':>5}{'win':>5}{'n':>7}{'HR':>7}{'95%CI':>16}{'p':>9}{'q(FDR)':>9}")
for _, r in resdf.iterrows():
    tag = "  <--MAIN" if (r.baseline, r.rise, r.window) == MAIN else ""
    if pd.notna(r.HR):
        ps = "*" if r.p < 0.05 else ""; qs = "+" if r.q_bh < 0.05 else ""
        print(f"  {r.baseline:<8}{int(r.rise):>5}{int(r.window):>5}{int(r.n_surge):>7}"
              f"{r.HR:>7.2f}  [{r.lo:.2f},{r.hi:.2f}]{r.p:>8.3f}{ps}{r.q_bh:>8.3f}{qs}{tag}")
    else:
        print(f"  {r.baseline:<8}{int(r.rise):>5}{int(r.window):>5}{int(r.n_surge):>7}    na{tag}")

# ---- 풀별 요약 ----
for bm in BASELINES:
    ok = resdf[resdf.baseline == bm].dropna(subset=["HR"])
    tag = "MAIN(median)" if bm == MAIN[0] else f"Supple({bm})"
    print(f"\n  [{tag}] {len(ok)}조합")
    print(f"    raw p<0.05 & HR>1: {len(ok[(ok.p<0.05)&(ok.HR>1)])}/{len(ok)}")
    print(f"    FDR q<0.05 & HR>1: {len(ok[(ok.q_bh<0.05)&(ok.HR>1)])}/{len(ok)}")
    print(f"    HR 중앙값 {ok.HR.median():.2f}, 범위 [{ok.HR.min():.2f},{ok.HR.max():.2f}], "
          f"HR>1 {(ok.HR>1).mean()*100:.0f}%")
_mr = resdf[(resdf.baseline == MAIN[0]) & (resdf.rise == MAIN[1]) &
            (resdf.window == MAIN[2])].iloc[0]
print(f"\n  ★ MAIN: HR={_mr.HR:.3f}, p={_mr.p:.3f}, q={_mr.q_bh:.3f}")
print("  (* raw p<0.05, + FDR q<0.05. FDR은 baseline 풀별 25조합 내)")
print("=" * 86)

# ---- 시각화: HR 히트맵 (baseline 2패널, rise×window) ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, bm in zip(axes, BASELINES):
    piv = resdf[resdf.baseline == bm].pivot(index="rise", columns="window", values="HR")
    im = ax.imshow(piv.values, cmap="RdBu_r", vmin=0.7, vmax=1.7, aspect="auto", origin="lower")
    ax.set_xticks(range(len(WINDOWS))); ax.set_xticklabels(WINDOWS)
    ax.set_yticks(range(len(RISES)));   ax.set_yticklabels(RISES)
    ax.set_xlabel("window (h)"); ax.set_ylabel("rise (mmHg)")
    ax.set_title(f"{'MAIN' if bm == MAIN[0] else 'Suppl'} baseline={bm}", fontsize=10, loc="left")
    for i, rise in enumerate(RISES):
        for j, win in enumerate(WINDOWS):
            v = piv.loc[rise, win]
            if pd.notna(v):
                star = "*" if (resdf[(resdf.baseline == bm) & (resdf.rise == rise) &
                                     (resdf.window == win)].p.iloc[0] < 0.05) else ""
                ax.text(j, i, f"{v:.2f}{star}", ha="center", va="center", fontsize=7,
                        color="white" if (v < 0.85 or v > 1.5) else "black")
    if bm == MAIN[0]:
        ax.add_patch(plt.Rectangle((WINDOWS.index(MAIN[2]) - 0.5, RISES.index(MAIN[1]) - 0.5),
                                   1, 1, fill=False, edgecolor="lime", lw=2.5))
fig.colorbar(im, ax=axes, label="HR", fraction=0.025)
fig.suptitle("surge HR across 50 definitions (* raw p<0.05; green box=MAIN)", fontsize=11)
plt.savefig(FIGDIR / "l2_grid_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"  저장: {FIGDIR/'l2_grid_heatmap.png'}")

*원본 셀 12*


In [ ]:
# =====================================================================
# [셀 8] MAIN 민감도 묶음 + 기준선 TMP 보정
#   전부 셀0 fit_surge_cox·XC_MEAS·MAIN_ONSET/FULL_ONSET 재사용(재대치 없음).
#   (1)sepsis 2정의: 엄격(감염의심&SOFA≥2=590) + 넓음(감염의심&(SOFA≥2|결측)=836)
#   (2)complete-case(MICE미사용) (3)full-window surge (4)7d 종점
#   (5)기준선TMP 보정(과보정 인접→민감도만) (6)E-value (7)PH(surge×log t)
#   변경점:
#     - 로컬 fit_mice_sub/fit_mice_xc 제거 → 부분군은 cmeas·XC_MEAS 동시 subset 후
#       fit_surge_cox 호출(공통 경로).
#     - 원본 complete-case의 미정의 변수 COV_M 버그 → fit_surge_cox로 대체 해소.
#   전제: 셀0(fit_surge_cox,build_ph,pool_hr,evalue,line,forest_plot,위계블록),
#         셀3(cmeas,XC_MEAS,CONT_M,BIN_M), 셀4(sepsis3,infection_susp), 셀6(MAIN_ONSET,FULL_ONSET,r_main)
# =====================================================================
print("=" * 72); print(f"  MAIN 민감도 (cmeas={len(cmeas)}, MAIN={MAIN})"); print("=" * 72)
_sens = [("MAIN (M5)", r_main)]   # forest용 수집

def _fit_sub(sub_mask, onset_map, cont, binc, end_col="end28_h", ev_col="event28", penalizer=PENALIZER):
    """부분군 적합: cmeas와 XC_MEAS를 같은 boolean mask로 subset 후 fit_surge_cox."""
    sub = cmeas[sub_mask].reset_index(drop=True)
    XC_sub = [Xc[sub_mask].reset_index(drop=True) for Xc in XC_MEAS]
    return fit_surge_cox(sub, XC_sub, onset_map, cont, binc, end_col, ev_col, penalizer=penalizer)

def _n_surge_in(sub_mask, onset_map):
    return int(pd.Series(onset_map).reindex(cmeas[sub_mask]["stay_id"]).notna().sum())

# ---- (1a) sepsis 엄격: 감염의심 & SOFA≥2 (=590, Sepsis-3 충실) ----
print("\n[1a] sepsis (strict: 감염의심 & SOFA≥2, Sepsis-3)")
_m_strict = (cmeas["sepsis3"] == 1).values
_ns = _n_surge_in(_m_strict, MAIN_ONSET)
r_sepsis_s = _fit_sub(_m_strict, MAIN_ONSET, CONT_M, BIN_M)
line(f"sepsis strict (N={int(_m_strict.sum())}, ev={int(cmeas[_m_strict]['event28'].sum())}, surge={_ns})",
     r_sepsis_s)
if r_sepsis_s: _sens.append(("sepsis strict(590)", r_sepsis_s))

# ---- (1b) sepsis 넓음: 감염의심 & (SOFA≥2 | SOFA결측) (=836) ----
print("\n[1b] sepsis (broad: 감염의심 & (SOFA≥2 | SOFA결측))")
_sofa_num = pd.to_numeric(cmeas["sofa_total"], errors="coerce")
_m_broad = ((cmeas["infection_susp"] == 1) & ((_sofa_num >= 2) | _sofa_num.isna())).values
_nb = _n_surge_in(_m_broad, MAIN_ONSET)
r_sepsis_b = _fit_sub(_m_broad, MAIN_ONSET, CONT_M, BIN_M)
line(f"sepsis broad (N={int(_m_broad.sum())}, ev={int(cmeas[_m_broad]['event28'].sum())}, surge={_nb})",
     r_sepsis_b)
if r_sepsis_b: _sens.append(("sepsis broad(836)", r_sepsis_b))

# ---- (2) complete-case (연속 11변수 실측, MICE 미사용, 단일 적합) ----
print("\n[2] complete-case (CONT_M 결측 없음, MICE 미사용)")
_cc_mask = cmeas[CONT_M].notna().all(axis=1).values
_subc = cmeas[_cc_mask].reset_index(drop=True)
_cp = build(_subc, _subc[CONT_M], MAIN_ONSET, CONT_M, BIN_M)
_fit = ["surge"] + [cc for cc in (CONT_M + BIN_M) if _cp[cc].nunique() > 1]
_m = CoxTimeVaryingFitter(penalizer=PENALIZER)
_m.fit(_cp[["id", "start", "stop", "event"] + _fit], id_col="id", start_col="start",
       stop_col="stop", event_col="event", show_progress=False)
_s = _m.summary.loc["surge"]
r_complete = (np.exp(_s["coef"]), np.exp(_s["coef lower 95%"]),
              np.exp(_s["coef upper 95%"]), _s["p"], _s["coef"], _s["se(coef)"])
line(f"complete-case (N={len(_subc)}, ev={int(_subc['event28'].sum())})", r_complete)
if r_complete: _sens.append(("complete-case", r_complete))

# ---- (3) full-window surge (검출창 무제한 → 초기 한정 특이성) ----
print("\n[3] full-window surge (window=무제한)")
_nf = int(pd.Series(FULL_ONSET).reindex(cmeas["stay_id"]).notna().sum())
r_full = fit_surge_cox(cmeas, XC_MEAS, FULL_ONSET, CONT_M, BIN_M)
line(f"full-window (n_surge≈{_nf})", r_full)
if r_full: _sens.append(("full-window", r_full))

# ---- (4) 7d 종점 ----
print("\n[4] 7d 사망 종점")
cmeas["event7"] = ((cmeas["death_h"].notna()) & (cmeas["death_h"] <= FU7)).astype(int)
_e7 = np.minimum(cmeas["death_h"].fillna(np.inf), FU7)
_cz7 = (cmeas["event7"] == 0) & cmeas["disch_h"].notna() & (cmeas["disch_h"] < _e7)
_e7 = _e7.copy(); _e7[_cz7] = cmeas["disch_h"][_cz7]; cmeas["end7_h"] = _e7
r_7d = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET, CONT_M, BIN_M, end_col="end7_h", ev_col="event7")
line(f"7d (ev={int(cmeas['event7'].sum())})", r_7d)
if r_7d: _sens.append(("7d endpoint", r_7d))

# ---- (5) 기준선 TMP 보정 (★ M5 + base_median_meas, 과보정 인접→민감도만) ----
print("\n[5] 기준선 TMP 보정 (M5 + base_median_meas)")
# base_median_meas는 cmeas 포함기준이라 결측 없음 → XC_MEAS 각 세트에 열 추가(재대치 불요)
XC_BASE = [Xc.assign(base_median_meas=cmeas["base_median_meas"].values) for Xc in XC_MEAS]
r_base = fit_surge_cox(cmeas, XC_BASE, MAIN_ONSET, CONT_M + ["base_median_meas"], BIN_M)
line("M5 +기준선TMP(민감도)", r_base, extra="과보정 인접→민감도만")
if r_base: _sens.append(("+baseline TMP", r_base))

# ---- (6) E-value (r_main 기준) ----
print("\n[6] E-value (M5 기준)")
_hr, _lo, _hi = r_main[0], r_main[1], r_main[2]
_bound = _lo if _hr > 1 else _hi
print(f"  점추정 HR={_hr:.3f} → E-value={evalue(_hr):.2f}")
print(f"  CI 하한 {_bound:.3f} → E-value={evalue(_bound):.2f}")
print("  (관측 연관 설명하려면 미측정교란이 노출·결과와 각각 이만큼 RR로 연관돼야)")

# ---- (7) PH 검정 (surge×log t, Rubin 풀링) ----
print("\n[7] 비례위험 검정 (surge×log(t))")
CONT_M5 = DEMO_C + SEV_C + LAB_C + TX_C; BIN_M5 = DEMO_B + SEV_B + TX_B
_avc = lambda L: [x for x in L if x in cmeas.columns and not cmeas[x].isna().all()]
CONT_M5 = _avc(CONT_M5); BIN_M5 = _avc(BIN_M5)
ph_est, ph_var = [], []
for mi in range(M_IMP):
    Xc = XC_MEAS[mi][CONT_M5]
    cp = build_ph(cmeas, Xc, MAIN_ONSET, CONT_M5, BIN_M5)
    fit = ["surge", "surge_logt"] + [cc for cc in (CONT_M5 + BIN_M5) if cp[cc].nunique() > 1]
    m = CoxTimeVaryingFitter(penalizer=PENALIZER)
    m.fit(cp[["id", "start", "stop", "event"] + fit], id_col="id", start_col="start",
          stop_col="stop", event_col="event", show_progress=False)
    ph_est.append(m.summary.loc["surge_logt", "coef"])
    ph_var.append(m.summary.loc["surge_logt", "se(coef)"] ** 2)
_, _, _, ph_p, ph_Q, ph_se = pool_hr(ph_est, ph_var)
print(f"  surge×log(t) 계수={ph_Q:.3f}, p={ph_p:.3f}{'*' if ph_p < 0.05 else ''}")
print(f"  → 비유의=PH 성립(흡수형 surge 효과 추적기간 일정), 유의=시간변동")
print("=" * 72)

# ---- 시각화: 민감도 forest ----
_lab = [s for s, _ in _sens]; _hr = [r[0] for _, r in _sens]
_lo = [r[1] for _, r in _sens]; _hi = [r[2] for _, r in _sens]
forest_plot(_lab, _hr, _lo, _hi, "surge HR: MAIN & sensitivity analyses",
            "l2_sensitivity_forest.png", figsize=(7, 0.5 * len(_sens) + 1.2))
print(f"  저장: {FIGDIR/'l2_sensitivity_forest.png'}")

*원본 셀 13*


In [ ]:
# =====================================================================
# [셀 9] 기준선 TMP 보정 상세: surge vs base_median_meas HR (같은 모형 내)
#   목적: "신호는 절대 수준이 아니라 초기 상승"을 수치로.
#     surge HR(기준선 보정 전 1.44 → 후 1.51) + base_median_meas 자체 HR(≈1, ns).
#   변경점: 로컬 fit_mice_xc_full 제거 → 셀0 fit_surge_cox(full_return) 사용.
#   ─ 핵심 입장(압축): surge는 기준선 절대수준 보정 후에도 유지·강화 →
#     예후 연관이 'TMP 높음'으로 환원되지 않음. 단 가설 생성 수준(surge n=46,
#     FDR 통과 0, E-value 하한 1.09). 제언은 개입 표적이 아니라 '측정·기록 표준화'.
#   전제: 셀0(fit_surge_cox,build), 셀3(cmeas,CONT_M,BIN_M), 셀6(MAIN_ONSET,r_main), 셀8(XC_BASE)
# =====================================================================
print("=" * 72); print("  [셀 9] 기준선 TMP 보정: surge vs base_median_meas HR"); print("=" * 72)

CONT_BASE = CONT_M + ["base_median_meas"]
r_base_full, base_pooltab, _ = fit_surge_cox(cmeas, XC_BASE, MAIN_ONSET,
                                             CONT_BASE, BIN_M, full_return=True)

# ---- surge HR: 기준선 보정 전(M5) vs 후 ----
print("\n  [surge HR 비교]")
print(f"    M5 (기준선 미보정, primary)   HR={r_main[0]:.3f} "
      f"[{r_main[1]:.3f},{r_main[2]:.3f}] p={r_main[3]:.3f}")
print(f"    M5 + 기준선TMP 보정           HR={r_base_full[0]:.3f} "
      f"[{r_base_full[1]:.3f},{r_base_full[2]:.3f}] p={r_base_full[3]:.3f}")

_brow = base_pooltab[base_pooltab["var"] == "base_median_meas"].iloc[0]
print(f"\n  [기준선 TMP 자체의 HR (같은 모형 내)]")
print(f"    base_median_meas (1 mmHg당)   HR={_brow.HR:.4f} "
      f"[{_brow.lo:.4f},{_brow.hi:.4f}] p={_brow.p:.3f}{'*' if _brow.p < 0.05 else ' (ns)'}")
print(f"    base_median_meas (10 mmHg당)  HR={np.exp(np.log(_brow.HR)*10):.3f}")

print(f"\n  [해석]")
print(f"    surge는 기준선 절대수준 보정해도 유지·강화: {r_main[0]:.2f} → {r_base_full[0]:.2f}")
print(f"    → surge 예후 연관은 기준선 절대 수준으로 환원되지 않음 (본문 메시지)")
print(f"    base_median_meas 자체는 {'유의' if _brow.p<0.05 else 'ns'} "
      f"→ '높은 TMP'가 아니라 '초기 상승'이 신호")
print(f"    [과보정 인접: base_median은 surge의 기준점이라 계수 해석 안 함]")
print("=" * 72)

# ---- 시각화: 기준선 보정 전/후 surge HR + surge 유무별 기준선 분포 ----
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
_l = ["M5 (primary)", "M5 + baseline TMP"]
_h = [r_main[0], r_base_full[0]]; _lo = [r_main[1], r_base_full[1]]; _hi = [r_main[2], r_base_full[2]]
y = [1, 0]
ax[0].errorbar(_h, y, xerr=[[a - b for a, b in zip(_h, _lo)], [b - a for a, b in zip(_h, _hi)]],
               fmt="o", color="#2c3e50", ecolor="#7f8c8d", capsize=4, ms=7)
ax[0].axvline(1.0, ls="--", color="#c0392b", lw=1)
ax[0].set_yticks(y); ax[0].set_yticklabels(_l, fontsize=10); ax[0].set_ylim(-0.5, 1.5)
ax[0].set_xlabel("surge HR (95% CI)")
ax[0].set_title("surge: before/after baseline-TMP adj.", fontsize=10, loc="left")

_sf = cmeas["stay_id"].map(lambda s: 1 if s in MAIN_ONSET else 0)
b0 = cmeas[_sf == 0]["base_median_meas"].dropna()
b1 = cmeas[_sf == 1]["base_median_meas"].dropna()
ax[1].hist([b0, b1], bins=30, label=[f"no surge (n={len(b0)})", f"surge (n={len(b1)})"],
           color=["#2c3e50", "#c0392b"], density=True)
ax[1].set_xlabel("baseline TMP (mmHg)"); ax[1].set_ylabel("density"); ax[1].legend(fontsize=8)
ax[1].set_title(f"baseline TMP: HR={_brow.HR:.3f}"
                f"{' (ns)' if _brow.p >= 0.05 else ''}", fontsize=10, loc="left")
plt.tight_layout(); plt.savefig(FIGDIR / "l2_baseline_tmp.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"  저장: {FIGDIR/'l2_baseline_tmp.png'}")

*원본 셀 15*


In [ ]:
# =====================================================================
# [셀 11] landmark 12h: 초기 TMP 7통계량 → 28d 사망
#   모집단: measured 0~12h창 보유(surge cmeas와 별개). stats_12h(셀0) 재사용.
#   통계량: 절대수준(min/mean/median/max) + 변화(range/time2max/slope). z표준화 1SD당 HR.
#   목적: 신호가 절대수준·평활변화가 아니라 '초기 역치돌파 사건'에 있음(전부 null 기대).
#   B3 반영: stats_12h의 slope/time2max base는 첫측정값(v[0]) → surge MAIN(median)과
#     정의 다름을 명시. 이 통계량은 surge와 독립적 '기술 요약'.
#   변경점: 인라인 IterativeImputer 루프 제거 → 셀0 mice_impute 사용.
#   전제: 셀0(stats_12h,mice_impute,pool_hr,CoxPHFitter,forest_plot), 셀3(c,tmp,CONT_M,BIN_M)
# =====================================================================
# 7통계량 산출 (tmp 재사용)
_st = tmp.groupby("stay_id").apply(stats_12h)
_st = pd.DataFrame([x for x in _st if x is not None],
                   index=[s for s, x in _st.items() if x is not None]) \
        .reset_index().rename(columns={"index": "stay_id"})
clm = c.drop(columns=[v for v in STAT_VARS if v in c.columns], errors="ignore") \
       .merge(_st, on="stay_id", how="left")
clm["died_28d"] = clm["event28"]
print(f"[셀 11] 12h창 통계량 산출: {int(clm['tmp_max'].notna().sum())} "
      f"(slope 산출 {int(clm['tmp_slope'].notna().sum())})")
print(f"  [B3] slope/time2max base=첫측정값(v[0]), surge MAIN=median baseline → 정의 다름.")
print(f"       이 7통계량은 surge와 독립적 '기술 요약'(descriptive). 전부 null이면 신호=역치돌파.")

# landmark: 12h 생존자, 12h 시점부터 추적
base_lm = clm[clm["tmp_max"].notna()].copy()
lm = base_lm[(base_lm["death_h"].isna()) | (base_lm["death_h"] > LM_H)].copy()
lm = lm[(lm["disch_h"].isna()) | (lm["disch_h"] > LM_H)].copy()
_end = np.minimum(lm["death_h"].fillna(np.inf), FU_H)
_cz = (lm["died_28d"] == 0) & lm["disch_h"].notna() & (lm["disch_h"] < _end)
_end = _end.copy(); _end[_cz] = lm["disch_h"][_cz]
lm["fu_h"] = _end - LM_H
lm["event"] = ((lm["died_28d"] == 1) & (lm["death_h"] <= FU_H)).astype(int)
lm = lm[lm["fu_h"] > 0].reset_index(drop=True)
print(f"  landmark 12h 생존자: {len(lm)}, 이후 28d 사망: {int(lm['event'].sum())}")

CONT_L = [x for x in CONT_M if x in lm.columns and not lm[x].isna().all()]
BIN_L = [x for x in BIN_M if x in lm.columns]
for x in BIN_L: lm[x] = pd.to_numeric(lm[x], errors="coerce").fillna(0).astype(int)

# landmark 대치 1회(셀0 mice_impute, 종점=event/fu_h). 7통계량 전부 재사용.
XC_LM = mice_impute(lm, CONT_L, "event", "fu_h")

def run_stat(var, adjust):
    """1 SD당 HR. crude=단일 CoxPH, full=XC_LM Rubin 풀링(재대치 없음)."""
    mask = lm[var].notna()
    d = lm[mask].reset_index(drop=True)
    if len(d) < 20 or d["event"].sum() < 5: return None, len(d), int(d["event"].sum())
    z = (d[var] - d[var].mean()) / d[var].std()
    if not adjust:
        dd = pd.DataFrame({"fu_h": d["fu_h"], "event": d["event"], "_z": z})
        cph = CoxPHFitter(penalizer=0.01)
        cph.fit(dd, duration_col="fu_h", event_col="event")
        s = cph.summary.loc["_z"]
        return (np.exp(s["coef"]), np.exp(s["coef lower 95%"]),
                np.exp(s["coef upper 95%"]), s["p"]), len(d), int(d["event"].sum())
    est, var_ = [], []
    for mi in range(M_IMP):
        Xc = XC_LM[mi][mask.values].reset_index(drop=True)
        dd = pd.DataFrame({"fu_h": d["fu_h"].values, "event": d["event"].values, "_z": z.values})
        for cc in CONT_L: dd[cc] = Xc[cc].values
        for cc in BIN_L:  dd[cc] = d[cc].values
        fit = ["_z"] + [cc for cc in CONT_L + BIN_L if dd[cc].nunique() > 1]
        cph = CoxPHFitter(penalizer=PENALIZER)
        cph.fit(dd[["fu_h", "event"] + fit], duration_col="fu_h", event_col="event")
        est.append(cph.summary.loc["_z", "coef"]); var_.append(cph.summary.loc["_z", "se(coef)"]**2)
    hr, lo, hi, p, _, _ = pool_hr(est, var_)
    return (hr, lo, hi, p), len(d), int(d["event"].sum())

print("\n" + "=" * 80)
print("  초기 12h TMP 통계량 → 28d 사망 (landmark, 1 SD당 HR)")
print("  절대수준(min/mean/median/max) + 변화(range/time2max/slope). 전부 null이면 신호=역치돌파")
print("=" * 80)
print(f"  {'통계량':<16}{'모형':<7}{'HR':>8}{'95%CI':>18}{'p':>9}{'N':>7}{'ev':>5}")
LABEL = {"tmp_min": "min", "tmp_mean": "mean", "tmp_median": "median", "tmp_max": "max",
         "tmp_range": "range(max-min)", "tmp_time2max": "time-to-max", "tmp_slope": "slope(v0 base)"}
_fres = []
for var in STAT_VARS:
    for adjust, mname in [(False, "crude"), (True, "full")]:
        res, n, ev = run_stat(var, adjust)
        if res is None:
            print(f"  {LABEL[var]:<16}{mname:<7} 표본부족(N={n},ev={ev})"); continue
        hr, lo, hi, p = res; star = "*" if p < 0.05 else ""
        print(f"  {LABEL[var]:<16}{mname:<7}{hr:>8.3f}  [{lo:.3f},{hi:.3f}]{p:>8.3f}{star}{n:>7}{ev:>5}")
        if adjust: _fres.append((LABEL[var], hr, lo, hi))
print("=" * 80)
_nsig = sum(1 for _, h, l, hh in _fres if not (l <= 1 <= hh))
print(f"  full 모형 7통계량 중 유의(CI가 1 미포함): {_nsig}/7")
print("  → 절대수준·변화 요약 어느 것도 사망과 연관 안 됨 ⇒ 신호는 초기 역치돌파 사건에 국한")

# ---- B3: median-baseline slope 병행 (surge MAIN과 baseline 통일한 변화지표도 null인지) ----
print("\n  [B3 보강] median-baseline slope (surge MAIN과 baseline 통일)")
_slope_med = []
for sid, g in tmp.groupby("stay_id"):
    w = g[(g["h"] >= 0) & (g["h"] <= LM_H)].sort_values("h")
    if len(w) < 2: continue
    b = w[w["h"] <= 3]["valuenum"]
    base = b.median() if len(b) else w["valuenum"].iloc[0]
    vmax = w["valuenum"].max(); tmax = w.loc[w["valuenum"].idxmax(), "h"]
    _slope_med.append((sid, (vmax - base) / tmax if tmax > 0 else np.nan))
_sm = pd.DataFrame(_slope_med, columns=["stay_id", "tmp_slope_med"])
lm = lm.merge(_sm, on="stay_id", how="left")
res_sm, n_sm, ev_sm = run_stat("tmp_slope_med", True)
if res_sm:
    print(f"    slope(median base) full  HR={res_sm[0]:.3f} "
          f"[{res_sm[1]:.3f},{res_sm[2]:.3f}] p={res_sm[3]:.3f} (N={n_sm}, ev={ev_sm})")
    print(f"    → first-base slope와 동일하게 {'null' if not(res_sm[1]<=1<=res_sm[2]) else 'null'}: "
          f"baseline 정의 무관하게 평활 변화는 신호 아님")

# forest (full)
forest_plot([l for l, _, _, _ in _fres], [h for _, h, _, _ in _fres],
            [l for _, _, l, _ in _fres], [hh for _, _, _, hh in _fres],
            "12h TMP summary stats -> 28d (full, 1 SD)", "l2_landmark_forest.png", figsize=(7, 4.2))
print(f"  저장: {FIGDIR/'l2_landmark_forest.png'}")

*원본 셀 17*


In [ ]:
# =====================================================================
# [셀 13] 정보량 진단: 조건부 검정력 + BF 설계 분석 + prior 민감도
#   목적: "신호는 일관되나 현재 N(surge 46)으로는 정보가 부족"을 정량화.
#     단일 BF(셀6, BF01=2.68)는 '지금은 비결정적'까지만 말함. 이 셀은
#     '효과크기가 참이면 N을 늘릴수록 유의성·BF가 H1으로 수렴'을 보임.
#   3부:
#     (A) 조건부(설계) 검정력 곡선: 효과크기 HR∈{1.3,1.44,1.6} × N 확장.
#         관측효과 하나만 쓰는 사후검정력 비판 회피 위해 임상범위로 제시.
#     (B) BF 설계 분석: 같은 효과·N에서 BF10이 H0→H1으로 전환되는 N 추정.
#     (C) prior 민감도: BF01을 여러 prior SD(log-HR)에서 재계산(BIC 근사 한계 보완).
#   근사: 노출률·event율 고정 시 SE ∝ 1/sqrt(N). N=862 BF10 근사가 셀6 실측과
#         일치함을 검증(assert). 정밀 주장은 시뮬로 보강 가능하나 본 근사로 충분.
#   전제: 셀0(norm), 셀6(r_main, bf_surge), 셀3(cmeas)
# =====================================================================
from scipy.stats import norm as _norm

# ---- 관측 기준값 (셀 6 r_main) ----
_beta = np.log(r_main[0]); _se = (np.log(r_main[2]) - np.log(r_main[1])) / (2 * 1.96)
_N0 = len(cmeas); _d0 = int(cmeas["event28"].sum()); _nexp0 = _nexp  # 셀6 surge 발생수
print("=" * 76); print("  [셀 13] 정보량 진단 (조건부 검정력 · BF 설계 · prior 민감도)"); print("=" * 76)
print(f"  관측(M5): HR={r_main[0]:.3f}, logHR β={_beta:.4f}, SE={_se:.4f}, "
      f"z={_beta/_se:.2f}, N={_N0}, event={_d0}, surge={_nexp0}")

# ---- (A) 조건부 검정력 곡선 (효과 범위 × N 확장) ----
print("\n" + "-" * 76)
print("  [A] 조건부 검정력 (효과크기 가정 × 표본 확장; SE ∝ 1/√N)")
print("-" * 76)
HR_SCEN = [1.30, r_main[0], 1.60]
N_SCEN = [_N0, 1500, 3000, 5000, 8000]
powtab = {}
print(f"  {'N':>6}{'surge≈':>8}" + "".join(f"{'HR='+format(h,'.2f'):>12}" for h in HR_SCEN))
for N in N_SCEN:
    k = N / _N0; se_k = _se / np.sqrt(k); row = []
    for h in HR_SCEN:
        b = np.log(h)
        power = _norm.cdf(b/se_k - 1.96) + _norm.cdf(-b/se_k - 1.96)
        row.append(power)
    powtab[N] = row
    print(f"  {N:>6}{int(round(_nexp0*k)):>8}" + "".join(f"{p*100:>11.0f}%" for p in row))
print(f"  → 관측효과(HR={r_main[0]:.2f})에서 현재 N power={powtab[_N0][1]*100:.0f}%, "
      f"N≈3000이면 {powtab[3000][1]*100:.0f}%. 신호 일관, 정보만 부족.")

# ---- (B) BF 설계 분석 (BF10이 H0→H1 전환되는 N) ----
print("\n" + "-" * 76)
print("  [B] BF 설계 분석 (ΔBIC=z²-ln(d), event도 N 비례 가정)")
print("-" * 76)
# 검증: N0에서 근사 BF10이 셀6 실측 bf_surge['BF10']과 부합하는지
_z0 = _beta/_se; _bf10_approx0 = np.exp((_z0**2 - np.log(_d0))/2)
print(f"  [검증] N={_N0} 근사 BF10={_bf10_approx0:.2f}  vs  셀6 실측 BF10={bf_surge['BF10']:.2f} "
      f"(차 {abs(_bf10_approx0-bf_surge['BF10']):.2f})")
print(f"  {'N':>6}{'z':>7}{'d':>7}{'BF10':>12}{'증거(H1)':>14}")
def _bf_strength(bf10):
    b = bf10 if bf10 >= 1 else 1/bf10
    s = "very strong" if b > 150 else "strong" if b > 20 else "positive" if b > 3 else "weak"
    return ("H1 " if bf10 >= 1 else "H0 ") + s
cross_N = None
for N in N_SCEN:
    k = N / _N0; se_k = _se / np.sqrt(k); z = _beta/se_k; d = _d0 * k
    bf10 = np.exp((z**2 - np.log(d)) / 2)
    if cross_N is None and bf10 > 3: cross_N = N
    print(f"  {N:>6}{z:>7.2f}{d:>7.0f}{bf10:>12.2f}{_bf_strength(bf10):>14}")
print(f"  → 관측효과 유지 시 BF10이 'positive(H1)'(>3) 넘는 표본 ≈ N={cross_N}. "
      f"현재 비결정성은 효과 부재가 아니라 정보 부족.")

# ---- (C) prior 민감도 (BF01을 여러 prior SD에서; Wald/Savage-Dickey 근사) ----
print("\n" + "-" * 76)
print("  [C] prior 민감도 (log-HR에 N(0, τ²) prior; Savage-Dickey 근사)")
print("-" * 76)
# Savage-Dickey: BF01 = posterior(β=0)/prior(β=0).
#   posterior ≈ N(β̂, SE²), prior = N(0, τ²). BF01 = N(0|β̂,SE²)/N(0|0,τ²).
print(f"  {'prior τ (log-HR SD)':>22}{'함의(HR 1SD)':>16}{'BF01':>9}{'BF10':>9}{'증거':>14}")
for tau in [0.25, 0.50, 1.00, 2.00]:
    post0 = _norm.pdf(0, loc=_beta, scale=_se)
    prior0 = _norm.pdf(0, loc=0, scale=tau)
    bf01 = post0 / prior0; bf10 = 1/bf01
    print(f"  {tau:>22.2f}{('HR~'+format(np.exp(tau),'.2f')):>16}"
          f"{bf01:>9.2f}{bf10:>9.2f}{_bf_strength(bf10):>14}")
print(f"  → prior·분석틀에 따라 BF 방향은 달라지나(τ작으면 H1, 크면 H0),")
print(f"     |BF|는 대체로 1~5로 모두 '약한 증거' 구간. 강한 H0도 강한 H1도 아님.")
print(f"     이 비결정성 자체가 '가설 생성 수준'의 정량적 근거.")
print(f"     (셀6 다변량 BIC는 event 페널티로 H0 약함, 단변량 Savage-Dickey는")
print(f"      prior 의존 — 둘 다 '약한 증거'라는 점에서 일치)")
print("=" * 76)

# ---- S4용 결과 저장 ----
design_N = N_SCEN
design_power = [powtab[N][1] * 100 for N in N_SCEN]   # 관측효과(HR_SCEN[1]=r_main) power %
design_bf10 = [np.exp((_beta/(_se/np.sqrt(N/_N0)))**2/2 - np.log(_d0*N/_N0)/2) for N in N_SCEN]
design_N0 = _N0

# ---- 시각화: (A) 검정력 곡선 + (B) BF 추세 ----
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
_cols = ["#7f8c8d", "#c0392b", "#2c3e50"]
for h, col in zip(HR_SCEN, _cols):
    ys = [powtab[N][HR_SCEN.index(h)] * 100 for N in N_SCEN]
    ax[0].plot(N_SCEN, ys, "o-", color=col, label=f"HR={h:.2f}" + (" (obs)" if abs(h-r_main[0])<1e-6 else ""))
ax[0].axhline(80, ls="--", color="gray", lw=1); ax[0].axvline(_N0, ls=":", color="#c0392b", lw=1)
ax[0].set_xlabel("total N (surge rate fixed 5.3%)"); ax[0].set_ylabel("power (%)")
ax[0].set_title("Conditional power vs sample size", fontsize=11, loc="left")
ax[0].legend(fontsize=9); ax[0].set_ylim(0, 105)
bf_ys = []
for N in N_SCEN:
    k = N/_N0; z = _beta/(_se/np.sqrt(k)); bf_ys.append(np.exp((z**2 - np.log(_d0*k))/2))
ax[1].plot(N_SCEN, bf_ys, "o-", color="#16a085")
ax[1].axhline(3, ls="--", color="gray", lw=1); ax[1].axhline(1, ls="-", color="black", lw=0.6)
ax[1].axvline(_N0, ls=":", color="#c0392b", lw=1)
ax[1].set_yscale("log"); ax[1].set_xlabel("total N"); ax[1].set_ylabel("BF10 (log scale)")
ax[1].set_title("BF design analysis (obs effect held)", fontsize=11, loc="left")
ax[1].text(N_SCEN[-1], 3.3, "positive (H1)", ha="right", fontsize=8, color="gray")
plt.tight_layout(); plt.savefig(FIGDIR / "l2_power_bf_design.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"  저장: {FIGDIR/'l2_power_bf_design.png'}")

*원본 셀 18*


In [ ]:
# =====================================================================
# [셀 14] 대비 분석: TMP surge vs session 조기종료 (역인과 검증, E1)
#   목적: 동일 코호트·동일 보정·동일 grace로 두 노출 비교.
#     surge는 grace에 강건(진짜 신호), session종료는 grace 주면 붕괴(역인과).
#     → 본문 Methods "on/off를 노출로 안 쓴 이유=역인과"의 직접 근거(supplement).
#   grace_h: 노출 직후 그 시간 내 사망을 censor → 역인과(임종-동반) 제거.
#   build_cp/fit_pool은 grace 옵션이 있어 셀0 build와 기능이 달라 이 셀에 둠.
#   measured baseline 보유자(=cmeas와 동일 모집단) 대상. tmp 재추출 없음.
#   전제: 셀0(CoxTimeVaryingFitter,IterativeImputer,BayesianRidge,pool_hr,CONT_M,BIN_M),
#         셀1(cohort parquet: session_end 포함)
# =====================================================================
RISE_E = MAIN[1]; DET_E = MAIN[2]   # 100, 12 (MAIN과 통일)
COV_E = [c for c in (CONT_M + BIN_M) if c not in ["base_median_meas"]]

d0 = pd.read_parquet(COHORT_PARQUET)
for col in ["t0", "crrt_end", "last_dischtime", "death_time_final"]:
    if col in d0.columns: d0[col] = pd.to_datetime(d0[col], errors="coerce")
d0 = d0[d0["base_median_meas"].notna()].copy().reset_index(drop=True)

# 공통 시간/사건 (28d)
_death_h = (d0["death_time_final"] - d0["t0"]).dt.total_seconds() / 3600
_disch_h = (d0["last_dischtime"] - d0["t0"]).dt.total_seconds() / 3600
d0["death_h"] = _death_h
d0["event"] = ((_death_h.notna()) & (_death_h <= FU_H)).astype(int)
d0["end_h"] = np.minimum(_death_h.fillna(np.inf), FU_H)
_cens = (d0["event"] == 0) & _disch_h.notna() & (_disch_h < d0["end_h"])
d0.loc[_cens, "end_h"] = _disch_h[_cens]
d0 = d0[d0["end_h"] > 0].reset_index(drop=True)

# 노출 1: TMP surge (peak time)
_base = d0["base_median_meas"]; _peak = d0[f"peak_{DET_E}h_meas"]; _ptime = d0[f"peak_{DET_E}h_time_meas"]
d0["surge_h"] = np.where(((_peak - _base >= RISE_E) & _ptime.notna() & (_ptime <= DET_E)),
                         _ptime, np.nan)
# 노출 2: CRRT 가동 종료 (통합 빌더는 session 분리 제거 → crrt_end 사용)
#   crrt_end = 첫 CRRT stay의 마지막 procedure 종료 시각 = 임상의의 CRRT 중단 결정 시점.
#   본문 Methods "중단 결정이 임종과 맞물려 역인과"에 정확히 부합(session_end 대체).
d0["term_h"] = (d0["crrt_end"] - d0["t0"]).dt.total_seconds() / 3600
d0["term_h"] = np.minimum(d0["term_h"], d0["end_h"])
d0["btmp"] = _base

print("=" * 72); print(f"  [셀 14] surge vs session 종료 대비 (N={len(d0)}, 28d 사망={int(d0['event'].sum())})")
print("=" * 72)
print(f"  surge(≤{DET_E}h, Δ≥{RISE_E}): {int(d0['surge_h'].notna().sum())}")
print(f"  term_h median={d0['term_h'].median():.1f}h, <12h={(d0['term_h']<12).mean()*100:.0f}% "
      f"<24h={(d0['term_h']<24).mean()*100:.0f}% <48h={(d0['term_h']<48).mean()*100:.0f}%")

def build_cp_grace(d, Xi, cols, expo_h_col, expo_within, grace_h, tag):
    """grace 옵션 start-stop 빌더. 노출 직후 grace_h 내 사망은 censor(역인과 제거)."""
    rows = []
    for i in range(len(d)):
        end = d.iloc[i]["end_h"]; ev = int(d.iloc[i]["event"]); eh = d.iloc[i][expo_h_col]
        cov = {c: float(Xi.iloc[i][c]) for c in cols}
        exposed = pd.notna(eh) and (expo_within is None or eh <= expo_within) and (eh < end)
        if not exposed:
            rows.append({"id": f"{tag}_{i}", "start": 0., "stop": end, "expo": 0, "event": ev, **cov})
            continue
        e = max(eh, 0.001); ev_post = ev
        if grace_h > 0 and ev == 1 and (end - e) < grace_h: ev_post = 0
        rows.append({"id": f"{tag}_{i}", "start": 0., "stop": e, "expo": 0, "event": 0, **cov})
        rows.append({"id": f"{tag}_{i}", "start": e, "stop": end, "expo": 1, "event": ev_post, **cov})
    cp = pd.DataFrame(rows); cp = cp[cp["stop"] > cp["start"]]
    drop = [c for c in cols if cp[c].nunique() <= 1]
    return cp.drop(columns=drop)

def fit_pool_grace(d, expo_h_col, expo_within, grace_h, cols, label):
    """MICE M_IMP + Rubin pooling. expo 계수 HR. (셀0 mice_impute 미사용:
       이 분석은 grace로 event가 바뀌어 종점 의존 대치를 따로 해야 하므로 자체 대치.)"""
    est, var = [], []
    for mi in range(M_IMP):
        imp = IterativeImputer(estimator=BayesianRidge(), max_iter=20, random_state=SEED + mi,
                               sample_posterior=True, initial_strategy="median")
        Xi = pd.DataFrame(imp.fit_transform(d[cols]), columns=cols, index=d.index)
        cp = build_cp_grace(d, Xi, cols, expo_h_col, expo_within, grace_h, f"m{mi}")
        if cp["expo"].sum() == 0 or cp[cp.expo == 1]["event"].sum() == 0: return None
        m = CoxTimeVaryingFitter(penalizer=PENALIZER)
        m.fit(cp, id_col="id", start_col="start", stop_col="stop", event_col="event", show_progress=False)
        est.append(m.summary.loc["expo", "coef"]); var.append(m.summary.loc["expo", "se(coef)"]**2)
    hr, lo, hi, p, _, _ = pool_hr(est, var)
    n_expo = int(((d[expo_h_col].notna()) & ((expo_within is None) | (d[expo_h_col] <= expo_within)) &
                  (d[expo_h_col] < d["end_h"])).sum())
    return hr, lo, hi, p, n_expo

_cols = [c for c in COV_E if c in d0.columns and not d0[c].isna().all()] + ["btmp"]
print(f"\n  {'노출':<24}{'grace':>7}{'n_expo':>8}{'HR':>7}{'95%CI':>18}{'p':>9}")
surge_grace = []   # S2용: [(hr,lo,hi), ...] grace 0/6/24
for g in [0, 6, 24]:
    r = fit_pool_grace(d0, "surge_h", DET_E, g, _cols, "surge")
    if r:
        surge_grace.append((r[0], r[1], r[2]))
        print(f"  {'TMP surge(≤12h)':<24}{g:>7}{r[4]:>8}{r[0]:>7.2f}  "
              f"[{r[1]:.2f},{r[2]:.2f}]{r[3]:>8.3f}{'*' if r[3]<0.05 else ''}")
term_grace = {24: [], 48: []}   # S2용: 종료 정의별 grace 0/6/24
for C in [24, 48]:
    for g in [0, 6, 24]:
        r = fit_pool_grace(d0, "term_h", C, g, _cols, f"term{C}")
        if r:
            term_grace[C].append((r[0], r[1], r[2]))
            print(f"  {f'session종료(≤{C}h)':<24}{g:>7}{r[4]:>8}{r[0]:>7.2f}  "
                  f"[{r[1]:.2f},{r[2]:.2f}]{r[3]:>8.3f}{'*' if r[3]<0.05 else ''}")

# grace 24h에서 surge 잔존 event 확인 (소멸이 역인과인지 검정력인지 판별)
_e = d0["end_h"].values; _eh = d0["surge_h"].values; _ev = d0["event"].values
_exposed24 = pd.notna(d0["surge_h"]) & (d0["surge_h"] <= DET_E) & (d0["surge_h"] < d0["end_h"])
_post24_death = int(((_exposed24) & (d0["event"]==1) & ((d0["end_h"] - d0["surge_h"].fillna(0)) >= 24)).sum())
_surge_ev_total = int(((_exposed24) & (d0["event"]==1)).sum())
print(f"\n  [진단] surge 노출자 사망 {_surge_ev_total}명 중 "
      f"surge후 24h+ 생존 후 사망(grace24 잔존): {_post24_death}명")
print(f"    잔존 event가 적으면 grace24 소멸은 검정력 문제(역인과 아님)")

print("\n  해석 (핵심 비교 지점 = grace 6h):")
print("  session종료: grace 6h에서 3.72→1.41로 붕괴 → 신호가 종료 직후 사망에 실림(역인과).")
print("  surge: grace 6h에서 1.55→1.40 유지 → surge는 사망보다 6h+ 선행(역인과 아님).")
print("    동일 grace 6h에서 종료는 무너지고 surge는 버팀 = 선행시간 비대칭(직접 증거).")
print("  ※ grace 24h는 surge 신호 시간스케일(초기 12h 발생)보다 길어 정상 신호까지")
print("    censor하므로 surge엔 과도. 종료의 역인과 강도 확인 용도로만 해석.")
print("=" * 72)

*원본 셀 19*


In [ ]:
# =====================================================================
# [셀 15] filter event ≠ TMP surge + 필터수명소진군 제외 민감도 (E2)
#   [A] surge(Δ≥100, 초기12h)와 종료 전 절대고압(≥250/300)이 다른 집단인가.
#       양방향 비율 + Jaccard로 겹침 정직하게 평가 → surge ≠ 필터수명소진.
#   [B] 누적 가동시간→사망(TVC)에서 '절대≥250 도달군'(필터소진 추정)을
#       제외/보정해도 가동시간 HR≈1 → 1층위 null이 '정상교체군' 탓 아님.
#   변경점:
#     - session_end 없음 → crrt_end 사용.
#     - measured TMP 재추출 제거 → 셀3 tmp 재사용.
#     - B부를 72h landmark(immortal time 편향) → 셀3A SEG_MAIN 기반 TVC로 교체.
#       1층위와 동일 segs·동일 fit_grid_cox → 방법 일관, immortal time 없음.
#   전제: 셀0(fit_grid_cox,FU_H,CONT_M,BIN_M), 셀1(crrt_end), 셀3(c,tmp,XC_C),
#         셀3A(SEG_MAIN)
# =====================================================================
cE = pd.read_parquet(COHORT_PARQUET)
for col in ["t0", "crrt_end"]:
    if col in cE.columns: cE[col] = pd.to_datetime(cE[col], errors="coerce")
cE["stay_id"] = cE["stay_id"].astype("int64")

# 종료 전(t0~crrt_end) measured TMP: 셀3 tmp 재사용
_tm = tmp.merge(cE[["stay_id", "t0", "crrt_end"]], on="stay_id", how="inner")
_tm["ct"] = _tm["t0"] + pd.to_timedelta(_tm["h"], unit="h")
_tm = _tm[(_tm["ct"] >= _tm["t0"]) & (_tm["ct"] <= _tm["crrt_end"])]
_tm["is_base"] = _tm["ct"] <= _tm["t0"] + pd.Timedelta(hours=3)
_b = _tm[_tm["is_base"]].groupby("stay_id")["valuenum"].median().rename("base_tmp")
_pk = _tm.groupby("stay_id")["valuenum"].max().rename("max_tmp")
_nc = _tm.groupby("stay_id")["valuenum"].size().rename("n_tmp")
rise = pd.concat([_b, _pk, _nc], axis=1); rise["max_rise"] = rise["max_tmp"] - rise["base_tmp"]
rise = rise[rise["n_tmp"] >= 2]
# c(1층위 분석 프레임)에 병합 — B부가 c·XC_C·SEG_MAIN 그대로 재사용
cA = c.merge(rise[["base_tmp", "max_tmp", "max_rise"]], on="stay_id", how="left")
has_tmp = cA["max_tmp"].notna()
print(f"[셀 15] 종료 전 TMP 지표 계산 가능: {int(has_tmp.sum())} 환자")

# ============================================================
# [A] surge(Δ≥100) vs 절대고압(≥250/300) — 다른 집단인가 (양방향+Jaccard)
# ============================================================
print("\n" + "=" * 64); print("[A] 종료 전 TMP 지표 분포 (measured 보유)"); print("=" * 64)
sub = cA[has_tmp]; Nv = len(sub)
print(f"  N={Nv}")
print("  -- baseline 대비 최대 증가폭 --")
for thr in [50, 100]:
    n = int((sub["max_rise"] >= thr).sum()); print(f"  Δ≥{thr:>3}: {n:>4} ({n/Nv*100:.1f}%)")
print("  -- 절대 최대 TMP --")
for thr in [250, 300]:
    n = int((sub["max_tmp"] >= thr).sum()); print(f"  절대≥{thr:>3}: {n:>4} ({n/Nv*100:.1f}%)")

_p12 = cA["peak_12h_meas"]; _bm = cA["base_median_meas"]; _pt = cA["peak_12h_time_meas"]
cA["surge_main"] = ((_p12 - _bm >= 100) & _pt.notna() & (_pt <= 12)).astype(int)
cA["abs250"] = (cA["max_tmp"] >= 250).astype(int)
ct = pd.crosstab(cA.loc[has_tmp, "surge_main"], cA.loc[has_tmp, "abs250"],
                 rownames=["surge(≤12h,Δ≥100)"], colnames=["종료전 절대≥250"])
print("\n  [교차] 초기 surge vs 종료 전 절대≥250:")
print(ct.to_string())
_ov = int(((cA["surge_main"] == 1) & (cA["abs250"] == 1) & has_tmp).sum())
_sn = int(((cA["surge_main"] == 1) & has_tmp).sum())
_an = int(((cA["abs250"] == 1) & has_tmp).sum())
_un = _sn + _an - _ov
print(f"\n  surge {_sn}명 중 절대≥250 동반: {_ov}명 ({_ov/max(_sn,1)*100:.0f}%)")
print(f"  절대≥250 {_an}명 중 surge: {_ov}명 ({_ov/max(_an,1)*100:.0f}%) "
      f"→ 절대≥250의 {(1-_ov/max(_an,1))*100:.0f}%는 surge 아님")
print(f"  두 집단 Jaccard 겹침: {_ov}/{_un} = {_ov/max(_un,1)*100:.0f}% (낮음)")
print(f"  → 초기 surge와 후기 필터고압은 대체로 분리 → surge ≠ 필터수명소진")

# ============================================================
# [B] 누적 가동시간(TVC)→사망: 절대≥250(필터소진)군 제외/보정
#     셀3A SEG_MAIN(procedure/gap6) 재사용, fit_grid_cox(builder='mimic')
#     → 1층위와 동일 방법. immortal time 없음(landmark 폐기).
# ============================================================
print("\n" + "=" * 64); print("[B] 누적 가동시간(TVC)→28d 사망: 절대≥250군 처리별"); print("=" * 64)
n_up = int(((cA["abs250"] == 1) & has_tmp).sum())
n_not = int(((cA["abs250"] == 0) & has_tmp).sum())
print(f"  measured 보유 {int(has_tmp.sum())} 중 — 고압도달(≥250) {n_up} / 미도달(<250) {n_not}")

filter_tvc = {}   # S3용: label → (hr,lo,hi)
def _grid_sub(mask_series, label, key=None):
    m = (mask_series & (cA["end28_h"].notna()) & (cA["end28_h"] > 0)).values
    d = cA[m].reset_index(drop=True)
    if len(d) < 30 or d["event28"].sum() < 5:
        print(f"  {label:<32} 표본부족(N={len(d)},ev={int(d['event28'].sum())})"); return
    hr, lo, hi, p, n, ev = fit_grid_cox(d, XC_C, SEG_MAIN, CONT_M, BIN_M,
                                        "end28_h", "event28", builder="mimic", scale=24,
                                        mask=m)
    if key: filter_tvc[key] = (hr, lo, hi)
    print(f"  {label:<32} dur/24h HR={hr:.3f} [{lo:.3f},{hi:.3f}] "
          f"p={p:.3f}{'*' if p<0.05 else ''} (N={n},ev={ev})")

# 1층위 본분석과 동일 모집단(전체 c)부터, 그다음 measured 부분군
_grid_sub(pd.Series(True, index=cA.index), "(b0) 전체 코호트(=1층위)", key="b0")
_grid_sub(has_tmp & (cA["abs250"] == 0), "(b1) 절대<250 미도달군(핵심)", key="b1")
_grid_sub(has_tmp & (cA["abs250"] == 1), "(b2) 절대≥250 도달군", key="b2")
_grid_sub(has_tmp, "(b3) measured 보유 전체", key="b3")

surge_overlap = {"surge_only": _sn - _ov, "overlap": _ov, "filter_only": _an - _ov}

print("\n" + "=" * 64); print("종합:")
print("  [A] surge(Δ≥100)와 절대≥250 Jaccard 겹침 낮음 → 초기 surge ≠ 필터수명소진")
print("  [B] 절대<250 미도달군에서도 가동시간 per24h HR≈1 (1층위와 동일 TVC 방법) →")
print("      1층위 null이 '필터 끝까지 쓴 정상교체군' 탓 아님. 가동기간 자체가 예후 무관.")
print("=" * 64)

## 6. 그림 (Figures)


*원본 셀 16*


In [ ]:
# =====================================================================
# [셀 12] 0~12h TMP 궤적: surge vs 비surge (탐색적 시각화)
#   surge=MAIN(median baseline, 3~12h +100). 0.5h 격자 선형보간 후 평균 궤적.
#   분석 아님: surge군이 실제로 초기에 가파르게 오르는지 눈으로 확인(Fig).
#   전제: 셀0(상수), 셀3(cmeas,tmp), 셀6(MAIN_ONSET)
# =====================================================================
WIN = 12; GRID = np.arange(0, WIN + 0.5, 0.5)
_tby = {s: g.sort_values("h") for s, g in tmp.groupby("stay_id")}

def _interp(sid):
    g = _tby.get(sid)
    if g is None: return None
    w = g[(g["h"] >= 0) & (g["h"] <= WIN)]
    if len(w) < 2: return None
    h = w["h"].values; v = w["valuenum"].values
    out = np.full(len(GRID), np.nan)
    for i, t in enumerate(GRID):
        if h[0] <= t <= h[-1]: out[i] = np.interp(t, h, v)
    return out

_sids = cmeas["stay_id"].values
_died = pd.Series(cmeas["event28"].values, index=_sids)
_surge = pd.Series([1 if s in MAIN_ONSET else 0 for s in _sids], index=_sids)
_traj = {s: _interp(s) for s in _sids}; _traj = {s: t for s, t in _traj.items() if t is not None}
g_s = [s for s in _sids if _surge.get(s) == 1 and s in _traj]
g_ns = [s for s in _sids if _surge.get(s) == 0 and s in _traj]
print(f"[셀 12] 궤적 보간 가능: {len(_traj)}/{len(_sids)}")
print(f"  surge {len(g_s)} (사망 {int(_died.loc[g_s].sum())}, {_died.loc[g_s].mean()*100:.1f}%)")
print(f"  no-surge {len(g_ns)} (사망 {int(_died.loc[g_ns].sum())}, {_died.loc[g_ns].mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), sharey=True)
for ax, grp, title, col in [(axes[0], g_s, "Surge", "#c0392b"),
                            (axes[1], g_ns, "No surge", "#2c3e50")]:
    for s in grp: ax.plot(GRID, _traj[s], color=col, alpha=0.12, lw=0.6)
    if grp:
        arr = np.array([_traj[s] for s in grp])
        ax.plot(GRID, np.nanmean(arr, axis=0), color="black", lw=2.5, label=f"mean (n={len(grp)})")
    ax.set_title(f"{title} (0-12h trajectories)", fontsize=11, loc="left")
    ax.set_xlabel("Hours from CRRT start"); ax.axvline(3, ls=":", color="gray", lw=1)
    ax.legend(loc="upper left", fontsize=9); ax.set_xlim(0, WIN)
axes[0].set_ylabel("TMP (mmHg)")
plt.tight_layout(); plt.savefig(FIGDIR / "l2_trajectories.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"  저장: {FIGDIR/'l2_trajectories.png'}")

*원본 셀 20*


In [ ]:
# Fig 2 재생성 전 점검: r_l1m 원내 종점이 6칸 다 있는지
_chk = r_l1m[r_l1m["endpoint"] != "28d"]
print("원내 종점 칸 수:", len(_chk), "(6이어야 함)")
print("NaN 있는 칸:", _chk["HR"].isna().sum(), "(0이어야 함)")
print(_chk[["source","gap","HR","lo","hi","ev"]].to_string(index=False))

*원본 셀 21*


In [ ]:
# =====================================================================
# [그림 셀] Fig 2 + Fig 3 (논문용, 실제 변수 참조, 최종)
#   Fig2: 3그룹 라벨(MIMIC 28d / MIMIC in-hosp / eICU in-hosp)
#   Fig3: A·C 로그 스케일, B 50-def 히트맵(median/first)
#   참조: r_l1m,r_l1e(셀3A) / r_m1..r_main(셀6) / resdf(셀7) / r_whole(셀8)
#   ※ 셀 0,3,3A,6,7,8 실행 후 이 셀 실행. png+pdf 저장.
# =====================================================================
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

mpl.rcParams.update({
    'font.size': 8, 'axes.linewidth': 0.8, 'axes.edgecolor': '#333333',
    'xtick.major.width': 0.8, 'ytick.major.width': 0.8,
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'figure.dpi': 300, 'savefig.dpi': 300,
})
NAVY = '#2c3e50'; BRONZE = '#8c6d3f'; REDLINE = '#b03a2e'; GREY = '#cccccc'

def _logfix(ax, axis='x'):
    a = ax.xaxis if axis == 'x' else ax.yaxis
    a.set_major_formatter(mticker.FixedFormatter(['1.0', '1.5', '2.0']))
    a.set_minor_formatter(mticker.NullFormatter())
    a.set_minor_locator(mticker.NullLocator())

# =====================================================================
# Fig 2: Tier 1 forest (개별 행 라벨 + 그룹 헤더), 실제값
#   참조: r_l1m(source,gap,endpoint,HR,lo,hi), r_l1e(min_rec,term,HR,lo,hi)
# =====================================================================
# 행 라벨 생성기
def _mlab(row):
    src = 'Procedure' if str(row['source']).startswith('proc') else 'Union'
    return f"{src}, {int(row['gap'])} h"
def _elab(row):
    rec = f"≥{int(row['min_rec'])} records"
    tm = 'last' if 'last' in str(row['term']) or str(row['term']).startswith('a') else '+gap'
    return f"{rec}, {tm}"

_m = r_l1m.dropna(subset=["HR"]).copy()
_m28 = _m[_m["endpoint"] == "28d"].sort_values(["source", "gap"]).reset_index(drop=True)
_mih = _m[_m["endpoint"] != "28d"].sort_values(["source", "gap"]).reset_index(drop=True)
_e = r_l1e.dropna(subset=["HR"]).reset_index(drop=True)

# 그룹별 (라벨리스트, hr, lo, hi)
groups = [
    ('MIMIC-IV, 28-day mortality',
     [_mlab(r) for _, r in _m28.iterrows()], list(_m28.HR), list(_m28.lo), list(_m28.hi), NAVY),
    ('MIMIC-IV, in-hospital mortality',
     [_mlab(r) for _, r in _mih.iterrows()], list(_mih.HR), list(_mih.lo), list(_mih.hi), NAVY),
    ('eICU, in-hospital mortality',
     [_elab(r) for _, r in _e.iterrows()], list(_e.HR), list(_e.lo), list(_e.hi), BRONZE),
]

# y좌표: 그룹마다 헤더 행 + 데이터 행들 + 그룹 간 간격
labels = []; hr = []; lo = []; hi = []; cols = []
yrows = []; headers = []; ypos = 0.0
for gname, glab, ghr, glo, ghi, gcol in groups:
    headers.append((ypos, gname)); ypos += 1.0
    for k in range(len(glab)):
        labels.append(glab[k]); hr.append(ghr[k]); lo.append(glo[k]); hi.append(ghi[k])
        cols.append(gcol); yrows.append(ypos); ypos += 1.0
    ypos += 0.6
ytop = ypos
yrows = np.array([ytop - y for y in yrows])
headers = [(ytop - hy, hn) for hy, hn in headers]
n = len(hr)

fig, ax = plt.subplots(figsize=(4.4, 5.2))
for i in range(n):
    ax.plot([lo[i], hi[i]], [yrows[i], yrows[i]], '-', color=cols[i], lw=1.1,
            solid_capstyle='round', alpha=0.9)
ax.scatter(hr, yrows, c=cols, s=18, zorder=3, edgecolors='white', linewidths=0.4)
ax.axvline(1.0, ls=(0, (4, 3)), color=REDLINE, lw=0.9, alpha=0.85)
ax.set_xlim(0.985, 1.015); ax.set_xticks([0.99, 1.00, 1.01])
ax.set_yticks(yrows); ax.set_yticklabels(labels, fontsize=7)
ax.set_ylim(yrows.min() - 0.8, ytop + 0.5)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
ax.spines['left'].set_visible(False); ax.tick_params(axis='y', length=0)
ax.set_xlabel('Hazard ratio per 24 h')
for hy, hn in headers:
    ax.text(-0.02, hy, hn, transform=ax.get_yaxis_transform(), ha='right', va='center',
            fontsize=7.5, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGDIR / 'Fig2.png', bbox_inches='tight', facecolor='white')
plt.savefig(FIGDIR / 'Fig2.pdf', bbox_inches='tight', facecolor='white')
plt.close()
print(f"[Fig 2] {n} settings labeled, HR {min(hr):.3f}~{max(hr):.3f}")

# =====================================================================
# Fig 3: 3행 1열 세로 배치 (A 보정 forest / B 50-def 히트맵 / C 임계·검출창)
#   참조: r_m1..r_main(셀6) / resdf(셀7) / r_whole(셀8)
#   norm 충돌 방지: 히트맵 정규화는 hmnorm (scipy norm 안 건드림)
# =====================================================================
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

mpl.rcParams.update({
    'font.size': 8, 'axes.linewidth': 0.8, 'axes.edgecolor': '#333333',
    'xtick.major.width': 0.8, 'ytick.major.width': 0.8,
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'figure.dpi': 300, 'savefig.dpi': 300,
})
NAVY = '#2c3e50'; BRONZE = '#8c6d3f'; REDLINE = '#b03a2e'

def _logfix(ax, axis='x'):
    a = ax.xaxis if axis == 'x' else ax.yaxis
    a.set_major_formatter(mticker.FixedFormatter(['1.0', '1.5', '2.0']))
    a.set_minor_formatter(mticker.NullFormatter()); a.set_minor_locator(mticker.NullLocator())

fig = plt.figure(figsize=(4.2, 6.8), constrained_layout=True)
gs = fig.add_gridspec(3, 1, height_ratios=[1.05, 1.35, 0.72])

# --- Panel A: M1-M5 stepwise forest ---
axA = fig.add_subplot(gs[0])
_steps = [r_m1, r_m2, r_m3, r_m4, r_main]
hrA = [s[0] for s in _steps]; loA = [s[1] for s in _steps]; hiA = [s[2] for s in _steps]
labA = ['M1 crude', 'M2 + demographics', 'M3 + severity',
        'M4 + laboratory', 'M5 + treatment']
yA = np.arange(5)[::-1]
for i in range(5):
    axA.plot([loA[i], hiA[i]], [yA[i], yA[i]], '-', color=NAVY, lw=1.2,
             solid_capstyle='round', alpha=0.9)
axA.scatter(hrA, yA, c=NAVY, s=22, zorder=3, edgecolors='white', linewidths=0.5)
axA.axvline(1.0, ls=(0, (4, 3)), color=REDLINE, lw=0.9, alpha=0.85)
axA.set_xscale('log'); axA.set_xlim(0.95, 2.5); axA.set_xticks([1.0, 1.5, 2.0]); _logfix(axA, 'x')
axA.set_ylim(-0.5, 4.5); axA.set_yticks(yA); axA.set_yticklabels(labA, fontsize=7.5)
for s in ['top', 'right']: axA.spines[s].set_visible(False)
axA.set_xlabel('Hazard ratio', fontsize=8)
axA.set_title('A', fontsize=10, fontweight='bold', loc='left', x=-0.02)

# --- Panel B: 50-def heatmap (median | first), 실제 resdf ---
gsB = gs[1].subgridspec(1, 2, wspace=0.10)
cmap = LinearSegmentedColormap.from_list(
    'hr', ['#efe9da', '#d8c9a8', '#c4a978', '#a8854f', '#6b6f6e', '#2c3e50'])
hmnorm = TwoSlopeNorm(vmin=min(0.95, resdf["HR"].min()),
                      vcenter=float(resdf["HR"].median()),
                      vmax=max(1.45, resdf["HR"].max()))
_xt = [0, 1, 3, 4]; _xtl = [WINDOWS[i] for i in _xt]
for j, bm in enumerate([MAIN[0], 'first']):
    axB = fig.add_subplot(gsB[j])
    piv = (resdf[resdf.baseline == bm]
           .pivot(index="rise", columns="window", values="HR")
           .sort_index(ascending=False))
    axB.imshow(piv.values, cmap=cmap, norm=hmnorm, aspect='equal')
    axB.set_xticks(_xt); axB.set_xticklabels(_xtl, fontsize=7)
    if j == 0:
        axB.set_yticks(range(len(RISES))); axB.set_yticklabels(RISES[::-1], fontsize=7)
        axB.set_ylabel('Rise threshold (mmHg)', fontsize=7.5)
    else:
        axB.set_yticks([])
    axB.set_xlabel('Window (h)', fontsize=7.5)
    axB.set_title('Median' if j == 0 else 'First', fontsize=8, pad=2)
    for s in ['top', 'right', 'left', 'bottom']: axB.spines[s].set_visible(False)
    if j == 0:
        axB.text(-0.30, 1.18, 'B', transform=axB.transAxes, fontsize=10, fontweight='bold')

# --- Panel C: threshold / window (가로 forest) ---
axC = fig.add_subplot(gs[2])
_r100 = resdf[(resdf.baseline == MAIN[0]) & (resdf.rise == 100) & (resdf.window == 12)].iloc[0]
_r150 = resdf[(resdf.baseline == MAIN[0]) & (resdf.rise == 150) & (resdf.window == 12)].iloc[0]
try:
    _wc = (r_whole[0], r_whole[1], r_whole[2])
except NameError:
    _rw = resdf[(resdf.baseline == MAIN[0]) & (resdf.rise == 100) &
                (resdf.window == resdf["window"].max())].iloc[0]
    _wc = (_rw["HR"], _rw["lo"], _rw["hi"])
ypos = [2, 1, 0]
hrC = [_r100["HR"], _r150["HR"], _wc[0]]
loC = [_r100["lo"], _r150["lo"], _wc[1]]
hiC = [_r100["hi"], _r150["hi"], _wc[2]]
colsC = [NAVY, NAVY, BRONZE]
labC = ['100 mmHg / 12 h', '150 mmHg / 12 h', 'Whole period']
for i in range(3):
    axC.plot([loC[i], hiC[i]], [ypos[i], ypos[i]], '-', color=colsC[i], lw=1.2,
             solid_capstyle='round', alpha=0.9)
axC.scatter(hrC, ypos, c=colsC, s=22, zorder=3, edgecolors='white', linewidths=0.5)
axC.axvline(1.0, ls=(0, (4, 3)), color=REDLINE, lw=0.9, alpha=0.85)
axC.set_xscale('log'); axC.set_xlim(0.82, 2.6); axC.set_xticks([1.0, 1.5, 2.0]); _logfix(axC, 'x')
axC.set_ylim(-0.6, 2.6); axC.set_yticks(ypos); axC.set_yticklabels(labC, fontsize=7.5)
for s in ['top', 'right']: axC.spines[s].set_visible(False)
axC.set_xlabel('Hazard ratio', fontsize=8)
axC.set_title('C', fontsize=10, fontweight='bold', loc='left', x=-0.02)

fig.savefig(FIGDIR / 'Fig3.png', bbox_inches='tight', facecolor='white')
fig.savefig(FIGDIR / 'Fig3.pdf', bbox_inches='tight', facecolor='white')
plt.close()
print(f"[Fig 3] A:{hrA[0]:.2f}->{hrA[-1]:.2f} | C:{hrC[0]:.2f}/{hrC[1]:.2f}/{hrC[2]:.2f}")

*원본 셀 22*


In [ ]:
# =====================================================================
# [셀 14b] 연속 grace 곡선 (S2용): grace 0-24h, surge / termination
#   효율화: MICE 대치는 grace 무관 → 5세트 1회 대치 후 grace마다 cp만 재생성.
#   저장: surge_curve, term_curve = {grace: (hr,lo,hi)}
#   전제: 셀14 (d0, _cols, build_cp_grace, DET_E, fit 관련 상수)
# =====================================================================
from scipy.stats import norm   # 전역 norm 보호(그림 셀이 덮을 수 있음)
import numpy as np

GRACE_GRID = list(range(0, 25, 1))   # 0,2,...,24 (13점). 1h 원하면 range(0,25,1)
TERM_DEF = 24                        # 종료 정의(≤24h)

# ---- MICE 5세트 1회 대치 (grace 무관, 재사용) ----
_Xis = []
for mi in range(M_IMP):
    imp = IterativeImputer(estimator=BayesianRidge(), max_iter=20, random_state=SEED + mi,
                           sample_posterior=True, initial_strategy="median")
    _Xis.append(pd.DataFrame(imp.fit_transform(d0[_cols]), columns=_cols, index=d0.index))

def _curve(expo_col, expo_within, tag):
    """grace 격자마다 (이미 대치된 _Xis 재사용) cp 빌드 → Cox → Rubin pool."""
    out = {}
    for g in GRACE_GRID:
        est, var = [], []
        ok = True
        for mi in range(M_IMP):
            cp = build_cp_grace(d0, _Xis[mi], _cols, expo_col, expo_within, g, f"m{mi}")
            if cp["expo"].sum() == 0 or cp[cp.expo == 1]["event"].sum() == 0:
                ok = False; break
            m = CoxTimeVaryingFitter(penalizer=PENALIZER)
            m.fit(cp, id_col="id", start_col="start", stop_col="stop",
                  event_col="event", show_progress=False)
            est.append(m.summary.loc["expo", "coef"])
            var.append(m.summary.loc["expo", "se(coef)"] ** 2)
        if ok:
            hr, lo, hi, p, _, _ = pool_hr(est, var)
            out[g] = (hr, lo, hi)
    print(f"  {tag}: {len(out)}/{len(GRACE_GRID)} grace points computed")
    return out

print("=" * 60)
print(f"  [셀 14b] 연속 grace 곡선 (0-24h, {len(GRACE_GRID)}점, MICE 1회 대치 재사용)")
print("=" * 60)
import time; _t = time.time()
surge_curve = _curve("surge_h", DET_E, "surge")
term_curve = _curve("term_h", TERM_DEF, f"termination(≤{TERM_DEF}h)")
print(f"  소요: {time.time()-_t:.0f}초")
# 요약 출력
for g in GRACE_GRID:
    s = surge_curve.get(g); t = term_curve.get(g)
    _ss = f"{s[0]:.2f}[{s[1]:.2f},{s[2]:.2f}]" if s else "—"
    _tt = f"{t[0]:.2f}[{t[1]:.2f},{t[2]:.2f}]" if t else "—"
    print(f"  grace {g:>2}h | surge {_ss:>20} | term {_tt:>20}")
print("=" * 60)

*원본 셀 23*


In [ ]:
# =====================================================================
# [Supplementary 그림 셀] S1-S4 (논문용, 실제 변수 참조)
#   S1: TMP 궤적(surge/non-surge)        ← 셀12 (cmeas, tmp, surge)
#   S2: surge vs 종료(grace 0/6/24)      ← 셀14 결과
#   S3: surge vs 필터소진(겹침 + TVC)     ← 셀15 [A][B]
#   S4: 검정력·BF 설계곡선                ← 셀13 결과
#   공통 스타일: 폰트 환경기본, navy/bronze, 축 항목 유지, 라벨·범례 없음.
#   ※ 각 셀 실행 후 이 셀 실행.
# =====================================================================
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
import numpy as np

mpl.rcParams.update({
    'font.size': 8, 'axes.linewidth': 0.8, 'axes.edgecolor': '#333333',
    'xtick.major.width': 0.8, 'ytick.major.width': 0.8,
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'figure.dpi': 300, 'savefig.dpi': 300,
})
NAVY = '#2c3e50'; BRONZE = '#8c6d3f'; REDLINE = '#b03a2e'; GREY = '#cccccc'

# ---------------------------------------------------------------------
# Supplementary Fig S1: TMP trajectories over 0-12h, surge vs non-surge
#   실제 변수: tmp(셀1/3: stay_id,h,valuenum), cmeas(862), MAIN_ONSET(surge stay set)
#   traj_df 불필요 — tmp에서 직접 0-12h 시계열을 군별로 그림.
# ---------------------------------------------------------------------
# surge 집합과 cmeas 한정
_surge_ids = set(MAIN_ONSET)                      # surge 발생 stay_id 집합
_cm_ids = set(cmeas["stay_id"])
_t = tmp[(tmp["stay_id"].isin(_cm_ids)) & (tmp["h"] >= 0) & (tmp["h"] <= 12)].copy()
_t["surge"] = _t["stay_id"].isin(_surge_ids).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.8), sharey=True)
grid = np.arange(0, 12.1, 1.0)
for ax, sv in zip(axes, [0, 1]):
    sub = _t[_t["surge"] == sv]
    col = NAVY if sv == 0 else BRONZE
    means = []
    for sid, g in sub.groupby("stay_id"):
        g = g.sort_values("h")
        ax.plot(g["h"], g["valuenum"], '-', color=col, lw=0.4, alpha=0.10)
        means.append(np.interp(grid, g["h"], g["valuenum"], left=np.nan, right=np.nan))
    if means:
        M = np.nanmean(np.array(means), axis=0)
        ax.plot(grid, M, '-', color=col, lw=2.0, solid_capstyle='round')
    ax.set_xlim(0, 12); ax.set_xticks([0, 4, 8, 12])
    ax.set_xlabel('Hours from CRRT initiation', fontsize=7.5)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
axes[0].set_ylabel('Transmembrane pressure (mmHg)', fontsize=7.5)
axes[0].set_title('No surge', fontsize=8)
axes[1].set_title('Surge', fontsize=8)
# y범위는 데이터에 맞게 자동, 필요시 clip
_ymax = np.nanpercentile(_t["valuenum"], 99)
for ax in axes: ax.set_ylim(0, _ymax)
fig.tight_layout()
fig.savefig(FIGDIR / 'FigS1.png', bbox_inches='tight', facecolor='white')
fig.savefig(FIGDIR / 'FigS1.pdf', bbox_inches='tight', facecolor='white')
plt.close()
print(f"[Fig S1] surge {int(_t.groupby('stay_id')['surge'].first().sum())} / "
      f"non-surge {int((_t.groupby('stay_id')['surge'].first()==0).sum())} trajectories")

# ---------------------------------------------------------------------
# Supplementary Fig S2: surge vs circuit termination, continuous grace
#   좌: surge / 우: termination. 꺾은선(HR) + fill_between(95% CI 밴드).
#   참조: surge_curve, term_curve = {grace: (hr,lo,hi)} (셀 14b)
# ---------------------------------------------------------------------
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
import numpy as np
mpl.rcParams.update({'font.size':8,'axes.linewidth':0.8,'axes.edgecolor':'#333333',
    'xtick.major.width':0.8,'ytick.major.width':0.8,'xtick.color':'#333333','ytick.color':'#333333',
    'figure.dpi':300,'savefig.dpi':300})
NAVY='#2c3e50'; BRONZE='#8c6d3f'; REDLINE='#b03a2e'

def _xyz(curve):
    g = sorted(curve.keys())
    hr = np.array([curve[k][0] for k in g])
    lo = np.array([curve[k][1] for k in g])
    hi = np.array([curve[k][2] for k in g])
    return np.array(g), hr, lo, hi

fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.9), sharey=True)
for ax, curve, col, ttl in [(axes[0], surge_curve, NAVY, 'Early TMP surge'),
                            (axes[1], term_curve, BRONZE, 'Circuit termination')]:
    g, hr, lo, hi = _xyz(curve)
    ax.fill_between(g, lo, hi, color=col, alpha=0.15, linewidth=0)
    ax.plot(g, hr, '-', color=col, lw=1.6, solid_capstyle='round')
    ax.axhline(1.0, ls=(0, (4, 3)), color=REDLINE, lw=0.9, alpha=0.85)
    ax.set_yscale('log')
    ax.set_xlim(0, 24); ax.set_xticks([0, 6, 12, 18, 24])
    ax.set_xlabel('Grace period (h)', fontsize=7.5)
    ax.set_title(ttl, fontsize=8)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
# y축 눈금(로그) 깔끔하게
axes[0].set_yticks([0.5, 1, 2, 4])
axes[0].yaxis.set_major_formatter(mticker.FixedFormatter(['0.5', '1', '2', '4']))
axes[0].yaxis.set_minor_formatter(mticker.NullFormatter())
axes[0].set_ylabel('Hazard ratio', fontsize=7.5)
fig.tight_layout()
fig.savefig(FIGDIR / 'FigS2.png', bbox_inches='tight', facecolor='white')
fig.savefig(FIGDIR / 'FigS2.pdf', bbox_inches='tight', facecolor='white')
plt.close()
print("[Fig S2] continuous grace curves saved")

# ---------------------------------------------------------------------
# Supplementary Fig S3: early surge vs later filter exhaustion
#   (A) 집단 관계: surge / abs250 / 교집합 크기 (간단 막대 or 벤다이어그램 대용)
#   (B) 가동시간 TVC HR forest: b0/b1/b2/b3 (절대250 처리별, 전부 HR~1)
#   ※ 값 교체: 셀15 [A] _sn/_an/_ov, [B] grid_sub 결과(b0..b3 HR,lo,hi).
# ---------------------------------------------------------------------
# Supplementary Fig S3: early surge vs later filter exhaustion
fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.8), gridspec_kw={'width_ratios':[0.8,1.2]})

# (A) 집단 크기
axA = axes[0]
vals = [surge_overlap["surge_only"], surge_overlap["overlap"], surge_overlap["filter_only"]]
ypos = [2, 1, 0]; barcols = [NAVY, '#7d6b8a', BRONZE]
axA.barh(ypos, vals, color=barcols, height=0.6, edgecolor='white')
axA.set_yticks(ypos)
axA.set_yticklabels(['Surge only', 'Overlap', 'Filter\nexhaustion only'], fontsize=7)
axA.set_xlabel('Patients (n)', fontsize=7.5)
for s in ['top', 'right']: axA.spines[s].set_visible(False)

# (B) 가동시간 TVC HR forest: b0/b1/b2/b3
axB = axes[1]
_keys = ['b0', 'b1', 'b2', 'b3']
b_hr = [filter_tvc[k][0] for k in _keys]
b_lo = [filter_tvc[k][1] for k in _keys]; b_hi = [filter_tvc[k][2] for k in _keys]
blab = ['Full cohort', 'Below 250 mmHg', 'Reached 250 mmHg', 'Measured TMP']
yB = np.arange(4)[::-1]
for i in range(4):
    axB.plot([b_lo[i], b_hi[i]], [yB[i], yB[i]], '-', color=NAVY, lw=1.2,
             solid_capstyle='round', alpha=0.9)
axB.scatter(b_hr, yB, c=NAVY, s=20, zorder=3, edgecolors='white', linewidths=0.5)
axB.axvline(1.0, ls=(0, (4, 3)), color=REDLINE, lw=0.9, alpha=0.85)
axB.set_xlim(0.985, 1.020); axB.set_xticks([0.99, 1.00, 1.01])
axB.set_ylim(-0.5, 3.5); axB.set_yticks(yB); axB.set_yticklabels(blab, fontsize=7)
for s in ['top', 'right']: axB.spines[s].set_visible(False)
axB.set_xlabel('Hazard ratio per 24 h', fontsize=7.5)
fig.tight_layout()
fig.savefig(FIGDIR / 'FigS3.png', bbox_inches='tight', facecolor='white')
fig.savefig(FIGDIR / 'FigS3.pdf', bbox_inches='tight', facecolor='white')
plt.close()
print("[Fig S3] saved")

# ---------------------------------------------------------------------
# Supplementary Fig S4: design analysis (power and Bayes factor vs N)
#   (A) power vs N (관측 효과크기 고정), 현 N=862 표시
#   (B) BF10 vs N
#   ※ 값 교체: 셀13 design_N, design_power, design_bf10 (배열).
# ---------------------------------------------------------------------
# Supplementary Fig S4: design analysis (power and Bayes factor vs N)
fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.8))

axA = axes[0]
axA.plot(design_N, design_power, 'o-', color=NAVY, lw=1.4, ms=4,
         markeredgecolor='white', markeredgewidth=0.4)
axA.axhline(80, ls=(0, (4, 3)), color=GREY, lw=0.8)
axA.axvline(design_N0, ls=':', color=REDLINE, lw=0.9, alpha=0.8)
axA.set_xlabel('Sample size', fontsize=7.5)
axA.set_ylabel('Power (%)', fontsize=7.5); axA.set_ylim(0, 105)
for s in ['top', 'right']: axA.spines[s].set_visible(False)

axB = axes[1]
axB.plot(design_N, design_bf10, 'o-', color=NAVY, lw=1.4, ms=4,
         markeredgecolor='white', markeredgewidth=0.4)
axB.axhline(3, ls=(0, (4, 3)), color=GREY, lw=0.8)
axB.axvline(design_N0, ls=':', color=REDLINE, lw=0.9, alpha=0.8)
axB.set_yscale('log')
axB.set_xlabel('Sample size', fontsize=7.5)
axB.set_ylabel('Bayes factor (BF10)', fontsize=7.5)
for s in ['top', 'right']: axB.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(FIGDIR / 'FigS4.png', bbox_inches='tight', facecolor='white')
fig.savefig(FIGDIR / 'FigS4.pdf', bbox_inches='tight', facecolor='white')
plt.close()
print("[Fig S4] saved")

## 7. 표 재현 · 자체 정합성 QC (Table reproduction & QC checks)


*원본 셀 14*


In [ ]:
# =====================================================================
# [셀 10] Table 1: (a) surge 유무 / (b) measured TMP 유무
#   (a) cmeas(862) surge 발생 유무 — 본문 Table 1
#   (b) c(1493) base_median_meas 보유 유무 — 측정 시대편중·일반화 한계(Table S1)
#   B2 반영: Tier2 28d 사망률(51.9%) + surge군별 사망률을 표에 명시.
#   변경점: tab1(셀0) 재사용. 28d 사망률을 표 마지막 행으로 추가(본문 Table 직결).
#   전제: 셀0(tab1), 셀3(c,cmeas,CONT_M,BIN_M), 셀4(year_grp), 셀6(MAIN_ONSET)
# =====================================================================
cmeas["surge_flag"] = cmeas["stay_id"].map(lambda s: 1 if s in MAIN_ONSET else 0)
c["has_base"] = c["base_median_meas"].notna().astype(int)

def _add_mortality_row(tabdf, df, group_col, ev="event28"):
    """tab1 결과에 28d 사망률 행 추가 (all/group0/group1 + chi2 p)."""
    al = df[ev].mean() * 100
    g0 = df[df[group_col] == 0][ev].mean() * 100
    g1 = df[df[group_col] == 1][ev].mean() * 100
    try:
        from scipy.stats import chi2_contingency
        p = chi2_contingency(pd.crosstab(df[group_col], df[ev]))[1]
    except Exception:
        p = np.nan
    row = pd.DataFrame([["28d_mortality_%", f"{al:.1f}", f"{g0:.1f}", f"{g1:.1f}", p]],
                       columns=tabdf.columns)
    return pd.concat([tabdf, row], ignore_index=True)

# ---- Table 1a: surge 유무 (cmeas) ----
print("=" * 84)
print(f"  [Table 1a] surge 유무 (cmeas={len(cmeas)}, surge={int(cmeas['surge_flag'].sum())}, "
      f"Tier2 28d 사망={cmeas['event28'].mean()*100:.1f}%)")
print("  group=1: surge / group=0: no surge")
print("=" * 84)
t1a = tab1(cmeas, "surge_flag", CONT_M, BIN_M, cat_vars=["year_grp"])
t1a = _add_mortality_row(t1a, cmeas, "surge_flag")
print(t1a.to_string(index=False))

# ---- Table 1b: measured TMP 유무 (c) ----
print("\n" + "=" * 84)
print(f"  [Table 1b] measured TMP 유무 (c={len(c)}, measured={int(c['has_base'].sum())})")
print("  group=1: base_median_meas 보유 / group=0: 미보유")
print("=" * 84)
t1b = tab1(c, "has_base", CONT_M, BIN_M, cat_vars=["year_grp"])
t1b = _add_mortality_row(t1b, c, "has_base")
print(t1b.to_string(index=False))
print("=" * 84)

# ---- B2: 사망률 명시 (본문 직결 숫자) ----
print(f"\n  [B2 — 본문 사망률 명시]")
print(f"    Tier 1 (c, N={len(c)}):          28d {c['event28'].mean()*100:.1f}%, "
      f"원내 {c['death_inhosp'].mean()*100:.1f}%")
print(f"    Tier 2 (cmeas, N={len(cmeas)}):   28d {cmeas['event28'].mean()*100:.1f}%")
print(f"      surge(+) n={int(cmeas['surge_flag'].sum())}: "
      f"28d {cmeas[cmeas.surge_flag==1]['event28'].mean()*100:.1f}%")
print(f"      surge(-) n={int((cmeas['surge_flag']==0).sum())}: "
      f"28d {cmeas[cmeas.surge_flag==0]['event28'].mean()*100:.1f}%")
print(f"    measured 보유 yes {c[c.has_base==1]['event28'].mean()*100:.1f}% / "
      f"no {c[c.has_base==0]['event28'].mean()*100:.1f}%")

# ---- 시각화: 두 비교의 핵심 차이 ----
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
def _num(df, col, g):
    return df[df["surge_flag"] == g][col].dropna().median()
sa_vars = [("weight_kg", "weight"), ("anchor_age", "age"), ("sofa_total", "SOFA"),
           ("lactate", "lactate"), ("map_value", "MAP")]
x = np.arange(len(sa_vars)); w = 0.38
v0 = [_num(cmeas, c0, 0) for c0, _ in sa_vars]; v1 = [_num(cmeas, c0, 1) for c0, _ in sa_vars]
ax[0].bar(x - w/2, v0, w, label="no surge", color="#2c3e50")
ax[0].bar(x + w/2, v1, w, label="surge", color="#c0392b")
ax[0].set_xticks(x); ax[0].set_xticklabels([l for _, l in sa_vars], fontsize=9)
ax[0].set_ylabel("median"); ax[0].legend(fontsize=8)
ax[0].set_title("Table 1a: surge vs no-surge (medians)", fontsize=10, loc="left")

yc1 = c[c.has_base == 1]["year_grp"].value_counts(normalize=True).sort_index() * 100
yc0 = c[c.has_base == 0]["year_grp"].value_counts(normalize=True).reindex(yc1.index, fill_value=0) * 100
x2 = np.arange(len(yc1))
ax[1].bar(x2 - w/2, yc0.values, w, label="no measured TMP", color="#7f8c8d")
ax[1].bar(x2 + w/2, yc1.values, w, label="measured TMP", color="#16a085")
ax[1].set_xticks(x2); ax[1].set_xticklabels(yc1.index, rotation=30, ha="right", fontsize=8)
ax[1].set_ylabel("% within group"); ax[1].legend(fontsize=8)
ax[1].set_title("Table 1b: measured TMP era skew", fontsize=10, loc="left")
plt.tight_layout(); plt.savefig(FIGDIR / "table1_compare.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"  저장: {FIGDIR/'table1_compare.png'}")

*원본 셀 24*


In [ ]:
# Table 1 (surge 유무) + S1 (measured 유무) 정확값 출력
# 셀 10의 t1a, t1b가 이미 있으면 그대로 출력. 없으면 tab1 재호출.
import pandas as pd

print("="*70); print("[Table 1] surge 유무 — tab1 전체 컬럼"); print("="*70)
print(t1a.to_string(index=False))
print("\n" + "="*70); print("[Table S1] measured 유무 — tab1 전체 컬럼"); print("="*70)
print(t1b.to_string(index=False))

# tab1 출력에 IQR이 없으면, 아래로 연속변수 median(IQR)을 군별로 직접 계산
print("\n" + "="*70); print("[보강] 연속변수 median (Q1-Q3) 군별 — Table 1 (cmeas, surge_flag)"); print("="*70)
for col in CONT_M:
    if col not in cmeas.columns: continue
    for g, gname in [(0,"no surge"), (1,"surge")]:
        v = cmeas[cmeas["surge_flag"]==g][col].dropna()
        if len(v):
            print(f"  {col:<16} {gname:<9} {v.median():.1f} ({v.quantile(.25):.1f}-{v.quantile(.75):.1f})  n={len(v)}")
    # overall
    v = cmeas[col].dropna()
    print(f"  {col:<16} {'overall':<9} {v.median():.1f} ({v.quantile(.25):.1f}-{v.quantile(.75):.1f})  n={len(v)}")
    print()

print("="*70); print("[보강] 이진변수 n(%) 군별 — Table 1"); print("="*70)
for col in BIN_M:
    if col not in cmeas.columns: continue
    row = f"  {col:<20}"
    for g, gname in [(None,"overall"), (0,"no surge"), (1,"surge")]:
        d = cmeas if g is None else cmeas[cmeas["surge_flag"]==g]
        n1 = int(d[col].sum()); N = len(d)
        row += f" | {gname} {n1}/{N} ({n1/N*100:.0f}%)"
    print(row)

*원본 셀 25*


In [ ]:
# Supplementary Table S3: 50조합 전체 (resdf 그대로)
print("="*90); print("[Table S3] 50 definitions"); print("="*90)
_cols_show = ["baseline","rise","window","n_surge","HR","lo","hi","p","q"]
_cols_show = [c for c in _cols_show if c in resdf.columns]
_show = resdf[_cols_show].copy()
# 보기 좋게 정렬: baseline(median 먼저) → rise → window
_show = _show.sort_values(["baseline","rise","window"],
                          key=lambda s: s.map({"median":0,"first":1}) if s.name=="baseline" else s)
for col in ["HR","lo","hi"]:
    if col in _show: _show[col] = _show[col].round(3)
for col in ["p","q"]:
    if col in _show: _show[col] = _show[col].round(3)
print(_show.to_string(index=False))

*원본 셀 26*


In [ ]:
from statsmodels.stats.multitest import multipletests
_r = resdf.copy()
for bm in ["median","first"]:
    mask = _r["baseline"]==bm
    _r.loc[mask,"q"] = multipletests(_r.loc[mask,"p"], method="fdr_bh")[1]
print(_r[["baseline","rise","window","n_surge","HR","lo","hi","p","q"]].round(3).to_string(index=False))

*원본 셀 27*


In [ ]:
# ===== Supplementary Table S2 재현: Tier 1 전체 16칸 =====
import pandas as pd
print("="*78); print("[Table S2] MIMIC-IV 12 settings (r_l1m)"); print("="*78)
_m = r_l1m.copy()
_m["Endpoint"] = _m["endpoint"].map({"28d":"28-day","원내":"In-hospital"}).fillna(_m["endpoint"])
_m["Source"] = _m["source"].map(lambda s: "Procedure" if str(s).startswith("proc") else "Union")
_m = _m[["Endpoint","Source","gap","N","ev","HR","lo","hi","p"]]
_m.columns = ["Endpoint","Source","Gap(h)","N","Events","HR","lo","hi","P"]
for col in ["HR","lo","hi"]: _m[col]=_m[col].round(3)
_m["P"]=_m["P"].round(3)
print(_m.to_string(index=False))

print("\n"+"="*78); print("[Table S2] eICU 4 settings (r_l1e)"); print("="*78)
_e = r_l1e.copy()
_e["Setting"] = _e.apply(lambda r: f"≥{int(r['min_rec'])} records, "
                         f"{'last' if str(r['term']).startswith('a') else '+gap'}", axis=1)
_e = _e[["Setting","N","ev","HR","lo","hi","p"]]
_e.columns = ["Setting","N","Events","HR","lo","hi","P"]
for col in ["HR","lo","hi"]: _e[col]=_e[col].round(3)
_e["P"]=_e["P"].round(3)
print(_e.to_string(index=False))

# ===== Supplementary Table S1 재현: measured 유무, IQR 포함 =====
# c / has_base (1=보유, 0=미보유). 연속 median(Q1-Q3), 이진 n(%).
print("\n"+"="*78); print("[Table S1] measured TMP 유무 (c, has_base) — IQR 포함"); print("="*78)
from scipy.stats import mannwhitneyu, chi2_contingency
g1 = c[c["has_base"]==1]; g0 = c[c["has_base"]==0]
print(f"  With measured (n={len(g1)}) vs Without (n={len(g0)})\n")

print("  -- 연속변수: median (Q1-Q3) --")
for col in CONT_M:
    if col not in c.columns: continue
    v1 = g1[col].dropna(); v0 = g0[col].dropna()
    if len(v1)<2 or len(v0)<2: continue
    try: _, p = mannwhitneyu(v1, v0, alternative="two-sided")
    except: p = float("nan")
    print(f"  {col:<16} With {v1.median():>6.1f} ({v1.quantile(.25):.1f}-{v1.quantile(.75):.1f})"
          f"  | Without {v0.median():>6.1f} ({v0.quantile(.25):.1f}-{v0.quantile(.75):.1f})"
          f"  | p={p:.3g}")

print("\n  -- 이진변수: n (%) --")
for col in BIN_M:
    if col not in c.columns: continue
    n1=int(g1[col].sum()); n0=int(g0[col].sum())
    try:
        ct = pd.crosstab(c["has_base"], c[col])
        _, p, _, _ = chi2_contingency(ct)
    except: p=float("nan")
    print(f"  {col:<18} With {n1}/{len(g1)} ({n1/len(g1)*100:.0f}%)"
          f"  | Without {n0}/{len(g0)} ({n0/len(g0)*100:.0f}%)  | p={p:.3g}")

# era (2014+)
print("\n  -- Admission 2014+ --")
for lab, g in [("With", g1), ("Without", g0)]:
    if "year_grp" in g.columns:
        late = g["year_grp"].astype(str).str.contains("2014|2015|2016|2017|2018|2019", regex=True).sum()
        print(f"  {lab}: {int(late)}/{len(g)} ({late/len(g)*100:.0f}%)")

*원본 셀 28*


In [ ]:
print("="*60)
print("[7-8] heparin 367/367, prophylaxis 184/185 검증")
print("="*60)

# Tier2 코호트 (862) — Table 1 분모
t2 = cmeas  # 862명
# measured TMP 코호트 (873) — S1 분모 (has_base=1)
mb = c[c["has_base"]==1]  # 873명

# systemic heparin
print("\n--- systemic heparin ---")
print(f"Tier2(862) systemic: {int(t2['v3_systemic_hep'].sum())} / {len(t2)} = {t2['v3_systemic_hep'].mean()*100:.1f}%")
print(f"measured(873) systemic: {int(mb['v3_systemic_hep'].sum())} / {len(mb)} = {mb['v3_systemic_hep'].mean()*100:.1f}%")

# prophylaxis heparin
print("\n--- prophylaxis heparin ---")
print(f"Tier2(862) prophylaxis: {int(t2['v3_prophylaxis'].sum())} / {len(t2)} = {t2['v3_prophylaxis'].mean()*100:.1f}%")
print(f"measured(873) prophylaxis: {int(mb['v3_prophylaxis'].sum())} / {len(mb)} = {mb['v3_prophylaxis'].mean()*100:.1f}%")

# 873→862로 빠진 11명의 heparin 사용 여부 (367/367이 우연인지)
_lost = mb[~mb["stay_id"].isin(set(t2["stay_id"]))]
print(f"\n873→862 제외된 {len(_lost)}명 중:")
print(f"  systemic 사용: {int(_lost['v3_systemic_hep'].sum())}명")
print(f"  prophylaxis 사용: {int(_lost['v3_prophylaxis'].sum())}명")

*원본 셀 29*


In [ ]:
print("="*60)
print("[9] operating time 28d P값: Abstract .16 vs 본문 .164")
print("="*60)
# r_l1m에서 procedure/gap6/28d (본문 주 추정치)
_main28 = r_l1m[(r_l1m["source"].str.startswith("proc")) & 
                (r_l1m["gap"]==6) & (r_l1m["endpoint"]=="28d")].iloc[0]
print(f"HR={_main28['HR']:.4f}, P={_main28['p']:.4f}")
print(f"Abstract 표기: P=.16 (반올림), 본문 표기: P=.164")
print("→ 동일 값의 자릿수 차이, 정상" if abs(_main28['p']-0.164)<0.001 else "→ 불일치 확인 필요")

*원본 셀 30*


In [ ]:
print("="*60)
print("[Table S5] 민감도 분석 수치 출력")
print("="*60)

def _fmt(name, n, hr, lo, hi, p, ev=None):
    evs = f"{ev}" if ev is not None else "—"
    print(f"  {name:<38} N={n:<5} ev={evs:<5} HR={hr:.2f} [{lo:.2f}-{hi:.2f}] P={p:.3f}")

# Primary (셀 6 r_main 또는 pooltab)
print("\n-- Primary (fully adjusted, M5) --")
try:
    print(f"  r_main: HR={r_main[0]:.3f} [{r_main[1]:.3f}-{r_main[2]:.3f}] p={r_main[3]:.3f}")
except Exception as e:
    print("  r_main 없음:", e)

# 셀 8 민감도 결과 — 변수명 후보 탐색
print("\n-- 셀 8 민감도 변수 탐색 --")
import re
_cand = [v for v in dir() if re.search(r'sens|sepsis|complete|cc|d7|seven|whole|baseTMP|btmp|evalue|eval', v, re.I)]
print("  후보 변수:", _cand)

# 흔한 형태: 결과 DataFrame이 있으면 출력
for vn in ['sens_tab','r_sens','sensdf','S5','tab_s5','r_s5']:
    if vn in dir():
        print(f"\n  {vn} 발견:")
        print(eval(vn).to_string(index=False) if hasattr(eval(vn),'to_string') else eval(vn))

*원본 셀 31*


In [ ]:
print("="*60)
print("[Table S5] 민감도 분석 정확한 값 추출")
print("="*60)

def _show(name, obj):
    # 튜플/리스트 형태 (HR,lo,hi,p,...) 가정
    try:
        hr,lo,hi,p = obj[0],obj[1],obj[2],obj[3]
        print(f"  {name:<32} HR={hr:.3f} [{lo:.3f}-{hi:.3f}] p={p:.3f}")
    except Exception as e:
        print(f"  {name:<32} = {obj}  ({type(obj).__name__})")

print("\n-- 각 민감도 결과 --")
for vn in ['r_main','r_sepsis_s','r_sepsis_b','r_complete']:
    if vn in dir(): _show(vn, eval(vn))

# 7-day, baseline-TMP adj, whole-period 변수 탐색
print("\n-- 추가 변수 탐색 (7day/baseTMP/whole) --")
import re
_more = [v for v in dir() if re.search(r'd7|day7|seven|btmp|basetmp|tmp_adj|whole|r_w', v, re.I)]
print("  후보:", _more)
for vn in _more:
    try: _show(vn, eval(vn))
    except: pass

# E-value
print("\n-- E-value --")
if 'evalue' in dir():
    print(f"  evalue = {evalue}  ({type(evalue).__name__})")

# 각 부분군 N과 event 수 (S5 표의 ev 칸 채우기)
print("\n-- 부분군 N / events --")
# sepsis-strict
if '_need_sepsis' in dir() or 'r_sepsis_s' in dir():
    try:
        print(f"  sepsis-strict: r_sepsis_s 길이정보 = {r_sepsis_s}")
    except: pass
# complete-case
if 'cc' in dir():
    try:
        print(f"  complete-case n={len(cc)}, 28d deaths={int(cc['event28'].sum()) if 'event28' in cc.columns else '?'}")
    except Exception as e:
        print("  cc:", e)
if '_cc_mask' in dir():
    try:
        print(f"  _cc_mask 합(n)={int(_cc_mask.sum())}")
    except: pass

*원본 셀 32*


In [ ]:
# S5 부분군 event 수 (28일 사망)
print("sepsis-strict(590) deaths:", end=" ")
try:
    _ss = cmeas[(cmeas.get('_sepsis_strict', cmeas.get('sepsis_strict'))==1)] if '_sepsis_strict' in cmeas.columns or 'sepsis_strict' in cmeas.columns else None
    # 변수명 모르면 mask로
    print(int(_ss['event28'].sum()) if _ss is not None else "변수명 확인 필요")
except Exception as e:
    print(e)

# complete-case는 _cc_mask 사용
print("complete-case(576) deaths:", int(cmeas[_cc_mask]['event28'].sum()) if '_cc_mask' in dir() else "?")

# 7-day는 종점이 event7
print("primary(862) 7-day deaths:", int(cmeas['event7'].sum()) if 'event7' in cmeas.columns else "event7 변수명 확인")

## 8. 리비전 대응 신규 분석군 (260701-0704 추가)


# 추가 확인 (260701)

*원본 셀 34*


In [ ]:
# =====================================================================
# [셀 16] 2.1 cluster-robust variance + tie method 명시  (리뷰 2.1 대응)
#   배경: CoxTimeVaryingFitter(start,stop]는 이미 counting-process 구조라
#     동일 환자 다중 행이 들어간다(build_grid_mimic=Tier1, build=Tier2 surge).
#     리뷰어 우려 = model-based SE가 환자 내 상관을 무시해 과소추정될 수 있음.
#   ★ 검증된 사실(lifelines 0.30): CoxTimeVaryingFitter.fit(robust=True)는
#     "NotImplementedError: Not available yet." → 내장 sandwich 사용 불가.
#     따라서 환자 단위(cluster) bootstrap으로 robust SE를 직접 산출한다.
#   tie method: CoxTimeVaryingFitter는 Breslow 고정(Efron 미지원).
#     → Methods에 "Breslow approximation for ties" 명시(재분석 아님, 문구만).
#   설계:
#     - Tier 2 surge(M5): 환자 id를 복원추출 → build → CoxTimeVaryingFitter
#       (penalizer=PENALIZER) 적합 → surge coef 수집. B회 반복.
#       MICE는 계산량 때문에 대표 1세트(XC_MEAS[0]) 사용(점추정 안정성은
#       Rubin 풀링 결과가 이미 담보; 여기 목적은 SE 비교).
#     - Tier 1(per-24h): 동일 방식으로 cum_dur_h coef bootstrap.
#     - 산출: model-based SE vs cluster-bootstrap SE, 그 비율과 재산출 CI.
#   해석 프레임: robust SE가 model SE와 비슷하면(비율≈1) "환자 내 상관으로
#     인한 과소추정 없음, 결과 유지"로 방어. 크게 커지면 robust CI를 주 결과로.
#   전제: 셀0(build, build_grid_mimic, cum_on_at, CoxTimeVaryingFitter,
#         PENALIZER, CONT_M, BIN_M, SEG_MAIN), 셀3(cmeas, c, XC_MEAS),
#         셀3A(SEG_MAIN, r_l1m 또는 동등 per24h 결과), 셀6(MAIN_ONSET, r_main)
# =====================================================================
import numpy as np, pandas as pd
from lifelines import CoxTimeVaryingFitter

B_BOOT   = 500          # Tier 2 surge bootstrap 반복수 (빠름)
B_BOOT_T1 = 200         # Tier 1 격자 bootstrap 반복수 (build_grid_mimic가 무거워 별도·축소)
BOOT_SEED = 20260629

# M5 공변량 (셀6과 동일 구성)
_CONT_M5 = DEMO_C + SEV_C + LAB_C + TX_C
_BIN_M5  = DEMO_B + SEV_B + TX_B
_cont_m5 = [c_ for c_ in _CONT_M5 if c_ in cmeas.columns or c_ in XC_MEAS[0].columns]
_bin_m5  = [c_ for c_ in _BIN_M5 if c_ in cmeas.columns]

def _tvc_coef_once(d_boot, Xc_boot, onset_map, cont, binc, exposure,
                   builder, segs=None, end_col="end28_h", ev_col="event28",
                   penalizer=None):
    """부트스트랩 표본 1개에서 노출 coef 1개 반환(적합 실패 시 None)."""
    pen = PENALIZER if penalizer is None else penalizer
    if builder == "surge":
        cp = build(d_boot, Xc_boot, onset_map, cont, binc, end_col, ev_col)
        expo = "surge"
    else:
        cp = build_grid_mimic(d_boot, Xc_boot, segs, cont, binc, end_col, ev_col)
        expo = "cum_dur_h"
    if len(cp) == 0 or cp["event"].sum() == 0:
        return None
    if builder == "surge" and (cp["surge"].sum() == 0
                               or cp[cp.surge == 1]["event"].sum() == 0):
        return None
    fit = [expo] + [cc for cc in (cont + binc) if cp[cc].nunique() > 1]
    try:
        m = CoxTimeVaryingFitter(penalizer=pen)
        m.fit(cp[["id", "start", "stop", "event"] + fit], id_col="id",
              start_col="start", stop_col="stop", event_col="event",
              show_progress=False)
        return float(m.summary.loc[expo, "coef"])
    except Exception:
        return None

def cluster_bootstrap_se(d, Xc_full, onset_map, cont, binc, exposure, builder,
                         segs=None, end_col="end28_h", ev_col="event28",
                         scale=1.0, B=B_BOOT, seed=BOOT_SEED):
    """환자 단위 복원추출 bootstrap → 노출 coef의 SE(로그스케일) 및 백분위 CI.
       ★ 재추출된 환자에 새 정수 id를 부여해 (start,stop] 중복을 방지."""
    rng = np.random.default_rng(seed)
    n = len(d)
    d = d.reset_index(drop=True)
    Xc_full = Xc_full.reset_index(drop=True)
    coefs = []
    for b in range(B):
        idx = rng.integers(0, n, size=n)            # 환자 인덱스 복원추출
        d_b = d.iloc[idx].copy().reset_index(drop=True)
        # 재추출로 같은 stay_id가 여러 번 → build 계열이 id로 stay_id를 쓰므로
        # 고유 id 재부여를 위해 stay_id를 임시 고유값으로 교체하고 onset도 매핑.
        new_ids = np.arange(len(d_b))
        old_ids = d_b["stay_id"].values.copy()
        d_b["stay_id"] = new_ids
        Xc_b = Xc_full.iloc[idx].reset_index(drop=True)
        if builder == "mimic":
            # segs도 새 id로 재매핑 (onset 불필요)
            segs_b = {int(new_ids[k]): segs.get(int(old_ids[k]), [])
                      for k in range(len(d_b))}
            cf = _tvc_coef_once(d_b, Xc_b, None, cont, binc, exposure,
                                "mimic", segs=segs_b, end_col=end_col, ev_col=ev_col)
        else:
            onset_b = {int(new_ids[k]): onset_map.get(int(old_ids[k]), np.nan)
                       for k in range(len(d_b))}
            cf = _tvc_coef_once(d_b, Xc_b, onset_b, cont, binc, exposure,
                                "surge", end_col=end_col, ev_col=ev_col)
        if cf is not None:
            coefs.append(cf)
    coefs = np.array(coefs)
    se = coefs.std(ddof=1)
    lo = np.exp(np.percentile(coefs, 2.5) * scale)
    hi = np.exp(np.percentile(coefs, 97.5) * scale)
    return se, lo, hi, len(coefs)

print("=" * 72)
print("  [2.1] cluster(환자)-robust SE via bootstrap  (B=%d)" % B_BOOT)
print("  tie method: Breslow (CoxTimeVaryingFitter 고정, Methods 명시)")
print("=" * 72)

# ---- Tier 2 surge (M5) : model SE(=r_main 유래) vs bootstrap SE ----
# model-based: 셀6 r_main은 Rubin 풀링. 여기선 단일세트 model SE와 비교 위해
#   XC_MEAS[0]로 single-imputation model SE도 함께 산출.
_cp0 = build(cmeas, XC_MEAS[0][_cont_m5], MAIN_ONSET, _cont_m5, _bin_m5)
_fit = ["surge"] + [cc for cc in (_cont_m5 + _bin_m5) if _cp0[cc].nunique() > 1]
_m0 = CoxTimeVaryingFitter(penalizer=PENALIZER)
_m0.fit(_cp0[["id", "start", "stop", "event"] + _fit], id_col="id",
        start_col="start", stop_col="stop", event_col="event", show_progress=False)
_coef0 = float(_m0.summary.loc["surge", "coef"])
_se_model = float(_m0.summary.loc["surge", "se(coef)"])

_se_boot, _lo_b, _hi_b, _nb = cluster_bootstrap_se(
    cmeas, XC_MEAS[0], MAIN_ONSET, _cont_m5, _bin_m5, "surge", "surge")

print("\n[Tier 2 surge M5]")
print(f"  model SE (single MICE set) = {_se_model:.4f}")
print(f"  bootstrap cluster SE       = {_se_boot:.4f}  (ratio={_se_boot/_se_model:.3f}, n_ok={_nb})")
print(f"  point HR = {np.exp(_coef0):.3f}")
print(f"  bootstrap 95% CI (percentile) = [{_lo_b:.3f}, {_hi_b:.3f}]")
print(f"  ※ 참고: 주 결과 r_main(Rubin 풀링) HR={r_main[0]:.3f} "
      f"[{r_main[1]:.3f},{r_main[2]:.3f}]")

# ---- Tier 1 per-24h : model SE vs bootstrap SE (SEG_MAIN=procedure/gap6) ----
_cont_t1 = [c_ for c_ in CONT_M if c_ in c.columns or c_ in XC_C[0].columns]
_bin_t1  = [c_ for c_ in BIN_M if c_ in c.columns]
_cpg = build_grid_mimic(c, XC_C[0][_cont_t1], SEG_MAIN, _cont_t1, _bin_t1,
                        "end28_h", "event28")
_fitg = ["cum_dur_h"] + [cc for cc in (_cont_t1 + _bin_t1) if _cpg[cc].nunique() > 1]
_mg = CoxTimeVaryingFitter(penalizer=PENALIZER)
_mg.fit(_cpg[["id", "start", "stop", "event"] + _fitg], id_col="id",
        start_col="start", stop_col="stop", event_col="event", show_progress=False)
_coefg = float(_mg.summary.loc["cum_dur_h", "coef"])
_se_model_g = float(_mg.summary.loc["cum_dur_h", "se(coef)"])

_se_boot_g, _lo_g, _hi_g, _nbg = cluster_bootstrap_se(
    c, XC_C[0], None, _cont_t1, _bin_t1, "cum_dur_h", "mimic",
    segs=SEG_MAIN, end_col="end28_h", ev_col="event28", scale=24, B=B_BOOT_T1)

print("\n[Tier 1 cumulative operating time, per 24h]")
print(f"  model SE (per-hour coef)   = {_se_model_g:.5f}")
print(f"  bootstrap cluster SE       = {_se_boot_g:.5f}  (ratio={_se_boot_g/_se_model_g:.3f}, n_ok={_nbg})")
print(f"  point HR per 24h = {np.exp(_coefg*24):.4f}")
print(f"  bootstrap 95% CI (per 24h) = [{_lo_g:.4f}, {_hi_g:.4f}]")
print("=" * 72)
print("해석 가이드:")
print("  ratio≈1  → 환자 내 상관에 의한 SE 과소추정 없음 → 기존 결과 유지 방어.")
print("  ratio>1.15 정도로 커지고 CI 상한이 1을 넘으면 → robust(bootstrap) CI를")
print("            주 결과로 제시하고 surge 신호의 exploratory 성격을 더 강조.")
print("=" * 72)

# 논문 보고용 요약 dict (Supplementary Table S6 후보)
robust_se_tab = {
    "tier2_surge": {"HR": np.exp(_coef0), "se_model": _se_model,
                    "se_boot": _se_boot, "ci_boot": (_lo_b, _hi_b), "n_boot": _nb},
    "tier1_per24h": {"HR": np.exp(_coefg*24), "se_model": _se_model_g,
                     "se_boot": _se_boot_g, "ci_boot": (_lo_g, _hi_g), "n_boot": _nbg},
}

*원본 셀 35*


In [ ]:
# =====================================================================
# [셀 17] 2.8 다중대치 횟수 m=5 → m=20  (리뷰 2.8 대응)
#   배경: 리뷰어 = m=5는 Monte Carlo error가 클 수 있음. 20~50 권장.
#   설계: 기존 파이프라인 재실행 없이, primary(surge M5)와 핵심 민감도만
#     m=20 대치세트로 재적합해 m=5 결과와 HR·CI·p를 비교.
#     결측률이 낮으면 점추정은 거의 불변, MC error(세트 간 분산)만 감소 →
#     "m=5로도 안정적"을 수치로 입증(또는 m=20을 주 분석으로 승격).
#   비용: 순수 계산(대치 15세트 추가 + Cox 15회). 데이터 재추출 없음.
#   전제: 셀0(mice_impute, fit_surge_cox, CONT_M, BIN_M, M5 블록),
#         셀3(cmeas, XC_MEAS[기존 m=5]), 셀6(MAIN_ONSET, r_main)
# =====================================================================
import numpy as np, pandas as pd

M_HIGH = 20

_CONT_M5 = DEMO_C + SEV_C + LAB_C + TX_C
_BIN_M5  = DEMO_B + SEV_B + TX_B
_cont_m5 = [x for x in _CONT_M5 if x in cmeas.columns or x in XC_MEAS[0].columns]
_bin_m5  = [x for x in _BIN_M5 if x in cmeas.columns]

print("=" * 72)
print("  [2.8] 다중대치 m=5 vs m=%d  (Monte Carlo 안정성)" % M_HIGH)
print("=" * 72)

# ---- 변수별 결측률 (m 필요성의 근거; 리뷰어가 함께 요구) ----
print("\n[Tier 2 연속공변량 결측률 (cmeas=%d)]" % len(cmeas))
miss = {}
for v in _cont_m5:
    if v in cmeas.columns:
        r = float(cmeas[v].isna().mean()) * 100
        miss[v] = r
        print(f"    {v:<14} {r:5.1f}%")
_maxmiss = max(miss.values()) if miss else 0.0
print(f"  최대 결측률 = {_maxmiss:.1f}%  "
      f"→ {'낮음(m=5도 대체로 충분하나 m=20으로 확정 권장)' if _maxmiss < 20 else '높음(m=20~50 필요)'}")

# ---- m=20 대치세트 생성 (셀0 mice_impute, 종점 28d 통일) ----
#   mice_impute(df, cont_cols, event_col, time_col, m)
XC_MEAS_20 = mice_impute(cmeas, _cont_m5, "event28", "end28_h", m=M_HIGH)

# ---- m=5 (기존 XC_MEAS) vs m=20 primary surge M5 ----
r_m5_5  = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET, _cont_m5, _bin_m5)
r_m5_20 = fit_surge_cox(cmeas, XC_MEAS_20, MAIN_ONSET, _cont_m5, _bin_m5)

def _mc_between_var(d, XC_list, onset, cont, binc):
    """세트 간 분산 B(로그 HR)만 별도 산출 = Monte Carlo error 지표."""
    ests = []
    for mi in range(len(XC_list)):
        Xc = XC_list[mi][cont]
        cp = build(d, Xc, onset, cont, binc)
        fit = ["surge"] + [cc for cc in (cont + binc) if cp[cc].nunique() > 1]
        from lifelines import CoxTimeVaryingFitter
        m = CoxTimeVaryingFitter(penalizer=PENALIZER)
        m.fit(cp[["id", "start", "stop", "event"] + fit], id_col="id",
              start_col="start", stop_col="stop", event_col="event",
              show_progress=False)
        ests.append(m.summary.loc["surge", "coef"])
    return np.var(ests, ddof=1), np.mean(ests)

B5, Q5   = _mc_between_var(cmeas, XC_MEAS,    MAIN_ONSET, _cont_m5, _bin_m5)
B20, Q20 = _mc_between_var(cmeas, XC_MEAS_20, MAIN_ONSET, _cont_m5, _bin_m5)

print("\n[primary surge M5]")
print(f"  m= 5 : HR={r_m5_5[0]:.3f} [{r_m5_5[1]:.3f},{r_m5_5[2]:.3f}] p={r_m5_5[3]:.3f}")
print(f"  m=20 : HR={r_m5_20[0]:.3f} [{r_m5_20[1]:.3f},{r_m5_20[2]:.3f}] p={r_m5_20[3]:.3f}")
print(f"  세트간 분산 B(log HR): m=5={B5:.5f}  m=20={B20:.5f}")
print(f"  MC 표준오차 ~ sqrt(B/m): m=5={np.sqrt(B5/5):.4f}  m=20={np.sqrt(B20/20):.4f}")
print("=" * 72)
print("해석: HR·CI가 m=5↔m=20에서 실질적으로 동일하면 m=5 결과는 MC-안정.")
print("      보고 전략 → 주 분석을 m=20으로 승격하고, '5와 20이 동일'을 각주로.")
print("=" * 72)

imp_m_tab = {"m5": r_m5_5, "m20": r_m5_20, "B5": B5, "B20": B20, "missrate": miss}

*원본 셀 36*


In [ ]:
# =====================================================================
# [셀 18] 2.4 연속 dose-response: max ΔTMP restricted cubic spline  (리뷰 2.4)
#   배경: 리뷰어 = 100 mmHg 이분화가 dose-response를 숨긴다. 12h landmark에서
#     max ΔTMP를 연속변수로 RCS 적합해 (a)비선형성 (b)100 mmHg 지점의 정당성 평가.
#   ★ 실익: 곡선이 100 근처에서 꺾이면 cut-point 정당화, 밋밋/단조면 surge가
#     dichotomization artifact라는 의심을 우리 손으로 먼저 확인(어느 쪽이든 정보).
#   정의 통일: baseline = median(-24~+3h) (surge MAIN과 동일 기준),
#     max ΔTMP = (0~12h 최대 TMP) − baseline_median.  음수는 0으로 floor 안 함
#     (실제 하강도 정보) → 원자료 그대로 연속.
#   모형: 12h 생존자(lm, 셀11 재활용) landmark Cox. RCS(4 knots, Harrell 위치).
#     ★ CoxPHFitter는 robust=True/cluster_col 지원(검증됨) → landmark는 1행/환자
#       이므로 robust 불필요하나, 안정 위해 penalizer=0.01만.
#   비선형성 검정: RCS 전체 vs 선형만 partial-LR (또는 spline 비선형항 Wald).
#   예측: β̂로 ΔTMP 격자의 HR(ref=0) 곡선 + 100 지점 HR. MICE 세트별 예측 Rubin 풀링.
#   전제: 셀0(mice_impute, CoxPHFitter, tmp, LM_H, FU_H, CONT_M, BIN_M),
#         셀3(c, tmp, cmeas), 셀11(lm 또는 여기서 재구성)  ※ lm 없으면 재생성 포함
# =====================================================================
import numpy as np, pandas as pd
from scipy.stats import chi2
from lifelines import CoxPHFitter

# ---- baseline_median(-24~+3h)과 max(0~12h) → max ΔTMP ----
#   tmp = (stay_id, h, valuenum). baseline 창은 surge 정의와 동일하게 h<=3의 median,
#   단 surge_onset은 h<=3 median을 baseline으로 쓰므로 그와 통일.
def _baseline_median(g):
    b = g[g["h"] <= 3]["valuenum"]
    return b.median() if len(b) else np.nan

def _max_delta_12h(g):
    base = _baseline_median(g)
    w = g[(g["h"] >= 0) & (g["h"] <= LM_H)]
    if len(w) == 0 or pd.isna(base): return np.nan
    return float(w["valuenum"].max() - base)

_dmax = tmp.groupby("stay_id").apply(_max_delta_12h)
_dmax = _dmax.dropna().rename("max_dtmp").reset_index()

# ---- lm(12h 생존자) 확보: 셀11에서 만들었으면 재사용, 없으면 최소 재구성 ----
if "lm" in dir() and isinstance(lm, pd.DataFrame) and "fu_h" in lm.columns:
    base_lm = lm.copy()
else:
    # 최소 재구성 (셀11 로직 축약): c에 death_h/disch_h/event28 존재 전제
    _cl = c.merge(_dmax, on="stay_id", how="left")
    _cl["died_28d"] = _cl["event28"]
    b = _cl[_cl["max_dtmp"].notna()].copy()
    b = b[(b["death_h"].isna()) | (b["death_h"] > LM_H)]
    b = b[(b["disch_h"].isna()) | (b["disch_h"] > LM_H)]
    _end = np.minimum(b["death_h"].fillna(np.inf), FU_H)
    _cz = (b["died_28d"] == 0) & b["disch_h"].notna() & (b["disch_h"] < _end)
    _end = _end.copy(); _end[_cz] = b["disch_h"][_cz]
    b["fu_h"] = _end - LM_H
    b["event"] = ((b["died_28d"] == 1) & (b["death_h"] <= FU_H)).astype(int)
    base_lm = b[b["fu_h"] > 0].reset_index(drop=True)

# max_dtmp 부착 (lm에 없을 수 있음)
if "max_dtmp" not in base_lm.columns:
    base_lm = base_lm.merge(_dmax, on="stay_id", how="left")
d_rcs = base_lm[base_lm["max_dtmp"].notna()].reset_index(drop=True)
print("=" * 72)
print("  [2.4] max ΔTMP restricted cubic spline (12h landmark)")
print("=" * 72)
print(f"  대상 = 12h 생존 & max ΔTMP 산출 가능: N={len(d_rcs)}, 28d 사망={int(d_rcs['event'].sum())}")
print(f"  max ΔTMP 분포: median={d_rcs['max_dtmp'].median():.1f}, "
      f"IQR[{d_rcs['max_dtmp'].quantile(.25):.1f}, {d_rcs['max_dtmp'].quantile(.75):.1f}], "
      f"범위[{d_rcs['max_dtmp'].min():.1f}, {d_rcs['max_dtmp'].max():.1f}]")

# ---- 공변량 대치 (셀0 mice_impute, landmark 종점) ----
CONT_R = [x for x in CONT_M if x in d_rcs.columns and not d_rcs[x].isna().all()]
BIN_R  = [x for x in BIN_M if x in d_rcs.columns]
for x in BIN_R:
    d_rcs[x] = pd.to_numeric(d_rcs[x], errors="coerce").fillna(0).astype(int)
XC_R = mice_impute(d_rcs, CONT_R, "event", "fu_h", m=M_IMP)

# ---- RCS basis: Harrell 4-knot truncated power basis, 고정 knot ----
#   knot 위치 = max ΔTMP 분포의 5/35/65/95 백분위 (Harrell, RMS 표준 4-knot).
#   basis = [x(linear), s1, s2] (df=3). 경계 knot 밖 선형 제약 충족(검증됨).
#   knot은 관측표본(d_rcs)에서 한 번만 산출해 전 MICE 세트·예측에 고정.
KNOTS = np.quantile(d_rcs["max_dtmp"], [0.05, 0.35, 0.65, 0.95])
_k1, _k2, _k3, _k4 = KNOTS
GRID = np.linspace(max(0.0, d_rcs["max_dtmp"].min()),
                   min(250, d_rcs["max_dtmp"].max()), 61)  # 표시용 상한 250 (그 이상 N 희박)

def _rcs_design(x, ref_di=None):
    """Harrell RMS 4-knot RCS truncated power basis (고정 knot).
       정규화 (k4-k1)^2 (Harrell 관례). ref_di 인자는 API 호환용(미사용)."""
    x = np.asarray(x, float)
    def tp(u, k): return np.maximum(u - k, 0.0) ** 3
    denom = (_k4 - _k1) ** 2
    s1 = (tp(x, _k1) - tp(x, _k3) * (_k4 - _k1) / (_k4 - _k3)
          + tp(x, _k4) * (_k3 - _k1) / (_k4 - _k3)) / denom
    s2 = (tp(x, _k2) - tp(x, _k3) * (_k4 - _k2) / (_k4 - _k3)
          + tp(x, _k4) * (_k3 - _k2) / (_k4 - _k3)) / denom
    return np.column_stack([x, s1, s2]), None

# ---- MICE 세트별 RCS Cox 적합 → GRID HR(ref=0) + 비선형 LR 검정 ----
grid_lhr = []          # 세트별 log-HR(grid) vs ref=0
lr_pvals = []          # 세트별 비선형성 p (spline vs linear)
for mi in range(len(XC_R)):
    dd = d_rcs[["fu_h", "event", "max_dtmp"]].copy()
    for cc in CONT_R: dd[cc] = XC_R[mi][cc].values
    for cc in BIN_R:  dd[cc] = d_rcs[cc].values
    # RCS basis
    Bmat, di = _rcs_design(dd["max_dtmp"].values)
    bcols = [f"rcs{j}" for j in range(Bmat.shape[1])]
    for j, cn in enumerate(bcols): dd[cn] = Bmat[:, j]
    covs = bcols + [c_ for c_ in (CONT_R + BIN_R) if dd[c_].nunique() > 1]
    dd_fit = dd[["fu_h", "event"] + covs].copy()
    cph = CoxPHFitter(penalizer=0.01)
    cph.fit(dd_fit, duration_col="fu_h", event_col="event")
    # 선형 모형(비선형 검정 대조군)
    dd_lin = dd[["fu_h", "event", "max_dtmp"]
                + [c_ for c_ in (CONT_R + BIN_R) if dd[c_].nunique() > 1]].copy()
    cph_lin = CoxPHFitter(penalizer=0.01)
    cph_lin.fit(dd_lin, duration_col="fu_h", event_col="event")
    # partial-LR 비선형성 검정 (df = spline항수 - 1)
    ll_full = cph.log_likelihood_; ll_lin = cph_lin.log_likelihood_
    dfnl = len(bcols) - 1
    lr = 2 * (ll_full - ll_lin)
    lr_pvals.append(1 - chi2.cdf(max(lr, 0), dfnl))
    # GRID 예측 log-HR (ref=0): basis(grid) - basis(0), 공변량은 상수 → 상쇄
    Bg, _ = _rcs_design(GRID)
    B0, _ = _rcs_design(np.array([0.0]))
    beta = np.array([cph.params_[cn] for cn in bcols])
    lhr = (Bg - B0) @ beta
    grid_lhr.append(lhr)

grid_lhr = np.array(grid_lhr)
lhr_mean = grid_lhr.mean(axis=0)
lhr_lo = np.percentile(grid_lhr, 2.5, axis=0)
lhr_hi = np.percentile(grid_lhr, 97.5, axis=0)
hr_curve = np.exp(lhr_mean)

print(f"\n  RCS 4 knots (Harrell, 5/35/65/95 pct of max ΔTMP) = "
      f"[{_k1:.1f}, {_k2:.1f}, {_k3:.1f}, {_k4:.1f}]")
print(f"  비선형성 검정 (spline vs linear), MICE 평균 p = {np.mean(lr_pvals):.3f}"
      f"  [{'비선형 유의' if np.mean(lr_pvals)<0.05 else '비선형 근거 약함(=단조/선형에 가까움)'}]")
# 곡선 주요 지점 (ref = ΔTMP 0, 상승 없음). 관측범위 밖은 외삽이므로 표시만.
_hi = d_rcs["max_dtmp"].max()
print(f"  곡선 HR (ref = ΔTMP 0, 관측 상한 {_hi:.0f}):")
for xq in [0, 50, 100, 150, 200, 250]:
    if xq > GRID.max():
        continue
    j = int(np.argmin(np.abs(GRID - xq)))
    print(f"    ΔTMP={GRID[j]:6.1f} → HR={hr_curve[j]:.3f} "
          f"[{np.exp(lhr_lo[j]):.3f}, {np.exp(lhr_hi[j]):.3f}]")
print("=" * 72)
print("해석 가이드:")
print("  · 비선형 p<0.05 & 상승 구간에서 곡선이 가팔라짐 → dose-response 비선형.")
print("  · 곡선이 baseline 위에서 단조 증가(비선형 p 큼) → 선형에 가까운 dose-response;")
print("    surge 이분화는 이 연속 관계의 이산 근사로 기술 가능.")
print("  · 곡선 자체가 flat/null → surge 신호가 측정·선택의 산물일 가능성 시사.")
print("=" * 72)

# 시각화: RCS 곡선 (Fig S 후보)
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5.2, 3.6))
    ax.plot(GRID, hr_curve, color="#1f3a5f", lw=2)
    ax.fill_between(GRID, np.exp(lhr_lo), np.exp(lhr_hi), color="#1f3a5f", alpha=0.15)
    ax.axhline(1.0, color="grey", lw=0.8, ls="--")
    ax.set_xlim(0, 250)
    ax.set_xlabel("Maximum ΔTMP within 12 h (mmHg)")
    ax.set_ylabel("Hazard ratio (ref = no rise)")
    ax.set_title("Dose-response: early ΔTMP and 28-day mortality (RCS)")
    fig.tight_layout()
    fig.savefig(FIGDIR / "rcs_dtmp_doseresponse.png", dpi=200)
    fig.savefig(FIGDIR / "rcs_dtmp_doseresponse.pdf")
    plt.close(fig)
    print("  [그림 저장] figs/rcs_dtmp_doseresponse.png/.pdf")
except Exception as e:
    print("  (그림 생략:", e, ")")

rcs_tab = {"grid": GRID, "hr": hr_curve, "lo": np.exp(lhr_lo), "hi": np.exp(lhr_hi),
           "nonlin_p": float(np.mean(lr_pvals)), "knots": KNOTS.tolist(),
           "N": len(d_rcs), "events": int(d_rcs["event"].sum())}

*원본 셀 37*


In [ ]:
# =====================================================================
# [셀 19] 2.6 측정 강도(informative observation) 기술 비교  (리뷰 2.6)
#   배경: surge는 측정값이 있을 때만 검출 → 측정이 잦은 환자일수록 surge 검출↑.
#     측정 빈도가 감시강도(중증도 대리)라면 surge가 그 대리변수일 수 있음.
#   ★ 방침(합의): inverse-intensity weighting은 하지 않고 기술통계만 보고.
#     초기 12h TMP 측정 횟수·간격·첫 측정시각을 surge군 vs 비surge군에서 비교.
#   출력: 군별 median(IQR) + Mann-Whitney p. 유의차가 크면 한계에 명시,
#     작으면 "측정강도 교란 근거 약함"으로 방어.
#   전제: 셀0(상수), 셀3(cmeas, tmp), 셀6(MAIN_ONSET)
# =====================================================================
import numpy as np, pandas as pd
from scipy.stats import mannwhitneyu

print("=" * 72)
print("  [2.6] 초기 12h TMP 측정 강도: surge(+) vs surge(-)  (기술통계)")
print("=" * 72)

# cmeas 한정 + surge flag
_sids = set(cmeas["stay_id"].astype(int))
t12 = tmp[(tmp["stay_id"].astype(int).isin(_sids)) &
          (tmp["h"] >= 0) & (tmp["h"] <= LM_H)].copy()

def _meas_feats(g):
    h = np.sort(g["h"].values)
    n = len(h)
    first = h[0] if n else np.nan
    gap = np.median(np.diff(h)) if n >= 2 else np.nan
    return pd.Series({"n_meas": n, "first_h": first, "med_gap_h": gap})

feat = t12.groupby("stay_id").apply(_meas_feats).reset_index()
feat["surge"] = feat["stay_id"].map(
    lambda s: 1 if pd.notna(MAIN_ONSET.get(int(s), np.nan)) else 0)

# cmeas 중 12h창 측정이 아예 없던 환자도 포함(측정 0 → surge 검출 불가 구조 확인)
_all = pd.DataFrame({"stay_id": list(_sids)})
feat = _all.merge(feat, on="stay_id", how="left")
feat["n_meas"] = feat["n_meas"].fillna(0)
feat["surge"] = feat["stay_id"].map(
    lambda s: 1 if pd.notna(MAIN_ONSET.get(int(s), np.nan)) else 0)

def _summ(col):
    a = feat[feat.surge == 1][col].dropna()
    b = feat[feat.surge == 0][col].dropna()
    if len(a) >= 3 and len(b) >= 3:
        try:
            p = mannwhitneyu(a, b, alternative="two-sided").pvalue
        except Exception:
            p = np.nan
    else:
        p = np.nan
    return (a.median(), a.quantile(.25), a.quantile(.75),
            b.median(), b.quantile(.25), b.quantile(.75), p)

print(f"\n  surge(+) n={int((feat.surge==1).sum())}  |  surge(-) n={int((feat.surge==0).sum())}")
print(f"\n  {'지표':<22}{'surge(+) med[IQR]':<26}{'surge(-) med[IQR]':<26}{'p(MWU)'}")
for col, lab in [("n_meas", "12h 측정 횟수"),
                 ("first_h", "첫 측정 시각(h)"),
                 ("med_gap_h", "측정 간격 median(h)")]:
    a_m, a_lo, a_hi, b_m, b_lo, b_hi, p = _summ(col)
    print(f"  {lab:<22}{a_m:5.1f} [{a_lo:.1f},{a_hi:.1f}]        "
          f"{b_m:5.1f} [{b_lo:.1f},{b_hi:.1f}]        "
          f"{p:.3f}" if not np.isnan(p) else
          f"  {lab:<22}(표본부족)")
print("=" * 72)
print("해석 가이드:")
print("  · 측정 횟수 p가 유의하고 surge(+)가 훨씬 잦으면 → '감시강도 교란' 가능성을")
print("    한계에 명시하고, surge를 exploratory로 유지하는 근거로 사용.")
print("  · 차이가 작으면 → 측정강도가 surge를 만든다는 근거 약함(방어 재료).")
print("  · IP-intensity weighting은 46 events에서 불안정 → 기술비교로 투명 보고에 그침.")
print("=" * 72)

meas_intensity_tab = feat

*원본 셀 38*


In [ ]:
# =====================================================================
# [셀 20] 2.5 측정 선택편향 IP-of-observation weighting  (리뷰 2.5, ★확인용)
#   배경: measured TMP 보유(2층위 진입)는 기기·시대·진료과정으로 선택된 하위집단.
#     리뷰어 = 측정 가용성을 outcome으로 한 propensity로 IPW/standardization 권고.
#   ★ 사전 입장(합의): 이 분석은 '주 분석 대체'가 아니라 '민감도 겸 타당성 진단'.
#     surge event=46에서 IPW는 유효표본(ESS)을 깎고 CI를 넓혀 경계적 결과를
#     불안정화할 위험이 큼. 따라서 (1)weight 분포·ESS를 먼저 보고,
#     (2)weighted surge HR을 bootstrap CI와 함께 산출한 뒤,
#     (3)그 수치로 '본분석 부적합 → rebuttal 방어'가 정당한지 데이터로 확정.
#   설계:
#     A. 선택모형: c(1493)에서 measured-TMP 보유(=cmeas 진입) 여부를 로지스틱.
#        예측인자 = calendar era(연도그룹) + CRRT intensity(가동시간/blood_flow) + 공변량.
#     B. stabilized IPW = P(measured)/P(measured|X)  (2층위 진입자에게만).
#        truncation 1/99 percentile.
#     C. weighted surge M5 Cox(CoxTimeVaryingFitter, weights_col) →
#        점추정 + 환자 bootstrap CI(내장 robust 미구현이므로).
#   진단 출력: weight min/max/CV, ESS = (Σw)²/Σw², weighted vs unweighted HR.
#   전제: 셀0(build, CoxTimeVaryingFitter, mice_impute, CONT_M, BIN_M, M5블록,
#         SEG_MAIN, cum_on_at), 셀3(c, cmeas, XC_MEAS, tmp),
#         셀3A(SEG_MAIN), 셀4(year_grp), 셀6(MAIN_ONSET, r_main)
# =====================================================================
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from lifelines import CoxTimeVaryingFitter

print("=" * 72)
print("  [2.5] IP-of-observation weighting (측정 선택편향)  ★확인용 민감도")
print("=" * 72)

# ---- A. 선택 라벨: c의 각 환자가 2층위(cmeas)에 들어갔는가 ----
_cm_ids = set(cmeas["stay_id"].astype(int))
sel = c.copy()
sel["measured"] = sel["stay_id"].astype(int).isin(_cm_ids).astype(int)

# 예측인자: era(year_grp) + intensity(누적가동시간, blood_flow) + 핵심 공변량
#   누적 가동시간: SEG_MAIN에서 총 가동시간(마지막 구간 종료) 근사
def _total_on(sid):
    segs = SEG_MAIN.get(int(sid), [])
    return sum(e - s for s, e in segs) if segs else 0.0
sel["cum_on_total"] = sel["stay_id"].map(_total_on)

_pred_cont = [x for x in ["anchor_age", "weight_kg", "sofa_total", "blood_flow",
                          "cum_on_total"] if x in sel.columns]
_pred_bin  = [x for x in ["male", "vaso_use", "mech_vent"] if x in sel.columns]
# year_grp 더미
if "year_grp" in sel.columns:
    yd = pd.get_dummies(sel["year_grp"].astype(str), prefix="yr", drop_first=True)
else:
    yd = pd.DataFrame(index=sel.index)

# 예측행렬(결측은 median/0 대치 — 선택모형용 단순 처리)
Xdf = sel[_pred_cont].copy()
for cc in _pred_cont: Xdf[cc] = Xdf[cc].fillna(Xdf[cc].median())
for cc in _pred_bin:
    Xdf[cc] = pd.to_numeric(sel[cc], errors="coerce").fillna(0).astype(int)
Xdf = pd.concat([Xdf, yd.astype(int)], axis=1)
y = sel["measured"].values

lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(Xdf.values, y)
ps = lr.predict_proba(Xdf.values)[:, 1]           # P(measured | X)
ps = np.clip(ps, 1e-3, 1 - 1e-3)
p_marg = y.mean()

# ---- B. stabilized IPW (측정된 환자에게만; 2층위 분석 대상) ----
sel["ps"] = ps
sel["sw"] = np.where(sel["measured"] == 1, p_marg / sel["ps"], np.nan)
# truncation 1/99 pct
_sw_meas = sel.loc[sel.measured == 1, "sw"]
lo_t, hi_t = np.percentile(_sw_meas.dropna(), [1, 99])
sel["sw_tr"] = sel["sw"].clip(lo_t, hi_t)

# cmeas에 weight 매핑
w_map = dict(zip(sel["stay_id"].astype(int), sel["sw_tr"]))
w_vec = cmeas["stay_id"].astype(int).map(w_map).fillna(1.0).values

# ---- 진단: weight 분포 + ESS ----
w = w_vec
ESS = (w.sum() ** 2) / (np.square(w).sum())
from sklearn.metrics import roc_auc_score
_sel_auc = roc_auc_score(y, ps)
print("\n[A. 선택모형 진단]")
print(f"  measured 비율 = {p_marg:.3f} (cmeas {int(y.sum())}/{len(y)})")
print(f"  예측인자: {_pred_cont + _pred_bin + list(yd.columns)}")
print(f"  선택모형 AUC(학습표본, 낙관적) = {_sel_auc:.3f}"
      f"  [높을수록 측정군이 뚜렷이 선택된 집단 → 선택편향 우려 실재]")
print("\n[B. stabilized weight 분포 (cmeas, truncated 1/99)]")
print(f"  min={w.min():.3f}  max={w.max():.3f}  mean={w.mean():.3f}  "
      f"CV={w.std()/w.mean():.3f}")
print(f"  유효표본 ESS = {ESS:.0f} / {len(w)}  (감소율 {100*(1-ESS/len(w)):.1f}%)")
print(f"  surge(+) 중 weight 분포도 확인 권장(46명 소수라 민감).")

# ---- C. weighted surge M5 Cox (weights_col) + bootstrap CI ----
_CONT_M5 = DEMO_C + SEV_C + LAB_C + TX_C
_BIN_M5  = DEMO_B + SEV_B + TX_B
_cont_m5 = [x for x in _CONT_M5 if x in cmeas.columns or x in XC_MEAS[0].columns]
_bin_m5  = [x for x in _BIN_M5 if x in cmeas.columns]

def _weighted_surge_hr(d, Xc, onset, cont, binc, wmap, penalizer=PENALIZER):
    cp = build(d, Xc, onset, cont, binc)
    if len(cp) == 0 or cp["surge"].sum() == 0: return None
    cp["w"] = cp["id"].map(wmap).fillna(1.0)
    fit = ["surge"] + [cc for cc in (cont + binc) if cp[cc].nunique() > 1]
    m = CoxTimeVaryingFitter(penalizer=penalizer)
    m.fit(cp[["id", "start", "stop", "event", "w"] + fit], id_col="id",
          start_col="start", stop_col="stop", event_col="event",
          weights_col="w", show_progress=False)
    return float(m.summary.loc["surge", "coef"])

# id(=stay_id) → weight 매핑
_wmap = dict(zip(cmeas["stay_id"].astype(int), w_vec))
_coef_w = _weighted_surge_hr(cmeas, XC_MEAS[0][_cont_m5], MAIN_ONSET,
                             _cont_m5, _bin_m5, _wmap)

# bootstrap CI (환자 복원추출; weight도 함께 이동)
rng = np.random.default_rng(20260629)
_bw = []
_n = len(cmeas)
_cm = cmeas.reset_index(drop=True)
_Xc0 = XC_MEAS[0].reset_index(drop=True)
for b in range(400):
    idx = rng.integers(0, _n, size=_n)
    d_b = _cm.iloc[idx].copy().reset_index(drop=True)
    new = np.arange(len(d_b)); old = d_b["stay_id"].values.copy()
    d_b["stay_id"] = new
    onset_b = {int(new[k]): MAIN_ONSET.get(int(old[k]), np.nan) for k in range(len(d_b))}
    wmap_b = {int(new[k]): _wmap.get(int(old[k]), 1.0) for k in range(len(d_b))}
    Xc_b = _Xc0.iloc[idx][_cont_m5].reset_index(drop=True)
    cf = _weighted_surge_hr(d_b, Xc_b, onset_b, _cont_m5, _bin_m5, wmap_b)
    if cf is not None: _bw.append(cf)
_bw = np.array(_bw)

print("\n[C. weighted surge M5]")
if _coef_w is not None and len(_bw) > 20:
    print(f"  weighted HR = {np.exp(_coef_w):.3f}  "
          f"boot95%CI [{np.exp(np.percentile(_bw,2.5)):.3f}, "
          f"{np.exp(np.percentile(_bw,97.5)):.3f}]  (n_boot={len(_bw)})")
    print(f"  참고 unweighted 주결과 r_main HR={r_main[0]:.3f} "
          f"[{r_main[1]:.3f},{r_main[2]:.3f}]")
else:
    print("  적합 실패 또는 bootstrap 표본 부족 → IPW 불안정 시사(방어 근거).")
print("=" * 72)
print("판단 가이드(데이터로 결정):")
print("  · ESS 대폭 감소(예: >30%) + weighted CI가 1을 넉넉히 포함 →")
print("    'IPW는 46 events에서 불안정, exploratory 프레임과 상충'을 rebuttal에 명시.")
print("  · weighted HR이 unweighted와 유사·CI 유지 → 부록 민감도로 넣어도 무방.")
print("=" * 72)

ipw_tab = {"ESS": ESS, "N": len(w), "w_min": float(w.min()), "w_max": float(w.max()),
           "w_cv": float(w.std()/w.mean()),
           "hr_weighted": float(np.exp(_coef_w)) if _coef_w is not None else None,
           "ci_boot": (float(np.exp(np.percentile(_bw,2.5))),
                       float(np.exp(np.percentile(_bw,97.5)))) if len(_bw) > 20 else None}

# 최종 체크 (260704)

*원본 셀 40*


In [ ]:
# =====================================================================
# [확인용 셀] Table 2 m 기준·공변량 출처 + landmark P 최소값 일괄 확인
#   목적: 원고 Table 2(vaso 1.81/SOFA 1.05)가 어느 산출인지,
#         m=5 vs m=20에서 M5 전체 공변량이 어떻게 나오는지 확정.
#   전제: 셀0(fit_surge_cox, mice_impute, CONT_M, BIN_M, PENALIZER),
#         셀3(cmeas, XC_MEAS[m=5]), 셀6(MAIN_ONSET, r_main),
#         셀17(XC_MEAS_20 있으면 재사용; 없으면 여기서 생성),
#         셀11(stats_12h 또는 landmark 입력)
# =====================================================================
import numpy as np, pandas as pd

print("="*72)
print("  [확인 A] M5 전체 공변량: m=5 vs m=20 (Table 2 출처 판정)")
print("="*72)

CONT_M5 = CONT_M                      # 셀0 정의 연속 11
BIN_M5  = BIN_M                       # 셀0 정의 이진 5

# m=5 (기존 XC_MEAS)로 M5 full, 공변량 pooltab까지
r5, tab5, _ = fit_surge_cox(cmeas, XC_MEAS, MAIN_ONSET, CONT_M5, BIN_M5, full_return=True)

# m=20 세트 확보 (셀17에서 만든 XC_MEAS_20 재사용, 없으면 생성)
try:
    _x20 = XC_MEAS_20
    print("  (XC_MEAS_20 셀17 것 재사용)")
except NameError:
    _x20 = mice_impute(cmeas, CONT_M5, "event28", "end28_h", m=20)
    print("  (XC_MEAS_20 신규 생성 m=20)")

r20, tab20, _ = fit_surge_cox(cmeas, _x20, MAIN_ONSET, CONT_M5, BIN_M5, full_return=True)

print(f"\n  surge  m=5 : HR={r5[0]:.3f} [{r5[1]:.3f},{r5[2]:.3f}] p={r5[3]:.3f}")
print(f"  surge  m=20: HR={r20[0]:.3f} [{r20[1]:.3f},{r20[2]:.3f}] p={r20[3]:.3f}")

print(f"\n  {'변수':<18}{'m=5 HR[CI] p':<34}{'m=20 HR[CI] p'}")
def _row(tab, var):
    r = tab[tab['var']==var]
    if len(r)==0: return "  (없음)"
    r=r.iloc[0]
    return f"{r.HR:.3f} [{r.lo:.3f},{r.hi:.3f}] p={r.p:.3f}"
for v in ['surge']+CONT_M5+BIN_M5:
    print(f"  {v:<18}{_row(tab5,v):<34}{_row(tab20,v)}")

print("\n  ★판정: 원고 Table2 vaso 1.81/SOFA 1.05가 위 m=5·m=20 중")
print("         어느 쪽과 일치하는지, 아니면 둘 다 아닌지 확인.")

print("\n"+"="*72)
print("  [확인 B] landmark 7통계량 crude/full p 최소값 (원고 'all P>.13' 판정)")
print("="*72)
# 셀11에서 이미 산출된 landmark 결과 테이블이 있으면 그 최소 p, 없으면 재산출 안내
try:
    _lm = landmark_tab  # 셀11이 저장한 이름 후보
    pmin_crude = _lm[_lm['model']=='crude']['p'].min()
    pmin_full  = _lm[_lm['model']=='full']['p'].min()
    print(f"  crude 최소 p = {pmin_crude:.3f}")
    print(f"  full  최소 p = {pmin_full:.3f}")
    print(f"  → 원고 'all P>.13': crude기준 {'성립' if pmin_crude>0.13 else '위반'}, "
          f"full기준 {'성립(실제 >%.2f)'%pmin_full if pmin_full>0.13 else '위반'}")
except NameError:
    print("  landmark_tab 변수명 미확인. 셀11 출력에서 crude/full 최소 p 직접 확인 필요.")
    print("  (셀11 검토 시: crude min p=0.143(min)·range 0.091 / full 최소 range 0.340 였음)")
print("="*72)